<a href="https://colab.research.google.com/github/DLHolmes4/Coding-demo/blob/main/VLCP_Walmart_BOL_API_Test.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import requests
import json
import os
from datetime import datetime

print("✓ VLCP Walmart/BOL API test environment ready")

In [ ]:
# CELL 2 — Mock Walmart BOL response for VLCP testing

mock_walmart_bol = {
    "source": "Walmart Mock API",
    "environment": "test",
    "shipment_id": "WMT-TEST-10001",
    "bol_number": "BOL-TEST-50001",

    "shipper": {
        "name": "Walmart Distribution Center",
        "city": "Bentonville",
        "state": "AR"
    },

    "receiver": {
        "name": "Retail Receiving Facility",
        "city": "Greenville",
        "state": "SC"
    },

    "carrier": {
        "name": "Test Carrier LLC",
        "usdot": "951224",
        "mc_number": "MC-123456"
    },

    "equipment": {
        "tractor_number": "TRK-101",
        "trailer_number": "TRL-501",
        "equipment_type": "53FT Dry Van"
    },

    "freight": {
        "commodity": "General Merchandise",
        "weight_lbs": 28750,
        "pieces": 22
    },

    "pickup": {
        "scheduled": "2026-08-25T08:00:00",
        "actual": "2026-08-25T07:52:00"
    },

    "delivery": {
        "scheduled": "2026-08-26T14:00:00",
        "actual": None
    }
}

print("✓ Mock Walmart BOL loaded")
print(json.dumps(mock_walmart_bol, indent=2))

In [ ]:
# CELL 3 — Normalize external BOL data into VLCP schema
# Dynamic: works with any BOL object passed into the function.

from datetime import datetime, UTC
import json


def normalize_bol_to_vlcp(bol):
    """
    Convert an external BOL/shipment record into the
    standardized VLCP shipment schema.

    The function does NOT hard-code a carrier or USDOT number.
    Whatever carrier information exists in the incoming BOL
    is carried into the VLCP record for later verification.
    """

    vlcp_record = {

        # -----------------------------------
        # VLCP RECORD INFORMATION
        # -----------------------------------
        "vlcp_record_type": "shipment",
        "schema_version": "0.1",

        # -----------------------------------
        # SOURCE DOCUMENT
        # -----------------------------------
        "document": {
            "document_type": "BOL",
            "bol_number": bol.get("bol_number"),
            "shipment_id": bol.get("shipment_id"),
            "source": bol.get("source"),
            "source_environment": bol.get("environment")
        },

        # -----------------------------------
        # SHIPPER
        # -----------------------------------
        "shipper": {
            "name": bol.get("shipper", {}).get("name"),
            "city": bol.get("shipper", {}).get("city"),
            "state": bol.get("shipper", {}).get("state")
        },

        # -----------------------------------
        # RECEIVER
        # -----------------------------------
        "receiver": {
            "name": bol.get("receiver", {}).get("name"),
            "city": bol.get("receiver", {}).get("city"),
            "state": bol.get("receiver", {}).get("state")
        },

        # -----------------------------------
        # CARRIER
        # -----------------------------------
        "carrier": {
            "name": bol.get("carrier", {}).get("name"),

            # Dynamic USDOT:
            # Uses whatever DOT exists in the incoming BOL.
            "usdot": str(
                bol.get("carrier", {}).get("usdot") or ""
            ).strip(),

            "mc_number": bol.get("carrier", {}).get("mc_number")
        },

        # -----------------------------------
        # EQUIPMENT
        # -----------------------------------
        "equipment": {
            "tractor_number": bol.get(
                "equipment", {}
            ).get("tractor_number"),

            "trailer_number": bol.get(
                "equipment", {}
            ).get("trailer_number"),

            "equipment_type": bol.get(
                "equipment", {}
            ).get("equipment_type")
        },

        # -----------------------------------
        # FREIGHT
        # -----------------------------------
        "freight": {
            "commodity": bol.get(
                "freight", {}
            ).get("commodity"),

            "weight_lbs": bol.get(
                "freight", {}
            ).get("weight_lbs"),

            "pieces": bol.get(
                "freight", {}
            ).get("pieces")
        },

        # -----------------------------------
        # PICKUP
        # -----------------------------------
        "pickup": {
            "scheduled": bol.get(
                "pickup", {}
            ).get("scheduled"),

            "actual": bol.get(
                "pickup", {}
            ).get("actual")
        },

        # -----------------------------------
        # DELIVERY
        # -----------------------------------
        "delivery": {
            "scheduled": bol.get(
                "delivery", {}
            ).get("scheduled"),

            "actual": bol.get(
                "delivery", {}
            ).get("actual")
        },

        # -----------------------------------
        # VLCP VERIFICATION STATE
        # -----------------------------------
        # Nothing is considered verified
        # merely because it appears on a BOL.
        "verification": {
            "carrier_verified": False,
            "equipment_verified": False,
            "document_verified": False,
            "verification_status": "PENDING"
        },

        # -----------------------------------
        # DATA PROVENANCE
        # -----------------------------------
        "provenance": {
            "ingested_at_utc": datetime.now(UTC).isoformat(),
            "original_source": bol.get("source")
        }
    }

    return vlcp_record


# -----------------------------------
# NORMALIZE CURRENT INPUT BOL
# -----------------------------------

vlcp_shipment = normalize_bol_to_vlcp(
    mock_walmart_bol
)


# -----------------------------------
# DISPLAY RESULT
# -----------------------------------

print("✓ BOL normalized into VLCP schema")
print(
    json.dumps(
        vlcp_shipment,
        indent=2
    )
)

In [ ]:
# CELL 3A — Fix UTC timestamp handling

from datetime import datetime, UTC

print("✓ Timezone-aware UTC enabled")

In [ ]:
# CELL 4 — Validate VLCP shipment before carrier verification
# Dynamic: works with any normalized VLCP shipment record.

def validate_vlcp_shipment(record):
    """
    Validate minimum requirements before VLCP attempts
    carrier or shipment verification.

    Returns:
        {
            "valid": True/False,
            "errors": [...],
            "warnings": [...]
        }
    """

    errors = []
    warnings = []

    # -----------------------------
    # RECORD TYPE
    # -----------------------------
    if record.get("vlcp_record_type") != "shipment":
        errors.append("Invalid or missing VLCP record type.")

    # -----------------------------
    # DOCUMENT
    # -----------------------------
    document = record.get("document", {})

    if not document.get("source"):
        errors.append("Document source is missing.")

    if not document.get("bol_number"):
        warnings.append("BOL number is missing.")

    if not document.get("shipment_id"):
        warnings.append("Shipment ID is missing.")

    # -----------------------------
    # CARRIER
    # -----------------------------
    carrier = record.get("carrier", {})

    usdot = str(
        carrier.get("usdot") or ""
    ).strip()

    if not usdot:
        errors.append("Carrier USDOT number is missing.")

    elif not usdot.isdigit():
        errors.append(
            f"Carrier USDOT must contain only numbers. Received: {usdot}"
        )

    if not carrier.get("name"):
        warnings.append(
            "Carrier name is missing. VLCP should retrieve it from an authoritative source."
        )

    # -----------------------------
    # SHIPPER
    # -----------------------------
    shipper = record.get("shipper", {})

    if not shipper.get("name"):
        warnings.append("Shipper name is missing.")

    # -----------------------------
    # RECEIVER
    # -----------------------------
    receiver = record.get("receiver", {})

    if not receiver.get("name"):
        warnings.append("Receiver name is missing.")

    # -----------------------------
    # FREIGHT
    # -----------------------------
    freight = record.get("freight", {})

    weight = freight.get("weight_lbs")

    if weight is not None:

        try:
            weight_value = float(weight)

            if weight_value <= 0:
                errors.append(
                    "Freight weight must be greater than zero."
                )

        except (TypeError, ValueError):
            errors.append(
                f"Invalid freight weight: {weight}"
            )

    # -----------------------------
    # EQUIPMENT
    # -----------------------------
    equipment = record.get("equipment", {})

    if not equipment.get("tractor_number"):
        warnings.append(
            "Tractor number is not provided."
        )

    if not equipment.get("trailer_number"):
        warnings.append(
            "Trailer number is not provided."
        )

    # -----------------------------
    # FINAL RESULT
    # -----------------------------
    return {
        "valid": len(errors) == 0,
        "errors": errors,
        "warnings": warnings
    }


validation_result = validate_vlcp_shipment(
    vlcp_shipment
)

print("VLCP SHIPMENT VALIDATION")
print("=" * 42)

if validation_result["valid"]:
    print("✓ Shipment record passed validation")
else:
    print("✗ Shipment record failed validation")

if validation_result["errors"]:
    print("\nERRORS")

    for error in validation_result["errors"]:
        print(f"  ✗ {error}")

if validation_result["warnings"]:
    print("\nWARNINGS")

    for warning in validation_result["warnings"]:
        print(f"  ⚠ {warning}")

if (
    not validation_result["errors"]
    and not validation_result["warnings"]
):
    print("\n✓ No errors or warnings found")

In [ ]:
# CELL 5 — Dynamically extract carrier USDOT from validated shipment

def get_carrier_usdot(record, validation_result=None):
    """
    Extract the USDOT number from any normalized VLCP shipment.

    This function does NOT hard-code a carrier.
    It uses the carrier contained in the current shipment record.
    """

    # Stop if the shipment already failed validation
    if validation_result is not None:
        if not validation_result.get("valid", False):
            raise ValueError(
                "VLCP shipment failed validation. Carrier verification blocked."
            )

    carrier = record.get("carrier", {})

    usdot = str(
        carrier.get("usdot") or ""
    ).strip()

    if not usdot:
        raise ValueError(
            "No USDOT number found in the shipment."
        )

    if not usdot.isdigit():
        raise ValueError(
            f"Invalid USDOT number: {usdot}"
        )

    return usdot


# -----------------------------------
# EXTRACT USDOT FROM CURRENT SHIPMENT
# -----------------------------------

carrier_dot = get_carrier_usdot(
    vlcp_shipment,
    validation_result
)

print("VLCP CARRIER INPUT")
print("=" * 42)
print(f"✓ USDOT extracted dynamically: {carrier_dot}")
print("✓ Carrier ready for external verification")

In [ ]:
# CELL 6 — Reusable FMCSA carrier lookup layer
# Dynamic: accepts any valid USDOT number.
# Uses the currently documented FMCSA QCMobile endpoint.

import requests


def get_fmcsa_carrier(usdot, webkey=None):
    """
    Retrieve FMCSA carrier information for any USDOT number.

    Parameters
    ----------
    usdot : str or int
        Any USDOT number.

    webkey : str, optional
        FMCSA QCMobile WebKey.

    Returns
    -------
    dict
        Structured lookup result for VLCP.
    """

    usdot = str(usdot).strip()

    # -----------------------------------
    # VALIDATE INPUT
    # -----------------------------------
    if not usdot:
        return {
            "success": False,
            "lookup_status": "INVALID_INPUT",
            "usdot": None,
            "error": "USDOT number is missing.",
            "data": None
        }

    if not usdot.isdigit():
        return {
            "success": False,
            "lookup_status": "INVALID_INPUT",
            "usdot": usdot,
            "error": "USDOT must contain only numbers.",
            "data": None
        }

    # -----------------------------------
    # OFFICIAL FMCSA QCMOBILE ENDPOINT
    # -----------------------------------
    base_url = "https://mobile.fmcsa.dot.gov/qc/services/carriers"

    url = f"{base_url}/{usdot}"

    # -----------------------------------
    # STOP CLEANLY IF NO WEBKEY EXISTS
    # -----------------------------------
    if not webkey:
        return {
            "success": False,
            "lookup_status": "WEBKEY_REQUIRED",
            "usdot": usdot,
            "source": "FMCSA QCMobile",
            "request_url": url,
            "error": (
                "FMCSA WebKey not configured. "
                "Carrier lookup was not attempted."
            ),
            "data": None
        }

    params = {
        "webKey": webkey
    }

    # -----------------------------------
    # API REQUEST
    # -----------------------------------
    try:

        response = requests.get(
            url,
            params=params,
            timeout=20
        )

        # -----------------------------------
        # AUTHENTICATION FAILURE
        # -----------------------------------
        if response.status_code == 401:

            return {
                "success": False,
                "lookup_status": "AUTHENTICATION_FAILED",
                "usdot": usdot,
                "status_code": 401,
                "source": "FMCSA QCMobile",
                "error": "FMCSA rejected the WebKey.",
                "data": None
            }

        # -----------------------------------
        # TRUE NOT FOUND
        # -----------------------------------
        if response.status_code == 404:

            return {
                "success": False,
                "lookup_status": "NOT_FOUND",
                "usdot": usdot,
                "status_code": 404,
                "source": "FMCSA QCMobile",
                "error": "USDOT record not found by FMCSA.",
                "data": None
            }

        # -----------------------------------
        # OTHER HTTP ERRORS
        # -----------------------------------
        response.raise_for_status()

        payload = response.json()

        # -----------------------------------
        # SUCCESS
        # -----------------------------------
        return {
            "success": True,
            "lookup_status": "FOUND",
            "usdot": usdot,
            "status_code": response.status_code,
            "source": "FMCSA QCMobile",
            "request_url": response.url,
            "data": payload
        }

    except requests.exceptions.Timeout:

        return {
            "success": False,
            "lookup_status": "TIMEOUT",
            "usdot": usdot,
            "error": "FMCSA request timed out.",
            "data": None
        }

    except requests.exceptions.ConnectionError:

        return {
            "success": False,
            "lookup_status": "CONNECTION_ERROR",
            "usdot": usdot,
            "error": "Unable to connect to FMCSA.",
            "data": None
        }

    except requests.exceptions.HTTPError as exc:

        return {
            "success": False,
            "lookup_status": "HTTP_ERROR",
            "usdot": usdot,
            "status_code": response.status_code,
            "error": f"FMCSA HTTP error: {exc}",
            "data": None
        }

    except ValueError:

        return {
            "success": False,
            "lookup_status": "INVALID_RESPONSE",
            "usdot": usdot,
            "error": "FMCSA returned a non-JSON response.",
            "data": None
        }

    except Exception as exc:

        return {
            "success": False,
            "lookup_status": "UNEXPECTED_ERROR",
            "usdot": usdot,
            "error": str(exc),
            "data": None
        }


# -----------------------------------
# CURRENT SHIPMENT LOOKUP
# -----------------------------------

fmcsa_result = get_fmcsa_carrier(
    carrier_dot
)


# -----------------------------------
# HUMAN-READABLE OUTPUT
# -----------------------------------

print("VLCP FMCSA LOOKUP")
print("=" * 42)

print(f"USDOT: {fmcsa_result.get('usdot')}")
print(f"Status: {fmcsa_result.get('lookup_status')}")

if fmcsa_result["success"]:
    print("✓ FMCSA carrier record retrieved")
else:
    print(f"⚠ {fmcsa_result.get('error')}")

In [ ]:
# CELL 7 — Load FMCSA WebKey securely from Colab Secrets

from google.colab import userdata

try:
    FMCSA_WEBKEY = userdata.get("FMCSA_WEBKEY")

    if FMCSA_WEBKEY:
        print("✓ FMCSA WebKey loaded securely")
    else:
        print("⚠ FMCSA WebKey not found")

except Exception as exc:
    FMCSA_WEBKEY = None
    print("⚠ Could not load FMCSA WebKey")
    print(f"Reason: {exc}")

In [ ]:
# CELL 8 — Run dynamic FMCSA lookup using secure WebKey

fmcsa_result = get_fmcsa_carrier(
    carrier_dot,
    webkey=FMCSA_WEBKEY
)

print("VLCP FMCSA LOOKUP")
print("=" * 42)

print(f"USDOT: {fmcsa_result.get('usdot')}")
print(f"Status: {fmcsa_result.get('lookup_status')}")

if fmcsa_result["success"]:
    print("✓ FMCSA carrier record retrieved")
    print(f"✓ HTTP status: {fmcsa_result.get('status_code')}")
else:
    print(f"⚠ {fmcsa_result.get('error')}")

In [ ]:
# CELL 9 — Normalize FMCSA carrier response into VLCP evidence
# Dynamic: works with any successful FMCSA carrier lookup.

def normalize_fmcsa_carrier_result(fmcsa_result):
    """
    Convert the FMCSA API response into a standardized
    VLCP carrier evidence object.
    """

    if not fmcsa_result.get("success"):
        return {
            "evidence_status": "UNAVAILABLE",
            "usdot": fmcsa_result.get("usdot"),
            "source": fmcsa_result.get("source"),
            "lookup_status": fmcsa_result.get("lookup_status"),
            "error": fmcsa_result.get("error"),
            "raw_data": None
        }

    payload = fmcsa_result.get("data") or {}

    # QCMobile responses commonly wrap carrier information
    # inside a "content" object.
    content = payload.get("content", payload)

    carrier = content.get("carrier", content)

    vlcp_carrier_evidence = {
        "evidence_type": "carrier_fmcsa",
        "evidence_status": "RETRIEVED",

        "source": {
            "name": "FMCSA QCMobile",
            "http_status": fmcsa_result.get("status_code"),
            "request_url": fmcsa_result.get("request_url")
        },

        "carrier": {
            "usdot": str(
                carrier.get("dotNumber")
                or carrier.get("usdot")
                or fmcsa_result.get("usdot")
                or ""
            ).strip(),

            "legal_name": (
                carrier.get("legalName")
                or carrier.get("legal_name")
            ),

            "dba_name": (
                carrier.get("dbaName")
                or carrier.get("dba_name")
            ),

            "mc_number": (
                carrier.get("mcNumber")
                or carrier.get("mc_number")
            ),

            "phy_city": (
                carrier.get("phyCity")
                or carrier.get("physicalCity")
            ),

            "phy_state": (
                carrier.get("phyState")
                or carrier.get("physicalState")
            ),

            "phy_zip": (
                carrier.get("phyZipcode")
                or carrier.get("physicalZip")
            ),

            "telephone": (
                carrier.get("telephone")
                or carrier.get("phone")
            ),

            "email": carrier.get("emailAddress"),

            "allowed_to_operate": carrier.get("allowedToOperate"),

            "status_code": (
                carrier.get("statusCode")
                or carrier.get("status")
            ),

            "safety_rating": (
                carrier.get("safetyRating")
                or carrier.get("safety_rating")
            ),

            "total_drivers": (
                carrier.get("totalDrivers")
                or carrier.get("drivers")
            ),

            "total_power_units": (
                carrier.get("totalPowerUnits")
                or carrier.get("powerUnits")
            )
        },

        "retrieved_at_utc": datetime.now(UTC).isoformat(),

        # Preserve original API response for later debugging/audit.
        "raw_data": payload
    }

    return vlcp_carrier_evidence


carrier_evidence = normalize_fmcsa_carrier_result(
    fmcsa_result
)

print("VLCP CARRIER EVIDENCE")
print("=" * 42)

print(f"Evidence status: {carrier_evidence.get('evidence_status')}")

carrier_info = carrier_evidence.get("carrier", {})

print(f"USDOT: {carrier_info.get('usdot')}")
print(f"Legal name: {carrier_info.get('legal_name')}")
print(f"DBA name: {carrier_info.get('dba_name')}")
print(f"Allowed to operate: {carrier_info.get('allowed_to_operate')}")
print(f"Safety rating: {carrier_info.get('safety_rating')}")
print(f"Power units: {carrier_info.get('total_power_units')}")
print(f"Drivers: {carrier_info.get('total_drivers')}")

In [ ]:
# CELL 10 — Compare BOL carrier claims against FMCSA evidence
# Dynamic: works with any normalized shipment and any FMCSA carrier result.

def normalize_name(name):
    """
    Normalize company names for comparison.
    Keeps comparison simple and explainable.
    """
    if not name:
        return ""

    return (
        str(name)
        .upper()
        .replace(",", "")
        .replace(".", "")
        .replace("-", " ")
        .strip()
    )


def compare_carrier_claim_to_fmcsa(vlcp_shipment, carrier_evidence):
    """
    Compare carrier claims from the shipment/BOL
    against FMCSA carrier evidence.
    """

    bol_carrier = vlcp_shipment.get("carrier", {})
    fmcsa_carrier = carrier_evidence.get("carrier", {})

    claimed_usdot = str(
        bol_carrier.get("usdot") or ""
    ).strip()

    verified_usdot = str(
        fmcsa_carrier.get("usdot") or ""
    ).strip()

    claimed_name = bol_carrier.get("name")
    verified_name = fmcsa_carrier.get("legal_name")

    dot_match = (
        bool(claimed_usdot)
        and bool(verified_usdot)
        and claimed_usdot == verified_usdot
    )

    name_match = (
        bool(claimed_name)
        and bool(verified_name)
        and normalize_name(claimed_name) == normalize_name(verified_name)
    )

    issues = []

    if not dot_match:
        issues.append(
            f"USDOT mismatch: BOL={claimed_usdot}, FMCSA={verified_usdot}"
        )

    if not name_match:
        issues.append(
            f"Carrier name mismatch: "
            f"BOL='{claimed_name}', "
            f"FMCSA='{verified_name}'"
        )

    # Overall verification result
    if dot_match and name_match:
        status = "VERIFIED"

    elif dot_match and not name_match:
        status = "CONFLICT"

    else:
        status = "FAILED"

    return {
        "verification_status": status,

        "dot_match": dot_match,
        "name_match": name_match,

        "claimed": {
            "usdot": claimed_usdot,
            "carrier_name": claimed_name
        },

        "verified": {
            "usdot": verified_usdot,
            "legal_name": verified_name,
            "allowed_to_operate": fmcsa_carrier.get("allowed_to_operate"),
            "safety_rating": fmcsa_carrier.get("safety_rating"),
            "power_units": fmcsa_carrier.get("total_power_units"),
            "drivers": fmcsa_carrier.get("total_drivers")
        },

        "issues": issues
    }


carrier_verification = compare_carrier_claim_to_fmcsa(
    vlcp_shipment,
    carrier_evidence
)

print("VLCP CARRIER CLAIM VERIFICATION")
print("=" * 42)

print(
    f"Status: "
    f"{carrier_verification['verification_status']}"
)

print(
    f"USDOT match: "
    f"{carrier_verification['dot_match']}"
)

print(
    f"Carrier name match: "
    f"{carrier_verification['name_match']}"
)

if carrier_verification["issues"]:
    print("\nISSUES")

    for issue in carrier_verification["issues"]:
        print(f"  ⚠ {issue}")
else:
    print("\n✓ No carrier identity conflicts found")

In [ ]:
# CELL 11 — Update VLCP shipment with carrier verification result
# Dynamic: works with any shipment and carrier verification result.

def apply_carrier_verification_to_shipment(
    vlcp_shipment,
    carrier_verification
):
    """
    Write carrier verification findings back into
    the VLCP shipment record.
    """

    status = carrier_verification.get(
        "verification_status",
        "UNVERIFIED"
    )

    # Update overall carrier verification flag
    vlcp_shipment["verification"]["carrier_verified"] = (
        status == "VERIFIED"
    )

    # Preserve current shipment-level status
    vlcp_shipment["verification"]["verification_status"] = status

    # Add detailed carrier verification section
    vlcp_shipment["verification"]["carrier_identity"] = {
        "status": status,

        "dot_match": carrier_verification.get(
            "dot_match"
        ),

        "name_match": carrier_verification.get(
            "name_match"
        ),

        "claimed": carrier_verification.get(
            "claimed"
        ),

        "verified": carrier_verification.get(
            "verified"
        ),

        "issues": carrier_verification.get(
            "issues",
            []
        ),

        "verified_at_utc": datetime.now(
            UTC
        ).isoformat(),

        "evidence_source": "FMCSA QCMobile"
    }

    return vlcp_shipment


vlcp_shipment = apply_carrier_verification_to_shipment(
    vlcp_shipment,
    carrier_verification
)

print("VLCP SHIPMENT VERIFICATION UPDATE")
print("=" * 42)

print(
    "Carrier verified:",
    vlcp_shipment["verification"]["carrier_verified"]
)

print(
    "Verification status:",
    vlcp_shipment["verification"]["verification_status"]
)

print(
    "USDOT match:",
    vlcp_shipment["verification"]["carrier_identity"]["dot_match"]
)

print(
    "Carrier name match:",
    vlcp_shipment["verification"]["carrier_identity"]["name_match"]
)

if vlcp_shipment["verification"]["carrier_identity"]["issues"]:
    print("\nISSUES")

    for issue in vlcp_shipment[
        "verification"
    ]["carrier_identity"]["issues"]:

        print(f"  ⚠ {issue}")

In [ ]:
# CELL 12 — Human-readable VLCP shipment verification report
# Dynamic: works with any normalized and verified VLCP shipment.

def print_vlcp_shipment_report(vlcp_shipment):
    """
    Print a clean, human-readable VLCP shipment verification report.
    """

    document = vlcp_shipment.get("document", {})
    shipper = vlcp_shipment.get("shipper", {})
    receiver = vlcp_shipment.get("receiver", {})
    carrier = vlcp_shipment.get("carrier", {})
    equipment = vlcp_shipment.get("equipment", {})
    freight = vlcp_shipment.get("freight", {})
    pickup = vlcp_shipment.get("pickup", {})
    delivery = vlcp_shipment.get("delivery", {})
    verification = vlcp_shipment.get("verification", {})
    carrier_identity = verification.get("carrier_identity", {})

    status = verification.get("verification_status", "UNVERIFIED")

    print()
    print("=" * 68)
    print("VLCP SHIPMENT VERIFICATION REPORT")
    print("=" * 68)

    # --------------------------------------------------
    # OVERALL STATUS
    # --------------------------------------------------
    print(f"\nOverall Status: {status}")

    if status == "VERIFIED":
        print("Result: ✓ Carrier identity verified")

    elif status == "CONFLICT":
        print("Result: ⚠ Verification conflict detected")

    elif status == "FAILED":
        print("Result: ✗ Verification failed")

    else:
        print("Result: ? Verification incomplete")

    # --------------------------------------------------
    # DOCUMENT
    # --------------------------------------------------
    print("\nDOCUMENT")
    print("-" * 68)

    print(
        f"BOL Number:      "
        f"{document.get('bol_number') or 'Not provided'}"
    )

    print(
        f"Shipment ID:     "
        f"{document.get('shipment_id') or 'Not provided'}"
    )

    print(
        f"Source:          "
        f"{document.get('source') or 'Unknown'}"
    )

    print(
        f"Environment:     "
        f"{document.get('source_environment') or 'Unknown'}"
    )

    # --------------------------------------------------
    # ROUTE
    # --------------------------------------------------
    print("\nROUTE")
    print("-" * 68)

    print(
        "Shipper:         "
        f"{shipper.get('name') or 'Not provided'}"
    )

    print(
        "Origin:          "
        f"{shipper.get('city') or 'Unknown'}, "
        f"{shipper.get('state') or 'Unknown'}"
    )

    print(
        "Receiver:        "
        f"{receiver.get('name') or 'Not provided'}"
    )

    print(
        "Destination:     "
        f"{receiver.get('city') or 'Unknown'}, "
        f"{receiver.get('state') or 'Unknown'}"
    )

    # --------------------------------------------------
    # CARRIER CLAIM
    # --------------------------------------------------
    print("\nCARRIER CLAIM")
    print("-" * 68)

    print(
        f"Carrier Name:    "
        f"{carrier.get('name') or 'Not provided'}"
    )

    print(
        f"USDOT:           "
        f"{carrier.get('usdot') or 'Not provided'}"
    )

    print(
        f"MC Number:       "
        f"{carrier.get('mc_number') or 'Not provided'}"
    )

    # --------------------------------------------------
    # FMCSA EVIDENCE
    # --------------------------------------------------
    print("\nFMCSA EVIDENCE")
    print("-" * 68)

    verified = carrier_identity.get("verified", {})

    print(
        f"Legal Name:      "
        f"{verified.get('legal_name') or 'Not available'}"
    )

    print(
        f"USDOT:           "
        f"{verified.get('usdot') or 'Not available'}"
    )

    print(
        f"Allowed Operate: "
        f"{verified.get('allowed_to_operate') or 'Not available'}"
    )

    print(
        f"Safety Rating:   "
        f"{verified.get('safety_rating') or 'Not available'}"
    )

    print(
        f"Power Units:     "
        f"{verified.get('power_units') if verified.get('power_units') is not None else 'Not available'}"
    )

    print(
        f"Drivers:         "
        f"{verified.get('drivers') if verified.get('drivers') is not None else 'Not available'}"
    )

    # --------------------------------------------------
    # IDENTITY COMPARISON
    # --------------------------------------------------
    print("\nIDENTITY VERIFICATION")
    print("-" * 68)

    dot_match = carrier_identity.get("dot_match")
    name_match = carrier_identity.get("name_match")

    print(
        "USDOT Match:     "
        + ("✓ MATCH" if dot_match else "✗ NO MATCH")
    )

    print(
        "Name Match:      "
        + ("✓ MATCH" if name_match else "⚠ CONFLICT")
    )

    # --------------------------------------------------
    # EQUIPMENT
    # --------------------------------------------------
    print("\nEQUIPMENT CLAIM")
    print("-" * 68)

    print(
        f"Tractor Number:  "
        f"{equipment.get('tractor_number') or 'Not provided'}"
    )

    print(
        f"Trailer Number:  "
        f"{equipment.get('trailer_number') or 'Not provided'}"
    )

    print(
        f"Equipment Type:  "
        f"{equipment.get('equipment_type') or 'Not provided'}"
    )

    # --------------------------------------------------
    # FREIGHT
    # --------------------------------------------------
    print("\nFREIGHT")
    print("-" * 68)

    print(
        f"Commodity:       "
        f"{freight.get('commodity') or 'Not provided'}"
    )

    print(
        f"Weight:          "
        f"{freight.get('weight_lbs') or 'Not provided'} lbs"
    )

    print(
        f"Pieces:          "
        f"{freight.get('pieces') or 'Not provided'}"
    )

    # --------------------------------------------------
    # TIMING
    # --------------------------------------------------
    print("\nTIMING")
    print("-" * 68)

    print(
        f"Pickup Scheduled:  "
        f"{pickup.get('scheduled') or 'Not provided'}"
    )

    print(
        f"Pickup Actual:     "
        f"{pickup.get('actual') or 'Not provided'}"
    )

    print(
        f"Delivery Scheduled:"
        f" {delivery.get('scheduled') or 'Not provided'}"
    )

    print(
        f"Delivery Actual:   "
        f"{delivery.get('actual') or 'Not provided'}"
    )

    # --------------------------------------------------
    # ISSUES
    # --------------------------------------------------
    issues = carrier_identity.get("issues", [])

    print("\nISSUES")
    print("-" * 68)

    if issues:
        for issue in issues:
            print(f"⚠ {issue}")
    else:
        print("✓ No carrier identity conflicts found")

    # --------------------------------------------------
    # EVIDENCE
    # --------------------------------------------------
    print("\nEVIDENCE")
    print("-" * 68)

    print(
        f"Evidence Source: "
        f"{carrier_identity.get('evidence_source') or 'Not available'}"
    )

    print(
        f"Verified At UTC: "
        f"{carrier_identity.get('verified_at_utc') or 'Not available'}"
    )

    print("=" * 68)
    print("END OF VLCP REPORT")
    print("=" * 68)
    print()


# --------------------------------------------------
# GENERATE REPORT
# --------------------------------------------------

print_vlcp_shipment_report(
    vlcp_shipment
)

In [ ]:
# CELL 13 — Generic VLCP equipment verification
# Dynamic: works with any shipment and any equipment evidence set.

def normalize_equipment_value(value):
    """
    Normalize equipment identifiers for comparison.
    """
    if value is None:
        return ""

    return (
        str(value)
        .upper()
        .strip()
        .replace(" ", "")
        .replace("-", "")
    )


def verify_equipment_claim(
    vlcp_shipment,
    equipment_evidence
):
    """
    Compare equipment claims in a shipment against
    externally retrieved equipment evidence.

    equipment_evidence should be a dictionary like:

    {
        "tractors": [...],
        "trailers": [...],
        "tractor_vins": [...],
        "trailer_vins": [...],
        "source": "FMCSA inspection evidence"
    }

    No carrier or equipment number is hard-coded.
    """

    shipment_equipment = vlcp_shipment.get(
        "equipment",
        {}
    )

    claimed_tractor = shipment_equipment.get(
        "tractor_number"
    )

    claimed_trailer = shipment_equipment.get(
        "trailer_number"
    )

    claimed_type = shipment_equipment.get(
        "equipment_type"
    )

    # Normalize claims
    tractor_claim_norm = normalize_equipment_value(
        claimed_tractor
    )

    trailer_claim_norm = normalize_equipment_value(
        claimed_trailer
    )

    # Normalize evidence
    known_tractors = {
        normalize_equipment_value(x)
        for x in equipment_evidence.get(
            "tractors",
            []
        )
        if x
    }

    known_trailers = {
        normalize_equipment_value(x)
        for x in equipment_evidence.get(
            "trailers",
            []
        )
        if x
    }

    # -----------------------------------
    # TRACTOR RESULT
    # -----------------------------------
    if not claimed_tractor:
        tractor_status = "UNVERIFIED"

    elif tractor_claim_norm in known_tractors:
        tractor_status = "VERIFIED"

    else:
        tractor_status = "NOT_FOUND"

    # -----------------------------------
    # TRAILER RESULT
    # -----------------------------------
    if not claimed_trailer:
        trailer_status = "UNVERIFIED"

    elif trailer_claim_norm in known_trailers:
        trailer_status = "VERIFIED"

    else:
        trailer_status = "NOT_FOUND"

    # -----------------------------------
    # OVERALL EQUIPMENT STATUS
    # -----------------------------------
    if (
        tractor_status == "VERIFIED"
        and trailer_status == "VERIFIED"
    ):
        overall_status = "VERIFIED"

    elif (
        tractor_status == "UNVERIFIED"
        and trailer_status == "UNVERIFIED"
    ):
        overall_status = "UNVERIFIED"

    else:
        overall_status = "PARTIAL_OR_CONFLICT"

    issues = []

    if tractor_status == "NOT_FOUND":
        issues.append(
            f"Claimed tractor '{claimed_tractor}' "
            "was not found in available equipment evidence."
        )

    if trailer_status == "NOT_FOUND":
        issues.append(
            f"Claimed trailer '{claimed_trailer}' "
            "was not found in available equipment evidence."
        )

    return {
        "verification_status": overall_status,

        "claimed": {
            "tractor_number": claimed_tractor,
            "trailer_number": claimed_trailer,
            "equipment_type": claimed_type
        },

        "tractor_status": tractor_status,
        "trailer_status": trailer_status,

        "evidence_source": equipment_evidence.get(
            "source"
        ),

        "issues": issues
    }

In [ ]:
equipment_evidence = {
    "tractors": [],
    "trailers": [],
    "tractor_vins": [],
    "trailer_vins": [],
    "source": "No equipment source connected yet"
}

equipment_verification = verify_equipment_claim(
    vlcp_shipment,
    equipment_evidence
)

print("VLCP EQUIPMENT VERIFICATION")
print("=" * 42)

print(
    "Status:",
    equipment_verification["verification_status"]
)

print(
    "Tractor:",
    equipment_verification["tractor_status"]
)

print(
    "Trailer:",
    equipment_verification["trailer_status"]
)

if equipment_verification["issues"]:
    print("\nISSUES")

    for issue in equipment_verification["issues"]:
        print(f"  ⚠ {issue}")

In [ ]:
# CELL 14 — Connect existing FMCSA inspection-unit logic to VLCP
# Dynamic: uses the USDOT from the current shipment.

def build_equipment_evidence_from_inspections(usdot):
    """
    Use the existing VLCP/FMCSA inspection-unit retrieval logic
    to create standardized equipment evidence for any USDOT.

    Requires the earlier working function:
        get_inspection_units(usdot)

    Returns:
        {
            "tractors": [...],
            "trailers": [...],
            "tractor_vins": [...],
            "trailer_vins": [...],
            "trailer_plates": [...],
            "source": ...
        }
    """

    usdot = str(usdot).strip()

    if not usdot:
        raise ValueError("USDOT number is required.")

    if not usdot.isdigit():
        raise ValueError(
            f"USDOT must contain only numbers. Received: {usdot}"
        )

    # --------------------------------------------------
    # MAKE SURE THE EXISTING FUNCTION IS AVAILABLE
    # --------------------------------------------------
    if "get_inspection_units" not in globals():
        return {
            "success": False,
            "status": "INSPECTION_FUNCTION_NOT_LOADED",
            "usdot": usdot,
            "tractors": [],
            "trailers": [],
            "tractor_vins": [],
            "trailer_vins": [],
            "trailer_plates": [],
            "source": "FMCSA inspection-unit evidence",
            "error": (
                "get_inspection_units() is not loaded in this notebook yet."
            )
        }

    # --------------------------------------------------
    # RETRIEVE INSPECTION UNITS
    # --------------------------------------------------
    try:
        inspection_units = get_inspection_units(usdot)

    except Exception as exc:
        return {
            "success": False,
            "status": "INSPECTION_LOOKUP_FAILED",
            "usdot": usdot,
            "tractors": [],
            "trailers": [],
            "tractor_vins": [],
            "trailer_vins": [],
            "trailer_plates": [],
            "source": "FMCSA inspection-unit evidence",
            "error": str(exc)
        }

    # --------------------------------------------------
    # STANDARDIZE RESULTS
    # --------------------------------------------------
    tractors = set()
    trailers = set()

    tractor_vins = set()
    trailer_vins = set()
    trailer_plates = set()

    # inspection_units may be a list of dicts
    if isinstance(inspection_units, list):

        for unit in inspection_units:

            if not isinstance(unit, dict):
                continue

            unit_type = str(
                unit.get("unit_type")
                or unit.get("unitType")
                or unit.get("type")
                or ""
            ).upper()

            unit_number = (
                unit.get("unit_number")
                or unit.get("unitNumber")
                or unit.get("vehicle_number")
                or unit.get("vehicleNumber")
            )

            vin = (
                unit.get("vin")
                or unit.get("vehicleVin")
                or unit.get("vehicle_vin")
            )

            plate = (
                unit.get("license_plate")
                or unit.get("licensePlate")
                or unit.get("plate")
            )

            # ------------------------------------------
            # TRACTOR / POWER UNIT
            # ------------------------------------------
            if any(
                word in unit_type
                for word in [
                    "TRACTOR",
                    "TRUCK",
                    "POWER",
                    "STRAIGHT"
                ]
            ):

                if unit_number:
                    tractors.add(str(unit_number).strip())

                if vin:
                    tractor_vins.add(str(vin).strip())

            # ------------------------------------------
            # TRAILER
            # ------------------------------------------
            if "TRAILER" in unit_type:

                if unit_number:
                    trailers.add(str(unit_number).strip())

                if vin:
                    trailer_vins.add(str(vin).strip())

                if plate:
                    trailer_plates.add(str(plate).strip())

    return {
        "success": True,
        "status": "EVIDENCE_BUILT",
        "usdot": usdot,

        "tractors": sorted(tractors),
        "trailers": sorted(trailers),

        "tractor_vins": sorted(tractor_vins),
        "trailer_vins": sorted(trailer_vins),
        "trailer_plates": sorted(trailer_plates),

        "source": "FMCSA inspection-unit evidence"
    }


# --------------------------------------------------
# BUILD EQUIPMENT EVIDENCE FOR CURRENT BOL CARRIER
# --------------------------------------------------

equipment_evidence = build_equipment_evidence_from_inspections(
    carrier_dot
)

print("VLCP EQUIPMENT EVIDENCE SOURCE")
print("=" * 42)

print("USDOT:", equipment_evidence.get("usdot"))
print("Status:", equipment_evidence.get("status"))

if equipment_evidence.get("success"):

    print(
        "Tractors found:",
        len(equipment_evidence.get("tractors", []))
    )

    print(
        "Trailers found:",
        len(equipment_evidence.get("trailers", []))
    )

    print(
        "Tractor VINs found:",
        len(equipment_evidence.get("tractor_vins", []))
    )

    print(
        "Trailer VINs found:",
        len(equipment_evidence.get("trailer_vins", []))
    )

else:

    print(
        "⚠",
        equipment_evidence.get("error")
    )

In [ ]:
# CELL 15 — VLCP inspection-unit evidence adapter
# Dynamic: accepts inspection-unit records for any USDOT.

def convert_inspection_units_to_vlcp_evidence(
    usdot,
    inspection_units,
    source_name="FMCSA Inspections Per Unit"
):
    """
    Convert inspection-unit records into standardized
    VLCP equipment evidence.

    Works with any USDOT and any list of inspection-unit
    dictionaries retrieved from a supported source.
    """

    usdot = str(usdot).strip()

    if not usdot:
        raise ValueError("USDOT number is required.")

    if not usdot.isdigit():
        raise ValueError(
            f"USDOT must contain only numbers. Received: {usdot}"
        )

    if inspection_units is None:
        inspection_units = []

    if not isinstance(inspection_units, list):
        raise TypeError(
            "inspection_units must be a list of records."
        )

    tractors = set()
    trailers = set()

    tractor_vins = set()
    trailer_vins = set()

    tractor_plates = set()
    trailer_plates = set()

    records_processed = 0

    for unit in inspection_units:

        if not isinstance(unit, dict):
            continue

        records_processed += 1

        # -----------------------------------
        # FLEXIBLE FIELD EXTRACTION
        # -----------------------------------
        unit_type = str(
            unit.get("unit_type")
            or unit.get("unitType")
            or unit.get("vehicle_type")
            or unit.get("vehicleType")
            or unit.get("UNIT_TYPE_DESC")
            or ""
        ).upper().strip()

        unit_number = (
            unit.get("unit_number")
            or unit.get("unitNumber")
            or unit.get("company_number")
            or unit.get("companyNumber")
            or unit.get("UNIT_NUMBER")
        )

        vin = (
            unit.get("vin")
            or unit.get("VIN")
            or unit.get("vehicle_vin")
            or unit.get("vehicleVin")
        )

        plate = (
            unit.get("license_plate")
            or unit.get("licensePlate")
            or unit.get("plate")
            or unit.get("LICENSE_PLATE")
        )

        # -----------------------------------
        # CLASSIFY TRACTOR / POWER UNIT
        # -----------------------------------
        is_tractor = any(
            keyword in unit_type
            for keyword in [
                "TRACTOR",
                "TRUCK",
                "POWER UNIT",
                "STRAIGHT TRUCK"
            ]
        )

        # -----------------------------------
        # CLASSIFY TRAILER
        # -----------------------------------
        is_trailer = any(
            keyword in unit_type
            for keyword in [
                "TRAILER",
                "SEMI TRAILER",
                "SEMITRAILER"
            ]
        )

        if is_tractor:

            if unit_number:
                tractors.add(
                    str(unit_number).strip()
                )

            if vin:
                tractor_vins.add(
                    str(vin).strip()
                )

            if plate:
                tractor_plates.add(
                    str(plate).strip()
                )

        elif is_trailer:

            if unit_number:
                trailers.add(
                    str(unit_number).strip()
                )

            if vin:
                trailer_vins.add(
                    str(vin).strip()
                )

            if plate:
                trailer_plates.add(
                    str(plate).strip()
                )

    return {
        "success": True,
        "status": "EVIDENCE_BUILT",

        "usdot": usdot,

        "records_processed": records_processed,

        "tractors": sorted(tractors),
        "trailers": sorted(trailers),

        "tractor_vins": sorted(tractor_vins),
        "trailer_vins": sorted(trailer_vins),

        "tractor_plates": sorted(tractor_plates),
        "trailer_plates": sorted(trailer_plates),

        "source": source_name
    }


print("✓ VLCP inspection-unit evidence adapter loaded")

In [ ]:
# CELL 16 — Retrieve real FMCSA inspection-unit evidence for any USDOT
# Dynamic: works with whatever USDOT is passed in.

import requests


VEHICLE_INSPECTION_DATASET = "fx4q-ay7w"
INSPECTION_UNIT_DATASET = "wt8s-2hbx"

SOCRATA_DOMAIN = "https://data.transportation.gov"


def _extract_socrata_rows(payload):
    """
    Handle common Socrata API response shapes.
    """
    if isinstance(payload, list):
        return payload

    if isinstance(payload, dict):
        for key in ["data", "results", "rows"]:
            if isinstance(payload.get(key), list):
                return payload[key]

    return []


def _socrata_query(dataset_id, query, page_size=5000):
    """
    Run a query against a DOT Socrata dataset.
    No app token required for light public testing.
    """

    url = (
        f"{SOCRATA_DOMAIN}/api/v3/views/"
        f"{dataset_id}/query.json"
    )

    body = {
        "query": query,
        "page": {
            "pageNumber": 1,
            "pageSize": page_size
        },
        "includeSynthetic": False
    }

    response = requests.post(
        url,
        json=body,
        timeout=60
    )

    response.raise_for_status()

    payload = response.json()

    return _extract_socrata_rows(payload)


def get_fmcsa_inspection_units(usdot):
    """
    Retrieve inspection-unit evidence for any USDOT.

    Flow:
      1. Find inspection IDs for the USDOT.
      2. Retrieve unit records tied to those inspection IDs.
      3. Return raw inspection-unit rows.
    """

    usdot = str(usdot).strip()

    if not usdot:
        raise ValueError("USDOT number is required.")

    if not usdot.isdigit():
        raise ValueError(
            f"USDOT must contain only numbers. Received: {usdot}"
        )

    # --------------------------------------------------
    # STEP 1 — GET INSPECTION IDs FOR THIS USDOT
    # --------------------------------------------------

    inspection_query = f"""
        SELECT inspection_id, dot_number
        WHERE dot_number = '{usdot}'
    """

    inspection_rows = _socrata_query(
        VEHICLE_INSPECTION_DATASET,
        inspection_query,
        page_size=5000
    )

    inspection_ids = sorted({
        str(row.get("inspection_id")).strip()
        for row in inspection_rows
        if row.get("inspection_id")
    })

    if not inspection_ids:
        return {
            "success": True,
            "status": "NO_INSPECTIONS_FOUND",
            "usdot": usdot,
            "inspection_count": 0,
            "unit_count": 0,
            "inspection_units": []
        }

    # --------------------------------------------------
    # STEP 2 — GET UNIT RECORDS FOR THOSE INSPECTIONS
    # --------------------------------------------------
    #
    # Query in batches so the request does not become
    # too large for carriers with many inspections.
    # --------------------------------------------------

    all_unit_rows = []

    batch_size = 100

    for start in range(
        0,
        len(inspection_ids),
        batch_size
    ):

        batch = inspection_ids[
            start:start + batch_size
        ]

        quoted_ids = ",".join(
            f"'{inspection_id}'"
            for inspection_id in batch
        )

        unit_query = f"""
            SELECT
                inspection_id,
                insp_unit_id,
                insp_unit_type_id,
                insp_unit_number,
                insp_unit_make,
                insp_unit_company,
                insp_unit_license,
                insp_unit_license_state,
                insp_unit_vehicle_id_number,
                insp_unit_decal,
                insp_unit_decal_number
            WHERE inspection_id IN ({quoted_ids})
        """

        unit_rows = _socrata_query(
            INSPECTION_UNIT_DATASET,
            unit_query,
            page_size=5000
        )

        all_unit_rows.extend(
            unit_rows
        )

    return {
        "success": True,
        "status": "FOUND",
        "usdot": usdot,
        "inspection_count": len(inspection_ids),
        "unit_count": len(all_unit_rows),
        "inspection_units": all_unit_rows,
        "source": "FMCSA / DOT Public Data Portal"
    }


# --------------------------------------------------
# RUN FOR THE CARRIER FROM THE CURRENT BOL
# --------------------------------------------------

inspection_result = get_fmcsa_inspection_units(
    carrier_dot
)

print("VLCP FMCSA INSPECTION LOOKUP")
print("=" * 42)

print(
    "USDOT:",
    inspection_result.get("usdot")
)

print(
    "Status:",
    inspection_result.get("status")
)

print(
    "Inspections found:",
    inspection_result.get("inspection_count")
)

print(
    "Inspection units found:",
    inspection_result.get("unit_count")
)

In [ ]:
# CELL 17 — Profile raw FMCSA inspection-unit evidence
# Purpose:
#   Inspect the real data returned by Cell 16 before building
#   VLCP equipment normalization and verification logic.
#
# This cell does NOT modify, normalize, or score the evidence.

from collections import Counter


def profile_fmcsa_inspection_units(inspection_result, sample_size=10):
    """
    Profile raw FMCSA inspection-unit evidence returned by
    get_fmcsa_inspection_units().

    This is a diagnostic/evidence-discovery function only.
    It does not classify or verify equipment.
    """

    if not isinstance(inspection_result, dict):
        raise TypeError(
            "inspection_result must be a dictionary."
        )

    status = inspection_result.get("status")
    usdot = inspection_result.get("usdot")
    rows = inspection_result.get("inspection_units", [])

    print("VLCP FMCSA INSPECTION-UNIT EVIDENCE PROFILE")
    print("=" * 60)
    print("USDOT:", usdot)
    print("Status:", status)
    print("Inspection count:", inspection_result.get("inspection_count"))
    print("Unit count:", len(rows))
    print()

    if not rows:
        print("No inspection-unit evidence available to profile.")
        return

    # --------------------------------------------------
    # 1. DISCOVER ACTUAL FIELDS RETURNED
    # --------------------------------------------------

    all_fields = sorted({
        key
        for row in rows
        if isinstance(row, dict)
        for key in row.keys()
    })

    print("FIELDS RETURNED")
    print("-" * 60)

    for field in all_fields:
        print(field)

    # --------------------------------------------------
    # 2. PROFILE UNIT TYPES
    # --------------------------------------------------

    unit_types = Counter(
        str(row.get("insp_unit_type_id")).strip()
        for row in rows
        if row.get("insp_unit_type_id") not in (None, "")
    )

    print()
    print("UNIT TYPE DISTRIBUTION")
    print("-" * 60)

    if unit_types:
        for unit_type, count in unit_types.most_common():
            print(f"Type {unit_type}: {count}")
    else:
        print("No unit-type values found.")

    # --------------------------------------------------
    # 3. FIELD COMPLETENESS
    # --------------------------------------------------

    important_fields = [
        "insp_unit_id",
        "insp_unit_type_id",
        "insp_unit_number",
        "insp_unit_make",
        "insp_unit_company",
        "insp_unit_license",
        "insp_unit_license_state",
        "insp_unit_vehicle_id_number",
        "insp_unit_decal",
        "insp_unit_decal_number",
    ]

    print()
    print("FIELD COMPLETENESS")
    print("-" * 60)

    total = len(rows)

    for field in important_fields:

        populated = sum(
            1
            for row in rows
            if row.get(field) not in (None, "")
        )

        percentage = (
            (populated / total) * 100
            if total
            else 0
        )

        print(
            f"{field:<30} "
            f"{populated:>5}/{total:<5} "
            f"({percentage:6.2f}%)"
        )

    # --------------------------------------------------
    # 4. UNIQUE MAKES
    # --------------------------------------------------

    makes = Counter(
        str(row.get("insp_unit_make")).strip().upper()
        for row in rows
        if row.get("insp_unit_make") not in (None, "")
    )

    print()
    print("MOST COMMON EQUIPMENT MAKES")
    print("-" * 60)

    if makes:
        for make, count in makes.most_common(15):
            print(f"{make:<20} {count}")
    else:
        print("No equipment make data found.")

    # --------------------------------------------------
    # 5. SAMPLE RAW RECORDS
    # --------------------------------------------------

    print()
    print(f"SAMPLE RAW RECORDS (first {min(sample_size, total)})")
    print("-" * 60)

    for index, row in enumerate(rows[:sample_size], start=1):

        print()
        print(f"RECORD {index}")

        for field in important_fields:
            print(
                f"  {field}: "
                f"{row.get(field)}"
            )

    # --------------------------------------------------
    # 6. BASIC DATA-QUALITY CHECK
    # --------------------------------------------------

    rows_with_vin = sum(
        1
        for row in rows
        if row.get("insp_unit_vehicle_id_number")
        not in (None, "")
    )

    rows_with_unit_number = sum(
        1
        for row in rows
        if row.get("insp_unit_number")
        not in (None, "")
    )

    rows_with_license = sum(
        1
        for row in rows
        if row.get("insp_unit_license")
        not in (None, "")
    )

    print()
    print("EVIDENCE AVAILABILITY SUMMARY")
    print("-" * 60)

    print(
        "Records with VIN:",
        rows_with_vin
    )

    print(
        "Records with unit number:",
        rows_with_unit_number
    )

    print(
        "Records with license:",
        rows_with_license
    )

    print()
    print("PROFILE COMPLETE")
    print("=" * 60)


# --------------------------------------------------
# RUN PROFILE AGAINST CELL 16 RESULT
# --------------------------------------------------

profile_fmcsa_inspection_units(
    inspection_result,
    sample_size=10
)

In [ ]:
# VLCP RUNTIME DIAGNOSTIC
# Checks whether Cell 16 dependencies still exist in memory.

required_objects = [
    "carrier_dot",
    "_extract_socrata_rows",
    "_socrata_query",
    "get_fmcsa_inspection_units",
    "inspection_result",
]

print("VLCP RUNTIME STATE")
print("=" * 45)

for name in required_objects:
    exists = name in globals()

    print(
        f"{'✓' if exists else '✗'} {name}"
    )

In [ ]:
# CELL 18 — Normalize and deduplicate FMCSA inspection-unit evidence
#
# Purpose:
#   Convert raw FMCSA inspection-unit rows into a consistent
#   VLCP evidence structure and identify repeated equipment.
#
# IMPORTANT:
#   This cell does NOT yet translate insp_unit_type_id into
#   "tractor", "trailer", etc. We will only do that after
#   verifying FMCSA's authoritative type definitions.

from collections import Counter, defaultdict


def _clean_text(value):
    """
    Normalize a text value without inventing missing data.
    """

    if value is None:
        return None

    value = str(value).strip()

    if not value:
        return None

    return value.upper()


def _clean_vin(value):
    """
    Normalize VIN-like identifiers for comparison.

    This does not determine whether the VIN is valid.
    """

    value = _clean_text(value)

    if value is None:
        return None

    return "".join(
        char for char in value
        if char.isalnum()
    )


def _clean_identifier(value):
    """
    Normalize equipment/unit identifiers for comparison.
    """

    value = _clean_text(value)

    if value is None:
        return None

    return "".join(
        char for char in value
        if char.isalnum()
    )


def normalize_fmcsa_inspection_units(inspection_result):
    """
    Normalize raw FMCSA inspection-unit evidence.

    Returns one normalized record for every raw inspection-unit
    observation.

    No equipment-type assumptions are made here.
    """

    if not isinstance(inspection_result, dict):
        raise TypeError(
            "inspection_result must be a dictionary."
        )

    raw_rows = inspection_result.get(
        "inspection_units",
        []
    )

    normalized_rows = []

    for row in raw_rows:

        if not isinstance(row, dict):
            continue

        normalized_rows.append({
            "source": "FMCSA / DOT Public Data Portal",

            "usdot": inspection_result.get("usdot"),

            "inspection_id": _clean_text(
                row.get("inspection_id")
            ),

            "inspection_unit_id": _clean_text(
                row.get("insp_unit_id")
            ),

            "unit_type_id": _clean_text(
                row.get("insp_unit_type_id")
            ),

            "unit_number_raw": _clean_text(
                row.get("insp_unit_number")
            ),

            "unit_number_normalized": _clean_identifier(
                row.get("insp_unit_number")
            ),

            "company_unit_id_raw": _clean_text(
                row.get("insp_unit_company")
            ),

            "company_unit_id_normalized": _clean_identifier(
                row.get("insp_unit_company")
            ),

            "make_raw": _clean_text(
                row.get("insp_unit_make")
            ),

            "license_raw": _clean_text(
                row.get("insp_unit_license")
            ),

            "license_normalized": _clean_identifier(
                row.get("insp_unit_license")
            ),

            "license_state": _clean_text(
                row.get("insp_unit_license_state")
            ),

            "vin_raw": _clean_text(
                row.get("insp_unit_vehicle_id_number")
            ),

            "vin_normalized": _clean_vin(
                row.get("insp_unit_vehicle_id_number")
            ),

            "decal": _clean_text(
                row.get("insp_unit_decal")
            ),

            "decal_number": _clean_text(
                row.get("insp_unit_decal_number")
            ),
        })

    return normalized_rows


def build_unique_equipment_index(normalized_rows):
    """
    Group repeated FMCSA inspection observations into likely
    unique equipment records.

    VIN is used as the primary identity key when available.

    If VIN is unavailable, state + license is used as a
    secondary evidence key.

    Records lacking both are preserved rather than discarded.
    """

    equipment_index = defaultdict(list)

    unidentified_counter = 0

    for row in normalized_rows:

        vin = row.get("vin_normalized")
        plate = row.get("license_normalized")
        state = row.get("license_state")

        if vin:
            identity_key = f"VIN:{vin}"

        elif plate and state:
            identity_key = (
                f"PLATE:{state}:{plate}"
            )

        else:
            unidentified_counter += 1

            identity_key = (
                f"UNIDENTIFIED:"
                f"{unidentified_counter}"
            )

        equipment_index[
            identity_key
        ].append(row)

    return dict(equipment_index)


def summarize_equipment_evidence(
    normalized_rows,
    equipment_index
):
    """
    Print a diagnostic summary of normalized and deduplicated
    FMCSA equipment evidence.
    """

    print("VLCP NORMALIZED EQUIPMENT EVIDENCE")
    print("=" * 60)

    print(
        "Raw inspection-unit observations:",
        len(normalized_rows)
    )

    print(
        "Likely unique equipment identities:",
        len(equipment_index)
    )

    duplicate_observations = (
        len(normalized_rows)
        - len(equipment_index)
    )

    print(
        "Repeated inspection observations:",
        duplicate_observations
    )

    # --------------------------------------------------
    # UNIT TYPES
    # --------------------------------------------------

    type_counts = Counter(
        row.get("unit_type_id")
        for row in normalized_rows
        if row.get("unit_type_id")
    )

    print()
    print("UNIT TYPE IDS")
    print("-" * 60)

    for unit_type, count in type_counts.most_common():

        print(
            f"Type {unit_type}: "
            f"{count} observations"
        )

    # --------------------------------------------------
    # IDENTITY STRENGTH
    # --------------------------------------------------

    vin_identity_count = sum(
        1
        for key in equipment_index
        if key.startswith("VIN:")
    )

    plate_identity_count = sum(
        1
        for key in equipment_index
        if key.startswith("PLATE:")
    )

    unidentified_count = sum(
        1
        for key in equipment_index
        if key.startswith("UNIDENTIFIED:")
    )

    print()
    print("IDENTITY BASIS")
    print("-" * 60)

    print(
        "VIN-based identities:",
        vin_identity_count
    )

    print(
        "Plate-based fallback identities:",
        plate_identity_count
    )

    print(
        "Unidentified identities:",
        unidentified_count
    )

    # --------------------------------------------------
    # REPEATED EQUIPMENT
    # --------------------------------------------------

    repeated = sorted(
        [
            (key, observations)
            for key, observations
            in equipment_index.items()
            if len(observations) > 1
        ],
        key=lambda item: len(item[1]),
        reverse=True
    )

    print()
    print("MOST FREQUENTLY OBSERVED EQUIPMENT")
    print("-" * 60)

    if not repeated:

        print(
            "No repeated equipment identities detected."
        )

    else:

        for key, observations in repeated[:10]:

            example = observations[0]

            print()
            print(
                "Identity:",
                key
            )

            print(
                "Inspection observations:",
                len(observations)
            )

            print(
                "Type ID:",
                example.get("unit_type_id")
            )

            print(
                "Make:",
                example.get("make_raw")
            )

            print(
                "Company unit ID:",
                example.get(
                    "company_unit_id_raw"
                )
            )

            print(
                "Plate:",
                example.get("license_raw"),
                example.get("license_state")
            )

    print()
    print("NORMALIZATION COMPLETE")
    print("=" * 60)


# --------------------------------------------------
# EXECUTE CELL 18
# --------------------------------------------------

normalized_inspection_units = (
    normalize_fmcsa_inspection_units(
        inspection_result
    )
)

equipment_index = (
    build_unique_equipment_index(
        normalized_inspection_units
    )
)

summarize_equipment_evidence(
    normalized_inspection_units,
    equipment_index
)

In [ ]:
# CELL 19 — Analyze FMCSA unit-type evidence
#
# Purpose:
#   Examine the real characteristics associated with each
#   insp_unit_type_id before assigning human-readable
#   equipment classifications.
#
# IMPORTANT:
#   This cell DOES NOT assume what type IDs 9, 10, 11, or 14 mean.

from collections import Counter, defaultdict


def analyze_fmcsa_unit_types(normalized_rows):
    """
    Analyze observed characteristics for each FMCSA
    insp_unit_type_id.

    No semantic equipment classification is assigned here.
    """

    if not isinstance(normalized_rows, list):
        raise TypeError(
            "normalized_rows must be a list."
        )

    grouped = defaultdict(list)

    for row in normalized_rows:

        if not isinstance(row, dict):
            continue

        unit_type = row.get("unit_type_id")

        if unit_type:
            grouped[unit_type].append(row)

    print("VLCP FMCSA UNIT-TYPE ANALYSIS")
    print("=" * 65)

    for unit_type in sorted(
        grouped.keys(),
        key=lambda value: int(value)
        if str(value).isdigit()
        else str(value)
    ):

        rows = grouped[unit_type]

        print()
        print("=" * 65)
        print(
            f"UNIT TYPE ID: {unit_type}"
        )
        print(
            f"Observations: {len(rows)}"
        )
        print("-" * 65)

        # ----------------------------------------------
        # MAKES
        # ----------------------------------------------

        makes = Counter(
            row.get("make_raw")
            for row in rows
            if row.get("make_raw")
        )

        print()
        print("TOP MAKES")

        for make, count in makes.most_common(15):
            print(
                f"  {make:<20} {count}"
            )

        # ----------------------------------------------
        # VIN PREFIXES
        # ----------------------------------------------

        vin_prefixes = Counter()

        for row in rows:

            vin = row.get("vin_normalized")

            if vin and len(vin) >= 3:
                vin_prefixes[
                    vin[:3]
                ] += 1

        print()
        print("TOP VIN PREFIXES")

        for prefix, count in vin_prefixes.most_common(15):
            print(
                f"  {prefix:<10} {count}"
            )

        # ----------------------------------------------
        # COMPANY UNIT IDENTIFIERS
        # ----------------------------------------------

        company_ids = [
            row.get("company_unit_id_raw")
            for row in rows
            if row.get("company_unit_id_raw")
        ]

        print()
        print(
            "Company/unit identifiers present:",
            len(company_ids),
            "/",
            len(rows)
        )

        # ----------------------------------------------
        # LICENSE STATES
        # ----------------------------------------------

        states = Counter(
            row.get("license_state")
            for row in rows
            if row.get("license_state")
        )

        print()
        print("TOP LICENSE STATES")

        for state, count in states.most_common(10):
            print(
                f"  {state:<5} {count}"
            )

        # ----------------------------------------------
        # SAMPLE RECORDS
        # ----------------------------------------------

        print()
        print("SAMPLE RECORDS")

        for row in rows[:5]:

            print(
                "  ",
                {
                    "inspection_id":
                        row.get("inspection_id"),

                    "type_id":
                        row.get("unit_type_id"),

                    "make":
                        row.get("make_raw"),

                    "company_unit":
                        row.get(
                            "company_unit_id_raw"
                        ),

                    "plate":
                        row.get("license_raw"),

                    "state":
                        row.get("license_state"),

                    "vin":
                        row.get("vin_raw"),
                }
            )

    print()
    print("=" * 65)
    print("UNIT-TYPE ANALYSIS COMPLETE")


# --------------------------------------------------
# EXECUTE CELL 19
# --------------------------------------------------

analyze_fmcsa_unit_types(
    normalized_inspection_units
)

In [24]:
# CELL 19 — VLCP Multi-USDOT Equipment Pipeline Test Harness
#
# Purpose:
#   Establish reusable multi-carrier regression testing for VLCP.
#
# Current pipeline under test:
#
#   USDOT
#     ↓
#   FMCSA inspection acquisition
#     ↓
#   normalization
#     ↓
#   equipment identity/deduplication
#
# IMPORTANT:
#   Test USDOTs exist ONLY in this test harness.
#   Production functions remain fully dynamic.

from collections import Counter
import traceback
import time


# ==========================================================
# TEST CONFIGURATION
# ==========================================================

TEST_USDOTS = [
    "951224",   # Existing VLCP test carrier
    "80806",    # Previously verified VLCP carrier
    "2232566",  # Different historical VLCP data condition
]


# ==========================================================
# SINGLE-CARRIER TEST
# ==========================================================

def run_equipment_pipeline_test(usdot):
    """
    Execute the current VLCP inspection/equipment pipeline
    for one USDOT and validate internal consistency.

    Returns a structured test result instead of stopping
    the entire suite when one carrier fails.
    """

    usdot = str(usdot).strip()

    result = {
        "usdot": usdot,
        "pipeline_status": "NOT_RUN",
        "source_status": None,
        "inspection_count": 0,
        "raw_unit_count": 0,
        "normalized_count": 0,
        "unique_equipment_count": 0,
        "vin_identity_count": 0,
        "plate_identity_count": 0,
        "unidentified_count": 0,
        "unit_types": {},
        "checks": {},
        "error": None,
    }

    try:

        # --------------------------------------------------
        # 1. ACQUISITION
        # --------------------------------------------------

        inspection_data = (
            get_fmcsa_inspection_units(usdot)
        )

        result["source_status"] = (
            inspection_data.get("status")
        )

        result["inspection_count"] = (
            inspection_data.get(
                "inspection_count",
                0
            )
        )

        raw_rows = inspection_data.get(
            "inspection_units",
            []
        )

        result["raw_unit_count"] = len(
            raw_rows
        )


        # --------------------------------------------------
        # 2. NORMALIZATION
        # --------------------------------------------------

        normalized_rows = (
            normalize_fmcsa_inspection_units(
                inspection_data
            )
        )

        result["normalized_count"] = len(
            normalized_rows
        )


        # --------------------------------------------------
        # 3. EQUIPMENT IDENTITY INDEX
        # --------------------------------------------------

        equipment_index = (
            build_unique_equipment_index(
                normalized_rows
            )
        )

        result[
            "unique_equipment_count"
        ] = len(equipment_index)


        # --------------------------------------------------
        # 4. IDENTITY BASIS
        # --------------------------------------------------

        result["vin_identity_count"] = sum(
            1
            for key in equipment_index
            if key.startswith("VIN:")
        )

        result[
            "plate_identity_count"
        ] = sum(
            1
            for key in equipment_index
            if key.startswith("PLATE:")
        )

        result["unidentified_count"] = sum(
            1
            for key in equipment_index
            if key.startswith(
                "UNIDENTIFIED:"
            )
        )


        # --------------------------------------------------
        # 5. UNIT-TYPE DISTRIBUTION
        # --------------------------------------------------

        type_counts = Counter(
            row.get("unit_type_id")
            for row in normalized_rows
            if row.get("unit_type_id")
        )

        result["unit_types"] = dict(
            type_counts
        )


        # --------------------------------------------------
        # 6. INTEGRITY CHECKS
        # --------------------------------------------------

        checks = {}


        checks["USDOT_MATCH"] = (
            str(
                inspection_data.get(
                    "usdot"
                )
            )
            ==
            usdot
        )


        checks[
            "RAW_EQUALS_NORMALIZED"
        ] = (
            len(raw_rows)
            ==
            len(normalized_rows)
        )


        checks[
            "UNIQUE_NOT_GREATER_THAN_RAW"
        ] = (
            len(equipment_index)
            <=
            len(normalized_rows)
        )


        checks[
            "NORMALIZED_USDOT_MATCH"
        ] = all(
            str(row.get("usdot"))
            ==
            usdot
            for row in normalized_rows
        )


        checks[
            "INSPECTION_IDS_PRESENT"
        ] = all(
            row.get("inspection_id")
            is not None
            for row in normalized_rows
        )


        checks[
            "UNIT_TYPE_PRESENT"
        ] = all(
            row.get("unit_type_id")
            is not None
            for row in normalized_rows
        )


        # Identity accounting should equal
        # total unique equipment identities.

        checks[
            "IDENTITY_ACCOUNTING"
        ] = (
            result["vin_identity_count"]
            +
            result["plate_identity_count"]
            +
            result["unidentified_count"]
            ==
            result[
                "unique_equipment_count"
            ]
        )


        result["checks"] = checks


        # --------------------------------------------------
        # 7. DETERMINE PIPELINE STATUS
        # --------------------------------------------------

        if all(checks.values()):

            result[
                "pipeline_status"
            ] = "PASS"

        else:

            result[
                "pipeline_status"
            ] = "FAIL"


    except Exception as exc:

        result["pipeline_status"] = (
            "ERROR"
        )

        result["error"] = (
            f"{type(exc).__name__}: {exc}"
        )


    return result


# ==========================================================
# MULTI-CARRIER TEST SUITE
# ==========================================================

def run_vlcp_equipment_test_suite(
    usdots,
    delay_seconds=0.25
):
    """
    Run the current VLCP equipment pipeline against multiple
    USDOTs.

    A failure for one carrier does not stop testing the
    remaining carriers.
    """

    print(
        "VLCP MULTI-USDOT EQUIPMENT TEST SUITE"
    )

    print("=" * 70)

    suite_results = []


    for index, usdot in enumerate(
        usdots,
        start=1
    ):

        print()
        print(
            f"[{index}/{len(usdots)}] "
            f"Testing USDOT {usdot}"
        )

        print("-" * 70)


        result = (
            run_equipment_pipeline_test(
                usdot
            )
        )

        suite_results.append(result)


        print(
            "Pipeline:",
            result["pipeline_status"]
        )

        print(
            "FMCSA status:",
            result["source_status"]
        )

        print(
            "Inspections:",
            result["inspection_count"]
        )

        print(
            "Raw unit observations:",
            result["raw_unit_count"]
        )

        print(
            "Normalized observations:",
            result["normalized_count"]
        )

        print(
            "Likely unique equipment:",
            result[
                "unique_equipment_count"
            ]
        )

        print(
            "VIN identities:",
            result[
                "vin_identity_count"
            ]
        )

        print(
            "Plate fallback identities:",
            result[
                "plate_identity_count"
            ]
        )

        print(
            "Unidentified:",
            result[
                "unidentified_count"
            ]
        )

        print(
            "Unit types:",
            result["unit_types"]
        )


        if result["checks"]:

            print()
            print("Integrity checks:")

            for name, passed in (
                result["checks"].items()
            ):

                symbol = (
                    "✓"
                    if passed
                    else "✗"
                )

                print(
                    f"  {symbol} {name}"
                )


        if result["error"]:

            print()
            print(
                "ERROR:",
                result["error"]
            )


        # Small pause between public API requests.

        if (
            index
            <
            len(usdots)
        ):

            time.sleep(
                delay_seconds
            )


    # ======================================================
    # SUITE SUMMARY
    # ======================================================

    print()
    print("=" * 70)
    print("VLCP TEST SUITE SUMMARY")
    print("=" * 70)


    passed = sum(
        1
        for result in suite_results
        if result[
            "pipeline_status"
        ] == "PASS"
    )


    failed = sum(
        1
        for result in suite_results
        if result[
            "pipeline_status"
        ] == "FAIL"
    )


    errors = sum(
        1
        for result in suite_results
        if result[
            "pipeline_status"
        ] == "ERROR"
    )


    print(
        "Carriers tested:",
        len(suite_results)
    )

    print(
        "Passed:",
        passed
    )

    print(
        "Failed:",
        failed
    )

    print(
        "Errors:",
        errors
    )


    print()

    for result in suite_results:

        symbol = {
            "PASS": "✓",
            "FAIL": "✗",
            "ERROR": "⚠",
        }.get(
            result["pipeline_status"],
            "?"
        )

        print(
            f"{symbol} USDOT "
            f"{result['usdot']} "
            f"— "
            f"{result['pipeline_status']}"
        )


    print()
    print("=" * 70)

    if (
        failed == 0
        and
        errors == 0
    ):

        print(
            "OVERALL RESULT: PASS"
        )

    else:

        print(
            "OVERALL RESULT: "
            "REVIEW REQUIRED"
        )

    print("=" * 70)


    return suite_results


# ==========================================================
# EXECUTE INITIAL REGRESSION SUITE
# ==========================================================

vlcp_equipment_test_results = (
    run_vlcp_equipment_test_suite(
        TEST_USDOTS
    )
)

VLCP MULTI-USDOT EQUIPMENT TEST SUITE

[1/3] Testing USDOT 951224
----------------------------------------------------------------------
Pipeline: PASS
FMCSA status: FOUND
Inspections: 142
Raw unit observations: 263
Normalized observations: 263
Likely unique equipment: 201
VIN identities: 200
Plate fallback identities: 1
Unidentified: 0
Unit types: {'11': 136, '14': 88, '9': 35, '10': 4}

Integrity checks:
  ✓ USDOT_MATCH
  ✓ RAW_EQUALS_NORMALIZED
  ✓ UNIQUE_NOT_GREATER_THAN_RAW
  ✓ NORMALIZED_USDOT_MATCH
  ✓ INSPECTION_IDS_PRESENT
  ✓ UNIT_TYPE_PRESENT
  ✓ IDENTITY_ACCOUNTING

[2/3] Testing USDOT 80806
----------------------------------------------------------------------
Pipeline: PASS
FMCSA status: FOUND
Inspections: 5000
Raw unit observations: 9052
Normalized observations: 9052
Likely unique equipment: 8077
VIN identities: 8039
Plate fallback identities: 37
Unidentified: 1
Unit types: {'11': 4657, '10': 349, '9': 3521, '14': 475, '7': 6, '3': 28, '15': 14, '2': 2}

Integrity checks

In [25]:
# VLCP — Carrier Identity Function Diagnostic
#
# Purpose:
# Determine which existing FMCSA carrier-identity functions
# are currently loaded before integrating company names into
# the multi-USDOT test harness.

possible_functions = [
    "get_fmcsa_carrier",
    "normalize_to_vlcp",
    "verify_carrier",
]

print("VLCP CARRIER IDENTITY FUNCTIONS")
print("=" * 50)

for name in possible_functions:

    exists = (
        name in globals()
        and callable(globals()[name])
    )

    print(
        f"{'✓' if exists else '✗'} {name}"
    )

VLCP CARRIER IDENTITY FUNCTIONS
✓ get_fmcsa_carrier
✗ normalize_to_vlcp
✗ verify_carrier


In [26]:
# VLCP — Inspect existing FMCSA carrier lookup output
#
# Purpose:
# Determine the actual schema returned by get_fmcsa_carrier()
# before integrating carrier identity into the test harness.
#
# No production functions are modified.

TEST_CARRIER_DOT = "951224"

print("VLCP FMCSA CARRIER IDENTITY DIAGNOSTIC")
print("=" * 60)
print("Requested USDOT:", TEST_CARRIER_DOT)
print()

try:

    carrier_identity_test = get_fmcsa_carrier(
        TEST_CARRIER_DOT
    )

    print(
        "Returned object type:",
        type(carrier_identity_test).__name__
    )

    print()
    print("RAW RETURN VALUE")
    print("-" * 60)

    print(
        carrier_identity_test
    )

    if isinstance(
        carrier_identity_test,
        dict
    ):

        print()
        print("AVAILABLE FIELDS")
        print("-" * 60)

        for key in sorted(
            carrier_identity_test.keys()
        ):

            print(
                f"{key}: "
                f"{carrier_identity_test.get(key)}"
            )

except Exception as exc:

    print("LOOKUP ERROR")
    print("-" * 60)

    print(
        f"{type(exc).__name__}: {exc}"
    )

VLCP FMCSA CARRIER IDENTITY DIAGNOSTIC
Requested USDOT: 951224

Returned object type: dict

RAW RETURN VALUE
------------------------------------------------------------
{'success': False, 'lookup_status': 'WEBKEY_REQUIRED', 'usdot': '951224', 'source': 'FMCSA QCMobile', 'request_url': 'https://mobile.fmcsa.dot.gov/qc/services/carriers/951224', 'error': 'FMCSA WebKey not configured. Carrier lookup was not attempted.', 'data': None}

AVAILABLE FIELDS
------------------------------------------------------------
data: None
error: FMCSA WebKey not configured. Carrier lookup was not attempted.
lookup_status: WEBKEY_REQUIRED
request_url: https://mobile.fmcsa.dot.gov/qc/services/carriers/951224
source: FMCSA QCMobile
success: False
usdot: 951224


In [27]:
# VLCP — Census Capability Diagnostic
#
# Purpose:
# Find existing Census/DOT carrier identity functionality
# before adding anything new.

possible_objects = [
    "FMCSA_URL",
    "CENSUS_URL",
    "CENSUS_DATASET",
    "get_census_carrier",
    "get_fmcsa_census",
    "get_carrier_census",
    "normalize_census_to_vlcp",
]

print("VLCP CENSUS CAPABILITY DIAGNOSTIC")
print("=" * 55)

for name in possible_objects:

    exists = name in globals()

    if exists:
        obj = globals()[name]

        if callable(obj):
            object_type = "FUNCTION"
        else:
            object_type = type(obj).__name__.upper()

        print(
            f"✓ {name:<30} {object_type}"
        )

    else:
        print(
            f"✗ {name}"
        )


# --------------------------------------------------
# ALSO FIND LOADED OBJECTS WITH RELEVANT NAMES
# --------------------------------------------------

print()
print("OTHER POSSIBLE CARRIER/CENSUS OBJECTS")
print("-" * 55)

keywords = (
    "census",
    "carrier",
    "fmcsa",
    "dot"
)

matches = []

for name in globals():

    lower_name = name.lower()

    if any(
        keyword in lower_name
        for keyword in keywords
    ):
        matches.append(name)


for name in sorted(matches):

    # Skip Python internal globals.
    if name.startswith("_"):
        continue

    try:
        obj = globals()[name]

        if callable(obj):
            object_type = "FUNCTION"
        else:
            object_type = type(obj).__name__.upper()

        print(
            f"{name:<40} {object_type}"
        )

    except Exception:
        pass

VLCP CENSUS CAPABILITY DIAGNOSTIC
✗ FMCSA_URL
✗ CENSUS_URL
✗ CENSUS_DATASET
✗ get_census_carrier
✗ get_fmcsa_census
✗ get_carrier_census
✗ normalize_census_to_vlcp

OTHER POSSIBLE CARRIER/CENSUS OBJECTS
-------------------------------------------------------


RuntimeError: dictionary changed size during iteration

In [ ]:
# VLCP — Census / Carrier Capability Diagnostic v2
#
# Safe for Google Colab / IPython.
# Does not modify any VLCP production functions.

print("VLCP CARRIER/CENSUS CAPABILITY DIAGNOSTIC")
print("=" * 65)


# --------------------------------------------------
# 1. CHECK EXPECTED OBJECTS
# --------------------------------------------------

possible_objects = [
    "FMCSA_URL",
    "CENSUS_URL",
    "CENSUS_DATASET",
    "get_census_carrier",
    "get_fmcsa_census",
    "get_carrier_census",
    "normalize_census_to_vlcp",
    "get_fmcsa_carrier",
]


for name in possible_objects:

    exists = name in globals()

    if exists:

        obj = globals()[name]

        object_type = (
            "FUNCTION"
            if callable(obj)
            else type(obj).__name__.upper()
        )

        print(
            f"✓ {name:<32} {object_type}"
        )

    else:

        print(
            f"✗ {name}"
        )


# --------------------------------------------------
# 2. SAFELY SNAPSHOT GLOBAL NAMES
# --------------------------------------------------

global_names = list(
    globals().keys()
)


# --------------------------------------------------
# 3. FIND POTENTIALLY RELEVANT OBJECTS
# --------------------------------------------------

keywords = (
    "census",
    "carrier",
    "fmcsa",
    "dot",
    "socrata",
    "company",
    "authority",
    "insurance",
)


matches = []

for name in global_names:

    if name.startswith("_"):
        continue

    lower_name = name.lower()

    if any(
        keyword in lower_name
        for keyword in keywords
    ):
        matches.append(name)


print()
print("DISCOVERED RELEVANT OBJECTS")
print("-" * 65)


if not matches:

    print(
        "No additional carrier/Census objects found."
    )

else:

    for name in sorted(set(matches)):

        try:

            obj = globals().get(name)

            object_type = (
                "FUNCTION"
                if callable(obj)
                else type(obj).__name__.upper()
            )

            print(
                f"{name:<45} {object_type}"
            )

        except Exception as exc:

            print(
                f"{name:<45} "
                f"UNREADABLE: {type(exc).__name__}"
            )


print()
print("=" * 65)
print("DIAGNOSTIC COMPLETE")

In [ ]:
# VLCP — Existing Carrier Identity Data Inspection
#
# Purpose:
# Inspect carrier-related objects already produced by the notebook
# before creating or modifying carrier identity logic.
#
# READ ONLY — does not modify VLCP state.

from pprint import pprint


OBJECTS_TO_INSPECT = [
    "carrier_info",
    "carrier_evidence",
    "carrier_verification",
    "fmcsa_result",
]


print("VLCP EXISTING CARRIER DATA INSPECTION")
print("=" * 70)


for object_name in OBJECTS_TO_INSPECT:

    print()
    print(object_name)
    print("-" * 70)

    if object_name not in globals():

        print("NOT LOADED")
        continue

    obj = globals()[object_name]

    print(
        "Type:",
        type(obj).__name__
    )

    if isinstance(obj, dict):

        print(
            "Keys:",
            list(obj.keys())
        )

        print()
        print("Contents:")

        pprint(obj)

    else:

        pprint(obj)


# --------------------------------------------------
# INSPECT FUNCTION SIGNATURES
# --------------------------------------------------

print()
print("=" * 70)
print("CARRIER FUNCTION SIGNATURES")
print("=" * 70)

import inspect


FUNCTIONS_TO_INSPECT = [
    "get_fmcsa_carrier",
    "normalize_fmcsa_carrier_result",
    "compare_carrier_claim_to_fmcsa",
    "apply_carrier_verification_to_shipment",
]


for function_name in FUNCTIONS_TO_INSPECT:

    print()

    if (
        function_name in globals()
        and callable(globals()[function_name])
    ):

        fn = globals()[function_name]

        try:

            print(
                f"{function_name}"
                f"{inspect.signature(fn)}"
            )

        except Exception as exc:

            print(
                f"{function_name}: "
                f"SIGNATURE UNAVAILABLE "
                f"({type(exc).__name__})"
            )

    else:

        print(
            f"{function_name}: NOT LOADED"
        )


print()
print("=" * 70)
print("INSPECTION COMPLETE")

In [ ]:
# ================================================================
# VLCP CELL 19
# MULTI-USDOT EQUIPMENT PIPELINE REGRESSION TEST HARNESS
# ================================================================
#
# Purpose:
#   Test the VLCP carrier identity + FMCSA inspection equipment
#   pipeline against multiple USDOTs.
#
# Permanent VLCP requirements:
#   - Dynamic USDOT input
#   - Company Name + USDOT displayed together
#   - No production logic hard-coded to a test carrier
#   - One carrier failure must not stop the suite
#   - Missing evidence is NOT automatically a failed carrier
#   - PASS means pipeline integrity, NOT carrier verification
#
# Dependencies:
#   get_fmcsa_carrier()
#   normalize_fmcsa_carrier_result()
#   get_fmcsa_inspection_units()
#   normalize_fmcsa_inspection_units()
#   build_unique_equipment_index()
#   FMCSA_WEBKEY
# ================================================================

import time


# ----------------------------------------------------------------
# TEST CONFIGURATION
# ----------------------------------------------------------------

TEST_USDOTS = [
    "951224",
    "80806",
    "2232566",
]


# ----------------------------------------------------------------
# CARRIER IDENTITY RESOLUTION
# ----------------------------------------------------------------

def resolve_carrier_identity(usdot):
    """
    Retrieve and normalize FMCSA carrier identity.

    Returns a stable identity structure without converting
    source failures into false NOT_FOUND results.
    """

    usdot = str(usdot).strip()

    identity = {
        "usdot": usdot,
        "legal_name": None,
        "dba_name": None,
        "display_name": None,
        "identity_status": "UNKNOWN",
        "identity_source": "FMCSA QCMobile",
        "error": None,
    }

    try:

        # Use the existing configured FMCSA WebKey.
        result = get_fmcsa_carrier(
            usdot,
            webkey=FMCSA_WEBKEY
        )

        identity["identity_status"] = result.get(
            "lookup_status",
            "UNKNOWN"
        )

        if not result.get("success"):

            identity["error"] = result.get("error")

            identity["display_name"] = (
                f"UNKNOWN CARRIER — USDOT {usdot}"
            )

            return identity


        normalized = normalize_fmcsa_carrier_result(
            result
        )


        # Existing normalizer may return either the normalized
        # carrier directly or an evidence wrapper.
        if isinstance(normalized, dict):

            if isinstance(
                normalized.get("carrier"),
                dict
            ):
                carrier = normalized["carrier"]

            else:
                carrier = normalized

        else:
            carrier = {}


        identity["legal_name"] = carrier.get(
            "legal_name"
        )

        identity["dba_name"] = carrier.get(
            "dba_name"
        )


        # Prefer legal name as canonical carrier identity.
        if identity["legal_name"]:

            identity["display_name"] = (
                identity["legal_name"]
            )

        elif identity["dba_name"]:

            identity["display_name"] = (
                identity["dba_name"]
            )

        else:

            identity["display_name"] = (
                f"UNKNOWN CARRIER — USDOT {usdot}"
            )


        return identity


    except Exception as exc:

        identity["identity_status"] = "ERROR"

        identity["error"] = (
            f"{type(exc).__name__}: {exc}"
        )

        identity["display_name"] = (
            f"UNKNOWN CARRIER — USDOT {usdot}"
        )

        return identity


# ----------------------------------------------------------------
# SINGLE USDOT PIPELINE TEST
# ----------------------------------------------------------------

def run_equipment_pipeline_test(usdot):

    usdot = str(usdot).strip()

    result = {
        "usdot": usdot,

        "legal_name": None,
        "dba_name": None,
        "company_name": None,

        "identity_status": "UNKNOWN",
        "identity_error": None,

        "source_status": "UNKNOWN",

        "inspection_count": 0,
        "raw_unit_count": 0,
        "normalized_count": 0,
        "unique_equipment_count": 0,

        "vin_identity_count": 0,
        "plate_identity_count": 0,
        "unidentified_count": 0,

        "unit_types": {},

        "checks": {},

        "pipeline_status": "NOT_RUN",

        "error": None,
    }


    try:

        # --------------------------------------------------------
        # 1. RESOLVE CARRIER IDENTITY
        # --------------------------------------------------------

        identity = resolve_carrier_identity(
            usdot
        )

        result["legal_name"] = identity.get(
            "legal_name"
        )

        result["dba_name"] = identity.get(
            "dba_name"
        )

        result["company_name"] = identity.get(
            "display_name"
        )

        result["identity_status"] = identity.get(
            "identity_status"
        )

        result["identity_error"] = identity.get(
            "error"
        )


        # --------------------------------------------------------
        # 2. RETRIEVE INSPECTION-UNIT EVIDENCE
        # --------------------------------------------------------

        inspection_result = (
            get_fmcsa_inspection_units(
                usdot
            )
        )

        result["source_status"] = (
            inspection_result.get(
                "status",
                "UNKNOWN"
            )
        )

        result["inspection_count"] = (
            inspection_result.get(
                "inspection_count",
                0
            )
        )

        raw_rows = (
            inspection_result.get(
                "inspection_units",
                []
            )
            or []
        )

        result["raw_unit_count"] = len(
            raw_rows
        )


        # --------------------------------------------------------
        # 3. NORMALIZE
        # --------------------------------------------------------

        normalized_rows = (
            normalize_fmcsa_inspection_units(
                inspection_result
            )
        )

        result["normalized_count"] = len(
            normalized_rows
        )


        # --------------------------------------------------------
        # 4. BUILD UNIQUE EQUIPMENT INDEX
        # --------------------------------------------------------

        equipment_index = (
            build_unique_equipment_index(
                normalized_rows
            )
        )

        result["unique_equipment_count"] = len(
            equipment_index
        )


        # --------------------------------------------------------
        # 5. IDENTITY-BASIS COUNTS
        # --------------------------------------------------------

        vin_count = 0
        plate_count = 0
        unidentified_count = 0


        for identity_key in equipment_index:

            key = str(identity_key)

            if key.startswith("VIN:"):

                vin_count += 1

            elif key.startswith("PLATE:"):

                plate_count += 1

            else:

                unidentified_count += 1


        result["vin_identity_count"] = (
            vin_count
        )

        result["plate_identity_count"] = (
            plate_count
        )

        result["unidentified_count"] = (
            unidentified_count
        )


        # --------------------------------------------------------
        # 6. UNIT TYPE DISTRIBUTION
        # --------------------------------------------------------

        unit_types = {}

        for row in normalized_rows:

            unit_type = row.get(
                "unit_type_id"
            )

            unit_type = (
                str(unit_type)
                if unit_type is not None
                else "UNKNOWN"
            )

            unit_types[unit_type] = (
                unit_types.get(
                    unit_type,
                    0
                )
                + 1
            )


        result["unit_types"] = unit_types


        # --------------------------------------------------------
        # 7. PIPELINE INTEGRITY CHECKS
        # --------------------------------------------------------

        checks = {}


        checks["USDOT_MATCH"] = (
            str(
                inspection_result.get(
                    "usdot",
                    ""
                )
            )
            == usdot
        )


        checks["RAW_EQUALS_NORMALIZED"] = (
            len(raw_rows)
            == len(normalized_rows)
        )


        checks["UNIQUE_NOT_GREATER_THAN_RAW"] = (
            len(equipment_index)
            <= len(raw_rows)
        )


        checks["NORMALIZED_USDOT_MATCH"] = all(

            str(
                row.get(
                    "usdot",
                    ""
                )
            )
            == usdot

            for row in normalized_rows
        )


        checks["INSPECTION_IDS_PRESENT"] = all(

            bool(
                row.get(
                    "inspection_id"
                )
            )

            for row in normalized_rows
        )


        checks["UNIT_TYPE_PRESENT"] = all(

            row.get(
                "unit_type_id"
            )
            is not None

            for row in normalized_rows
        )


        checks["IDENTITY_ACCOUNTING"] = (

            vin_count
            + plate_count
            + unidentified_count

            == len(equipment_index)
        )


        result["checks"] = checks


        # --------------------------------------------------------
        # 8. DETERMINE SOFTWARE PIPELINE RESULT
        # --------------------------------------------------------

        if all(checks.values()):

            result["pipeline_status"] = "PASS"

        else:

            result["pipeline_status"] = "FAIL"


        return result


    except Exception as exc:

        result["pipeline_status"] = "ERROR"

        result["error"] = (
            f"{type(exc).__name__}: {exc}"
        )

        return result


# ----------------------------------------------------------------
# MULTI-USDOT SUITE
# ----------------------------------------------------------------

def run_vlcp_equipment_test_suite(
    usdots,
    delay_seconds=0.25
):

    results = []


    print()
    print(
        "VLCP MULTI-USDOT EQUIPMENT REGRESSION"
    )
    print("=" * 78)


    for index, usdot in enumerate(
        usdots,
        start=1
    ):

        usdot = str(usdot).strip()

        print()
        print(
            f"[{index}/{len(usdots)}] "
            f"Resolving USDOT {usdot}..."
        )


        test_result = (
            run_equipment_pipeline_test(
                usdot
            )
        )

        results.append(
            test_result
        )


        company_name = (
            test_result.get(
                "company_name"
            )
            or "UNKNOWN CARRIER"
        )


        print()
        print(company_name)
        print(
            f"USDOT: {usdot}"
        )
        print("-" * 78)

        print(
            "Identity status:       ",
            test_result[
                "identity_status"
            ]
        )

        if test_result.get(
            "dba_name"
        ):

            print(
                "DBA:                   ",
                test_result[
                    "dba_name"
                ]
            )


        print(
            "Inspection status:     ",
            test_result[
                "source_status"
            ]
        )

        print(
            "Inspections:           ",
            test_result[
                "inspection_count"
            ]
        )

        print(
            "Raw unit observations: ",
            test_result[
                "raw_unit_count"
            ]
        )

        print(
            "Normalized observations:",
            test_result[
                "normalized_count"
            ]
        )

        print(
            "Unique equipment:      ",
            test_result[
                "unique_equipment_count"
            ]
        )

        print(
            "VIN identities:        ",
            test_result[
                "vin_identity_count"
            ]
        )

        print(
            "Plate identities:      ",
            test_result[
                "plate_identity_count"
            ]
        )

        print(
            "Unidentified:          ",
            test_result[
                "unidentified_count"
            ]
        )

        print(
            "Unit types:            ",
            test_result[
                "unit_types"
            ]
        )


        print()
        print("Integrity checks:")


        for check_name, passed in (
            test_result.get(
                "checks",
                {}
            )
            .items()
        ):

            symbol = (
                "✓"
                if passed
                else "✗"
            )

            print(
                f"  {symbol} {check_name}"
            )


        if test_result.get(
            "identity_error"
        ):

            print()
            print(
                "Identity note:",
                test_result[
                    "identity_error"
                ]
            )


        if test_result.get(
            "error"
        ):

            print()
            print(
                "Pipeline error:",
                test_result[
                    "error"
                ]
            )


        print()
        print(
            "PIPELINE RESULT:",
            test_result[
                "pipeline_status"
            ]
        )


        if index < len(usdots):

            time.sleep(
                delay_seconds
            )


    # ------------------------------------------------------------
    # SUITE SUMMARY
    # ------------------------------------------------------------

    passed = sum(
        result["pipeline_status"] == "PASS"
        for result in results
    )

    failed = sum(
        result["pipeline_status"] == "FAIL"
        for result in results
    )

    errors = sum(
        result["pipeline_status"] == "ERROR"
        for result in results
    )


    print()
    print("=" * 78)
    print(
        "VLCP REGRESSION SUMMARY"
    )
    print("=" * 78)


    for result in results:

        symbol = {
            "PASS": "✓",
            "FAIL": "✗",
            "ERROR": "!",
        }.get(
            result["pipeline_status"],
            "?"
        )

        company_name = (
            result.get(
                "company_name"
            )
            or "UNKNOWN CARRIER"
        )

        print(
            f"{symbol} "
            f"{company_name} — "
            f"USDOT {result['usdot']} — "
            f"{result['pipeline_status']}"
        )


    print()
    print(
        f"Passed: {passed}"
    )

    print(
        f"Failed: {failed}"
    )

    print(
        f"Errors: {errors}"
    )


    print()

    if failed == 0 and errors == 0:

        print(
            "OVERALL PIPELINE STATUS: PASS"
        )

    else:

        print(
            "OVERALL PIPELINE STATUS: "
            "REVIEW REQUIRED"
        )


    print()
    print(
        "NOTE: PASS indicates VLCP software pipeline "
        "integrity for this test."
    )

    print(
        "It does NOT independently certify the "
        "carrier or equipment."
    )


    return results


# ----------------------------------------------------------------
# EXECUTE REGRESSION SUITE
# ----------------------------------------------------------------

vlcp_equipment_test_results = (
    run_vlcp_equipment_test_suite(
        TEST_USDOTS
    )
)

In [ ]:
# ================================================================
# VLCP CELL 20
# SOCRATA PAGINATION BOUNDARY DIAGNOSTIC
# ================================================================
#
# Purpose:
# Determine whether the current 5,000-inspection result for
# J B HUNT TRANSPORT INC (USDOT 80806) represents:
#
#   A) the complete available result set
#   B) the first page of a larger result set
#
# READ ONLY:
#   This cell does not modify the acquisition pipeline.
# ================================================================


TEST_PAGINATION_USDOT = "80806"
PAGE_SIZE = 5000


print("VLCP SOCRATA PAGINATION BOUNDARY TEST")
print("=" * 70)

print(
    "Carrier: J B HUNT TRANSPORT INC"
)

print(
    f"USDOT:   {TEST_PAGINATION_USDOT}"
)

print()


# ---------------------------------------------------------------
# PAGE 1
# ---------------------------------------------------------------

page_1_query = f"""
SELECT inspection_id, dot_number
WHERE dot_number = '{TEST_PAGINATION_USDOT}'
LIMIT {PAGE_SIZE}
OFFSET 0
"""


page_1_payload = _socrata_query(
    VEHICLE_INSPECTION_DATASET,
    page_1_query,
    page_size=PAGE_SIZE
)


page_1_rows = _extract_socrata_rows(
    page_1_payload
)


# ---------------------------------------------------------------
# PAGE 2
# ---------------------------------------------------------------

page_2_query = f"""
SELECT inspection_id, dot_number
WHERE dot_number = '{TEST_PAGINATION_USDOT}'
LIMIT {PAGE_SIZE}
OFFSET {PAGE_SIZE}
"""


page_2_payload = _socrata_query(
    VEHICLE_INSPECTION_DATASET,
    page_2_query,
    page_size=PAGE_SIZE
)


page_2_rows = _extract_socrata_rows(
    page_2_payload
)


# ---------------------------------------------------------------
# RESULTS
# ---------------------------------------------------------------

print(
    "Page 1 rows:",
    len(page_1_rows)
)

print(
    "Page 2 rows:",
    len(page_2_rows)
)

print()


if len(page_2_rows) > 0:

    print(
        "⚠ PAGINATION REQUIRED"
    )

    print(
        "Additional inspection records exist "
        "beyond the first 5,000."
    )

    print()

    print(
        "Current get_fmcsa_inspection_units() "
        "does not retrieve the complete inspection "
        "history for this carrier."
    )

else:

    print(
        "✓ NO ADDITIONAL PAGE FOUND"
    )

    print(
        "The 5,000-row result appears to represent "
        "the complete available inspection set."
    )


# ---------------------------------------------------------------
# DUPLICATE / OVERLAP CHECK
# ---------------------------------------------------------------

page_1_ids = {
    str(row.get("inspection_id"))
    for row in page_1_rows
    if row.get("inspection_id") is not None
}

page_2_ids = {
    str(row.get("inspection_id"))
    for row in page_2_rows
    if row.get("inspection_id") is not None
}


overlap = (
    page_1_ids
    & page_2_ids
)


print()
print("UNIQUE-ID CHECK")
print("-" * 70)

print(
    "Page 1 unique inspection IDs:",
    len(page_1_ids)
)

print(
    "Page 2 unique inspection IDs:",
    len(page_2_ids)
)

print(
    "IDs appearing on both pages:",
    len(overlap)
)


print()
print("=" * 70)
print("DIAGNOSTIC COMPLETE")

In [ ]:
# ================================================================
# VLCP CELL 21
# PAGINATED FMCSA INSPECTION ACQUISITION
# ================================================================
#
# Fix:
#   Removes the 5,000-inspection truncation discovered with
#   J B HUNT TRANSPORT INC — USDOT 80806.
#
# Architecture:
#   USDOT
#     ↓
#   Paginated inspection-ID acquisition
#     ↓
#   Deduplicated inspection IDs
#     ↓
#   Batched inspection-unit acquisition
#     ↓
#   Raw evidence returned with pagination metadata
#
# IMPORTANT:
#   - Dynamic for any USDOT
#   - No carrier-specific production logic
#   - Replaces get_fmcsa_inspection_units()
#   - Preserves downstream return fields
# ================================================================


INSPECTION_PAGE_SIZE = 5000
INSPECTION_UNIT_BATCH_SIZE = 100


def get_fmcsa_inspection_units(
    usdot,
    page_size=INSPECTION_PAGE_SIZE
):
    """
    Retrieve FMCSA inspection-unit evidence for any USDOT.

    Inspection IDs are retrieved using Socrata pagination until
    the source returns fewer rows than the requested page size.

    Inspection-unit records are then retrieved in batches.

    Returns raw source evidence. Equipment classification and
    normalization remain downstream responsibilities.
    """

    # ------------------------------------------------------------
    # 1. VALIDATE USDOT
    # ------------------------------------------------------------

    usdot = str(usdot).strip()

    if not usdot:

        return {
            "success": False,
            "status": "INVALID_USDOT",
            "usdot": usdot,
            "inspection_count": 0,
            "unit_count": 0,
            "inspection_units": [],
            "pagination": None,
            "error": "USDOT cannot be empty.",
        }


    if not usdot.isdigit():

        return {
            "success": False,
            "status": "INVALID_USDOT",
            "usdot": usdot,
            "inspection_count": 0,
            "unit_count": 0,
            "inspection_units": [],
            "pagination": None,
            "error": "USDOT must contain digits only.",
        }


    if not isinstance(page_size, int) or page_size <= 0:

        return {
            "success": False,
            "status": "INVALID_PAGE_SIZE",
            "usdot": usdot,
            "inspection_count": 0,
            "unit_count": 0,
            "inspection_units": [],
            "pagination": None,
            "error": "page_size must be a positive integer.",
        }


    try:

        # --------------------------------------------------------
        # 2. PAGINATE INSPECTION IDS
        # --------------------------------------------------------

        inspection_ids = []

        seen_inspection_ids = set()

        page_number = 0
        offset = 0
        total_rows_received = 0


        while True:

            page_number += 1


            inspection_query = f"""
            SELECT inspection_id, dot_number
            WHERE dot_number = '{usdot}'
            ORDER BY inspection_id
            LIMIT {page_size}
            OFFSET {offset}
            """


            inspection_payload = _socrata_query(
                VEHICLE_INSPECTION_DATASET,
                inspection_query,
                page_size=page_size
            )


            page_rows = _extract_socrata_rows(
                inspection_payload
            )


            page_row_count = len(
                page_rows
            )


            total_rows_received += (
                page_row_count
            )


            # ----------------------------------------------------
            # COLLECT UNIQUE INSPECTION IDS
            # ----------------------------------------------------

            for row in page_rows:

                inspection_id = row.get(
                    "inspection_id"
                )

                if inspection_id is None:
                    continue


                inspection_id = str(
                    inspection_id
                ).strip()


                if not inspection_id:
                    continue


                if (
                    inspection_id
                    not in seen_inspection_ids
                ):

                    seen_inspection_ids.add(
                        inspection_id
                    )

                    inspection_ids.append(
                        inspection_id
                    )


            # ----------------------------------------------------
            # PAGINATION TERMINATION
            # ----------------------------------------------------

            if page_row_count < page_size:
                break


            offset += page_size


        # --------------------------------------------------------
        # 3. HANDLE ZERO INSPECTIONS
        # --------------------------------------------------------

        if not inspection_ids:

            return {
                "success": True,
                "status": "NO_INSPECTIONS_FOUND",
                "usdot": usdot,

                "inspection_count": 0,
                "unit_count": 0,

                "inspection_units": [],

                "pagination": {
                    "page_size": page_size,
                    "pages_retrieved": page_number,
                    "rows_received": total_rows_received,
                    "unique_inspection_ids": 0,
                    "complete": True,
                },

                "source": (
                    "FMCSA / DOT Public Data Portal"
                ),
            }


        # --------------------------------------------------------
        # 4. RETRIEVE INSPECTION UNITS
        # --------------------------------------------------------

        inspection_units = []


        requested_fields = """
            inspection_id,
            insp_unit_id,
            insp_unit_type_id,
            insp_unit_number,
            insp_unit_make,
            insp_unit_company,
            insp_unit_license,
            insp_unit_license_state,
            insp_unit_vehicle_id_number,
            insp_unit_decal,
            insp_unit_decal_number
        """


        for start in range(
            0,
            len(inspection_ids),
            INSPECTION_UNIT_BATCH_SIZE
        ):

            batch = inspection_ids[
                start:
                start + INSPECTION_UNIT_BATCH_SIZE
            ]


            # Inspection IDs were sourced directly from DOT data.
            # Escape single quotes defensively before constructing
            # the Socrata IN clause.

            escaped_ids = [
                inspection_id.replace(
                    "'",
                    "''"
                )
                for inspection_id in batch
            ]


            quoted_ids = ",".join(
                f"'{inspection_id}'"
                for inspection_id in escaped_ids
            )


            unit_query = f"""
            SELECT {requested_fields}
            WHERE inspection_id IN ({quoted_ids})
            """


            unit_payload = _socrata_query(
                INSPECTION_UNIT_DATASET,
                unit_query,
                page_size=5000
            )


            unit_rows = _extract_socrata_rows(
                unit_payload
            )


            inspection_units.extend(
                unit_rows
            )


        # --------------------------------------------------------
        # 5. RETURN RAW EVIDENCE
        # --------------------------------------------------------

        return {
            "success": True,
            "status": "FOUND",
            "usdot": usdot,

            "inspection_count": len(
                inspection_ids
            ),

            "unit_count": len(
                inspection_units
            ),

            "inspection_units": (
                inspection_units
            ),

            "pagination": {
                "page_size": page_size,

                "pages_retrieved": (
                    page_number
                ),

                "rows_received": (
                    total_rows_received
                ),

                "unique_inspection_ids": (
                    len(inspection_ids)
                ),

                "complete": True,
            },

            "source": (
                "FMCSA / DOT Public Data Portal"
            ),
        }


    except Exception as exc:

        return {
            "success": False,
            "status": "SOURCE_ERROR",
            "usdot": usdot,

            "inspection_count": 0,
            "unit_count": 0,

            "inspection_units": [],

            "pagination": {
                "page_size": page_size,
                "complete": False,
            },

            "source": (
                "FMCSA / DOT Public Data Portal"
            ),

            "error": (
                f"{type(exc).__name__}: {exc}"
            ),
        }


print(
    "✓ Paginated get_fmcsa_inspection_units() loaded."
)

In [37]:
# ============================================================
# CELL 22 — FMCSA EQUIPMENT TYPE EVIDENCE DIAGNOSTIC
# ============================================================
#
# PURPOSE:
# Examine the raw evidence associated with each FMCSA
# insp_unit_type_id before we assign authoritative meanings.
#
# IMPORTANT:
# - Diagnostic only
# - Does NOT modify the verified VLCP pipeline
# - Does NOT hard-code equipment type meanings
# - Accepts any USDOT dynamically
# ============================================================

from collections import Counter, defaultdict


def analyze_equipment_type_evidence(usdot, sample_size=8):

    usdot = str(usdot).strip()

    print("=" * 78)
    print("VLCP FMCSA EQUIPMENT TYPE EVIDENCE DIAGNOSTIC")
    print("=" * 78)
    print(f"USDOT: {usdot}")
    print()

    # --------------------------------------------------------
    # STEP 1 — Retrieve inspection/equipment evidence
    # --------------------------------------------------------

    inspection_result = get_fmcsa_inspection_units(usdot)

    status = inspection_result.get("status", "UNKNOWN")

    print(f"Inspection source status: {status}")
    print(
        f"Inspections retrieved: "
        f"{inspection_result.get('inspection_count', 0)}"
    )
    print(
        f"Raw unit observations: "
        f"{inspection_result.get('unit_count', 0)}"
    )

    if status != "FOUND":

        print()
        print("No equipment observations available for analysis.")

        return {
            "usdot": usdot,
            "status": status,
            "type_analysis": {}
        }

    # --------------------------------------------------------
    # STEP 2 — Normalize evidence
    # --------------------------------------------------------

    normalized_rows = normalize_fmcsa_inspection_units(
        inspection_result
    )

    print(
        f"Normalized observations: {len(normalized_rows)}"
    )

    # --------------------------------------------------------
    # STEP 3 — Group observations by FMCSA unit type
    # --------------------------------------------------------

    type_data = defaultdict(
        lambda: {
            "count": 0,
            "makes": Counter(),
            "vin_count": 0,
            "plate_count": 0,
            "sample_rows": []
        }
    )

    for row in normalized_rows:

        type_id = row.get("unit_type_id") or "UNKNOWN"

        record = type_data[type_id]

        record["count"] += 1

        make = row.get("make_raw")

        if make:
            record["makes"][make] += 1

        if row.get("vin_normalized"):
            record["vin_count"] += 1

        if (
            row.get("license_state")
            and row.get("license_normalized")
        ):
            record["plate_count"] += 1

        if len(record["sample_rows"]) < sample_size:

            record["sample_rows"].append({

                "inspection_id":
                    row.get("inspection_id"),

                "make":
                    row.get("make_raw"),

                "vin":
                    row.get("vin_normalized"),

                "license_state":
                    row.get("license_state"),

                "license":
                    row.get("license_normalized"),

                "company_unit_id":
                    row.get("company_unit_id_normalized")
            })

    # --------------------------------------------------------
    # STEP 4 — Print diagnostic report
    # --------------------------------------------------------

    print()
    print("=" * 78)
    print("UNIT TYPE BREAKDOWN")
    print("=" * 78)

    for type_id in sorted(
        type_data.keys(),
        key=lambda x: int(x) if str(x).isdigit() else 999
    ):

        record = type_data[type_id]

        print()
        print(f"FMCSA UNIT TYPE ID: {type_id}")
        print("-" * 78)

        print(f"Observations: {record['count']}")
        print(f"VIN present: {record['vin_count']}")
        print(f"Plate present: {record['plate_count']}")

        print()
        print("Most common makes:")

        for make, count in record["makes"].most_common(10):
            print(f"  {make}: {count}")

        print()
        print("Sample evidence:")

        for sample in record["sample_rows"]:

            print(
                f"  Inspection={sample['inspection_id']} | "
                f"Make={sample['make']} | "
                f"VIN={sample['vin']} | "
                f"Plate={sample['license_state']} "
                f"{sample['license']} | "
                f"CompanyUnit={sample['company_unit_id']}"
            )

    # --------------------------------------------------------
    # STEP 5 — Return structured result
    # --------------------------------------------------------

    return {
        "usdot": usdot,
        "status": status,
        "inspection_count":
            inspection_result.get("inspection_count", 0),
        "normalized_count":
            len(normalized_rows),
        "type_analysis":
            dict(type_data)
    }


# ============================================================
# TEST
# ============================================================

equipment_type_analysis = analyze_equipment_type_evidence(
    "951224",
    sample_size=8
)

VLCP FMCSA EQUIPMENT TYPE EVIDENCE DIAGNOSTIC
USDOT: 951224

Inspection source status: FOUND
Inspections retrieved: 142
Raw unit observations: 263
Normalized observations: 263

UNIT TYPE BREAKDOWN

FMCSA UNIT TYPE ID: 9
------------------------------------------------------------------------------
Observations: 35
VIN present: 34
Plate present: 35

Most common makes:
  CIMC VEHIC: 11
  DORS: 3
  INTERNATIO: 2
  HERCULES: 2
  UNKNOWN: 2
  OTHR: 2
  HERC: 2
  BERI: 1
  MONN: 1
  OT: 1

Sample evidence:
  Inspection=79871216 | Make=DORS | VIN=None | Plate=SC 26963SG | CompanyUnit=None
  Inspection=82813306 | Make=CIMC VEHIC | VIN=LJRC4126362003506 | Plate=TN T951966 | CompanyUnit=NA
  Inspection=83445809 | Make=CIMC VEHIC | VIN=LJRC41268HT000064 | Plate=TN U811162 | CompanyUnit=None
  Inspection=83470268 | Make=CIMC VEHIC | VIN=LJRC2825451008529 | Plate=TN U776134 | CompanyUnit=None
  Inspection=83980263 | Make=INTERNATIO | VIN=3HSDJAPR9EN098735 | Plate=FL XE256G | CompanyUnit=150
  Inspe

In [38]:
# ============================================================
# CELL 24 — VLCP HUMAN EQUIPMENT DISPLAY CONTRACT
# ============================================================
#
# PURPOSE:
# Establish the permanent human-readable equipment format.
#
# DISPLAY PRIORITY:
#
# 1. Type + Year + Make + Model
#       Tractor | 2018 Freightliner Cascadia
#
# 2. Type + Year + Make
#       Tractor | 2018 Freightliner
#
# 3. Type + VIN when year cannot be established
#       Tractor | VIN: 1FUJGLDR5JLJR2003
#
# IMPORTANT:
# - Presentation layer only
# - Does NOT replace or modify existing VLCP functions
# - Does NOT guess tractor/trailer
# - Does NOT guess year/model
# - Existing evidence/provenance remains untouched
# ============================================================


def build_equipment_display(
    equipment_type=None,
    vin=None,
    year=None,
    make=None,
    model=None
):
    """
    Build the permanent VLCP human-readable equipment result.

    Future classification and VIN-decoding layers can pass verified
    values into this function without changing the display contract.
    """

    def clean(value):
        if value is None:
            return None

        value = str(value).strip()

        if not value:
            return None

        if value.upper() in {
            "NONE",
            "NULL",
            "N/A",
            "UNKNOWN"
        }:
            return None

        return value


    equipment_type = clean(equipment_type)
    vin = clean(vin)
    year = clean(year)
    make = clean(make)
    model = clean(model)


    # --------------------------------------------------------
    # TYPE
    # --------------------------------------------------------

    display_type = equipment_type or "Equipment"


    # --------------------------------------------------------
    # EQUIPMENT DESCRIPTION
    #
    # Rule:
    # If year is known, display year + make/model.
    # If year is not known, fall back to VIN.
    # --------------------------------------------------------

    if year:

        description_parts = [year]

        if make:
            description_parts.append(make)

        if model:
            description_parts.append(model)

        equipment_description = " ".join(description_parts)

    elif vin:

        equipment_description = f"VIN: {vin}"

    elif make:

        # Last-resort readable fallback.
        # This does NOT invent year/model information.

        equipment_description = make

    else:

        equipment_description = "Identification unavailable"


    return {
        "type": display_type,
        "equipment": equipment_description,

        # Structured values retained for downstream VLCP use.
        "year": year,
        "make": make,
        "model": model,
        "vin": vin
    }


# ============================================================
# FORMAT MULTIPLE EQUIPMENT RECORDS
# ============================================================

def build_equipment_display_report(equipment_records):
    """
    Convert classified/decoded equipment records into the
    permanent VLCP human-readable format.

    Expected future input:
    {
        "equipment_type": "Tractor",
        "vin": "...",
        "year": "2018",
        "make": "Freightliner",
        "model": "Cascadia"
    }
    """

    report = []

    for record in equipment_records:

        display = build_equipment_display(

            equipment_type=record.get("equipment_type"),

            vin=(
                record.get("vin")
                or record.get("vin_normalized")
            ),

            year=record.get("year"),

            make=(
                record.get("make")
                or record.get("make_raw")
            ),

            model=record.get("model")
        )

        report.append(display)

    return report


# ============================================================
# HUMAN-READABLE PRINT FUNCTION
# ============================================================

def print_equipment_display_report(equipment_records):

    report = build_equipment_display_report(
        equipment_records
    )

    print()
    print("=" * 78)
    print("VLCP EQUIPMENT")
    print("=" * 78)

    print(
        f"{'TYPE':<15} | EQUIPMENT"
    )

    print("-" * 78)

    for item in report:

        print(
            f"{item['type']:<15} | "
            f"{item['equipment']}"
        )

    print("-" * 78)

    return report


print("✓ VLCP human equipment display contract loaded.")

✓ VLCP human equipment display contract loaded.


In [ ]:
# ============================================================
# CELL 25 — VLCP SINGLE USDOT EQUIPMENT TEST
# Corrected for existing equipment_index structure
# ============================================================

# CHANGE ONLY THIS NUMBER FOR FUTURE TESTS
TEST_DOT = "951224"


def test_vlcp_dot(usdot):

    usdot = str(usdot).strip()

    print("=" * 78)
    print("VLCP CARRIER / EQUIPMENT TEST")
    print("=" * 78)

    # --------------------------------------------------------
    # 1. CARRIER IDENTITY
    # --------------------------------------------------------

    identity = resolve_carrier_identity(usdot)

    company_name = (
        identity.get("display_name")
        or identity.get("legal_name")
        or "UNKNOWN CARRIER"
    )

    print(f"Company: {company_name}")
    print(f"USDOT:   {usdot}")
    print()
    print(
        "Carrier identity:",
        identity.get("identity_status", "UNKNOWN")
    )

    # --------------------------------------------------------
    # 2. FMCSA INSPECTION / EQUIPMENT EVIDENCE
    # --------------------------------------------------------

    inspection_result = get_fmcsa_inspection_units(usdot)

    inspection_status = inspection_result.get(
        "status",
        "UNKNOWN"
    )

    print(f"Inspection evidence: {inspection_status}")
    print(
        "Inspections found:",
        inspection_result.get("inspection_count", 0)
    )

    # --------------------------------------------------------
    # 3. NO-EVIDENCE / SOURCE-ERROR HANDLING
    # --------------------------------------------------------

    if inspection_status != "FOUND":

        print()
        print("=" * 78)
        print("VLCP RESULT")
        print("=" * 78)

        if inspection_status == "NO_INSPECTIONS_FOUND":

            print(
                "No FMCSA inspection equipment evidence "
                "was found for this USDOT."
            )

        else:

            print(
                "Equipment evidence could not be retrieved."
            )

        return {
            "usdot": usdot,
            "company_name": company_name,
            "status": inspection_status,
            "equipment": []
        }

    # --------------------------------------------------------
    # 4. NORMALIZE
    # --------------------------------------------------------

    normalized_rows = normalize_fmcsa_inspection_units(
        inspection_result
    )

    # --------------------------------------------------------
    # 5. UNIQUE EQUIPMENT INDEX
    # --------------------------------------------------------

    equipment_index = build_unique_equipment_index(
        normalized_rows
    )

    print(
        "Equipment observations:",
        len(normalized_rows)
    )

    print(
        "Likely unique equipment:",
        len(equipment_index)
    )

    # --------------------------------------------------------
    # 6. BUILD DISPLAY RECORDS
    #
    # Existing equipment_index structure:
    #
    # identity_key -> [observation, observation, ...]
    #
    # We preserve ALL observations while using the first
    # observation for the current human-readable display.
    # --------------------------------------------------------

    equipment_records = []

    for identity_key, observations in equipment_index.items():

        if not observations:
            continue

        first_observation = observations[0]

        # Collect every type ever observed for this equipment
        type_ids = sorted(
            {
                str(obs.get("unit_type_id"))
                for obs in observations
                if obs.get("unit_type_id") is not None
            },
            key=lambda x: (
                int(x)
                if x.isdigit()
                else 999
            )
        )

        if type_ids:

            type_label = (
                "FMCSA Type "
                + "/".join(type_ids)
            )

        else:

            type_label = "Equipment"

        # Use the normalized VIN from the observation.
        vin = first_observation.get(
            "vin_normalized"
        )

        # Preserve currently available make.
        make = first_observation.get(
            "make_raw"
        )

        equipment_records.append({

            "equipment_type": type_label,

            "vin": vin,

            # Not populated until authoritative
            # VIN decoding is connected.
            "year": None,

            "make": make,

            "model": None,

            # Keep audit information internally.
            "identity_key": identity_key,

            "observation_count": len(observations),

            "source_type_ids": type_ids
        })

    # --------------------------------------------------------
    # 7. HUMAN-READABLE OUTPUT
    # --------------------------------------------------------

    print_equipment_display_report(
        equipment_records
    )

    # --------------------------------------------------------
    # 8. STRUCTURED RESULT
    # --------------------------------------------------------

    return {

        "usdot":
            usdot,

        "company_name":
            company_name,

        "carrier_identity_status":
            identity.get("identity_status"),

        "inspection_status":
            inspection_status,

        "inspection_count":
            inspection_result.get(
                "inspection_count",
                0
            ),

        "equipment_observation_count":
            len(normalized_rows),

        "unique_equipment_count":
            len(equipment_index),

        "equipment":
            equipment_records
    }


# ============================================================
# RUN TEST
# ============================================================

vlcp_dot_test = test_vlcp_dot(TEST_DOT)

In [39]:
# ============================================================
# CELL 26 — VLCP AUTHORITATIVE VIN DECODER
# Source: NHTSA vPIC
#
# PURPOSE
# -------
# Decode VINs from the existing VLCP equipment index using
# NHTSA's authoritative vPIC service.
#
# IMPORTANT
# ---------
# - Does NOT modify Cells 19, 21, 22, 24, or 25.
# - Does NOT alter equipment_index.
# - Does NOT overwrite FMCSA observations.
# - Does NOT infer FMCSA unit type IDs.
# - Preserves provenance.
# - Works with arbitrary VINs / arbitrary USDOT pipelines.
# ============================================================

import requests
import time
from datetime import datetime, timezone


NHTSA_VPIC_BASE_URL = (
    "https://vpic.nhtsa.dot.gov/api/vehicles/"
    "DecodeVinValuesExtended"
)


def normalize_vin(vin):
    """
    Normalize a VIN without changing its substantive value.
    """

    if vin is None:
        return None

    vin = str(vin).strip().upper()

    if not vin:
        return None

    return vin


def is_candidate_vin(vin):
    """
    Basic gate before sending a value to NHTSA.

    Standard modern VINs contain 17 characters.
    This is an input-quality check, NOT proof that the VIN
    is valid or decodable.
    """

    vin = normalize_vin(vin)

    if not vin:
        return False

    return len(vin) == 17


def decode_vin_nhtsa(vin, timeout=30):
    """
    Decode one VIN using NHTSA vPIC.

    Returns a structured VLCP evidence object.

    No FMCSA equipment classification is inferred here.
    """

    vin = normalize_vin(vin)

    result = {
        "vin": vin,

        "decode_status": "NOT_ATTEMPTED",

        "year": None,
        "make": None,
        "model": None,

        "manufacturer": None,
        "vehicle_type": None,
        "body_class": None,
        "series": None,
        "trim": None,

        "nhtsa_error_code": None,
        "nhtsa_error_text": None,

        "provenance": {
            "source": "NHTSA vPIC",
            "source_agency":
                "National Highway Traffic Safety Administration",
            "retrieved_at_utc": None,
            "endpoint": None
        },

        "raw_decode": None
    }

    # --------------------------------------------------------
    # INPUT GATE
    # --------------------------------------------------------

    if not is_candidate_vin(vin):

        result["decode_status"] = "INVALID_VIN_FORMAT"

        result["nhtsa_error_text"] = (
            "VIN is missing or is not 17 characters."
        )

        return result

    # --------------------------------------------------------
    # REQUEST
    # --------------------------------------------------------

    url = f"{NHTSA_VPIC_BASE_URL}/{vin}"

    params = {
        "format": "json"
    }

    result["provenance"]["endpoint"] = url

    try:

        response = requests.get(
            url,
            params=params,
            timeout=timeout
        )

        response.raise_for_status()

        payload = response.json()

        rows = payload.get("Results") or []

        if not rows:

            result["decode_status"] = "NO_RESULT"

            result["nhtsa_error_text"] = (
                "NHTSA returned no VIN decode result."
            )

            return result

        decoded = rows[0]

        result["raw_decode"] = decoded

        # ----------------------------------------------------
        # AUTHORITATIVE DECODE FIELDS
        # ----------------------------------------------------

        result["year"] = (
            decoded.get("ModelYear") or None
        )

        result["make"] = (
            decoded.get("Make") or None
        )

        result["model"] = (
            decoded.get("Model") or None
        )

        result["manufacturer"] = (
            decoded.get("Manufacturer") or None
        )

        result["vehicle_type"] = (
            decoded.get("VehicleType") or None
        )

        result["body_class"] = (
            decoded.get("BodyClass") or None
        )

        result["series"] = (
            decoded.get("Series") or None
        )

        result["trim"] = (
            decoded.get("Trim") or None
        )

        result["nhtsa_error_code"] = (
            decoded.get("ErrorCode") or None
        )

        result["nhtsa_error_text"] = (
            decoded.get("ErrorText") or None
        )

        result["provenance"]["retrieved_at_utc"] = (
            datetime.now(
                timezone.utc
            ).isoformat()
        )

        # ----------------------------------------------------
        # DECODE STATUS
        #
        # We require useful identity information before
        # calling the VIN decoded.
        # ----------------------------------------------------

        useful_identity = any([
            result["year"],
            result["make"],
            result["model"],
            result["manufacturer"]
        ])

        if useful_identity:

            result["decode_status"] = "DECODED"

        else:

            result["decode_status"] = "UNRESOLVED"

        return result

    except requests.exceptions.Timeout:

        result["decode_status"] = "TIMEOUT"

        result["nhtsa_error_text"] = (
            "NHTSA vPIC request timed out."
        )

        return result

    except requests.exceptions.RequestException as exc:

        result["decode_status"] = "REQUEST_ERROR"

        result["nhtsa_error_text"] = str(exc)

        return result

    except Exception as exc:

        result["decode_status"] = "UNEXPECTED_ERROR"

        result["nhtsa_error_text"] = str(exc)

        return result


def collect_vins_from_equipment_index(equipment_index):
    """
    Collect unique VINs from the EXISTING verified VLCP
    equipment index.

    Expected index contract:

        identity_key -> list[observations]

    VIN-first identity keys are supported, but we also inspect
    observations so we do not depend solely on the key format.
    """

    vins = set()

    for identity_key, observations in equipment_index.items():

        # ----------------------------------------------------
        # VIN FROM IDENTITY KEY
        # ----------------------------------------------------

        if (
            isinstance(identity_key, str)
            and identity_key.startswith("VIN:")
        ):

            vin = normalize_vin(
                identity_key.split(
                    "VIN:",
                    1
                )[1]
            )

            if is_candidate_vin(vin):
                vins.add(vin)

        # ----------------------------------------------------
        # VIN FROM OBSERVATION HISTORY
        # ----------------------------------------------------

        if not isinstance(observations, list):
            continue

        for observation in observations:

            if not isinstance(observation, dict):
                continue

            vin = normalize_vin(
                observation.get("vin")
                or observation.get(
                    "insp_unit_vehicle_id_number"
                )
                or observation.get(
                    "vehicle_vin"
                )
            )

            if is_candidate_vin(vin):
                vins.add(vin)

    return sorted(vins)


def build_vin_decode_index(
    equipment_index,
    delay_seconds=0.10,
    max_vins=None
):
    """
    Build a separate VIN -> NHTSA decode evidence index.

    equipment_index is NEVER mutated.

    max_vins:
        Optional development/testing limit.
        None = process all candidate VINs.
    """

    vins = collect_vins_from_equipment_index(
        equipment_index
    )

    if max_vins is not None:
        vins = vins[:max_vins]

    decode_index = {}

    total = len(vins)

    print("=" * 70)
    print("VLCP AUTHORITATIVE VIN DECODING")
    print("=" * 70)

    print(
        f"Candidate VINs: {total}"
    )

    print(
        "Source: NHTSA vPIC"
    )

    print("-" * 70)

    for number, vin in enumerate(
        vins,
        start=1
    ):

        decode_index[vin] = (
            decode_vin_nhtsa(vin)
        )

        status = decode_index[
            vin
        ]["decode_status"]

        print(
            f"[{number}/{total}] "
            f"{vin} -> {status}"
        )

        # Gentle request pacing.
        if delay_seconds:
            time.sleep(delay_seconds)

    print("-" * 70)

    status_counts = {}

    for evidence in decode_index.values():

        status = evidence.get(
            "decode_status",
            "UNKNOWN"
        )

        status_counts[status] = (
            status_counts.get(
                status,
                0
            ) + 1
        )

    print("Decode summary:")

    for status, count in sorted(
        status_counts.items()
    ):

        print(
            f"  {status}: {count}"
        )

    print("=" * 70)

    return decode_index


def format_vin_identity(vin, vin_decode_index):
    """
    Human-readable VIN identity.

    Does NOT classify Tractor/Trailer yet.

    Examples:
        2018 Freightliner Cascadia
        2023 Dorsey
        VIN: 1ABC...
    """

    vin = normalize_vin(vin)

    evidence = vin_decode_index.get(
        vin,
        {}
    )

    if (
        evidence.get("decode_status")
        == "DECODED"
    ):

        parts = []

        if evidence.get("year"):
            parts.append(
                str(evidence["year"])
            )

        if evidence.get("make"):
            parts.append(
                str(evidence["make"])
            )

        if evidence.get("model"):
            parts.append(
                str(evidence["model"])
            )

        if parts:
            return " ".join(parts)

    return f"VIN: {vin}"


print(
    "✓ Cell 26 loaded — "
    "authoritative NHTSA VIN decoder ready"
)

✓ Cell 26 loaded — authoritative NHTSA VIN decoder ready


In [40]:
# ============================================================
# CELL 27 — CONTROLLED NHTSA VIN DECODE VALIDATION
#
# PURPOSE
# -------
# Validate Cell 26 against a small sample of unique VINs from
# the EXISTING verified equipment_index.
#
# DOES NOT:
# - modify equipment_index
# - modify Cells 19, 21, 22, 24, or 25
# - classify FMCSA unit type IDs
# - discard conflicting FMCSA observations
# ============================================================


# ------------------------------------------------------------
# PRE-FLIGHT CHECK
# ------------------------------------------------------------

if "equipment_index" not in globals():
    raise NameError(
        "equipment_index is not loaded. "
        "Run the verified equipment-index cell first."
    )

if "build_vin_decode_index" not in globals():
    raise NameError(
        "Cell 26 is not loaded."
    )


# ------------------------------------------------------------
# COUNT UNIQUE VIN CANDIDATES
# ------------------------------------------------------------

candidate_vins = collect_vins_from_equipment_index(
    equipment_index
)

print("=" * 70)
print("VLCP VIN DECODE VALIDATION")
print("=" * 70)

print(
    f"Unique candidate VINs in current equipment index: "
    f"{len(candidate_vins)}"
)

print(
    "Controlled test size: 5 VINs"
)

print(
    "Authoritative source: NHTSA vPIC"
)

print("=" * 70)


# ------------------------------------------------------------
# CONTROLLED DECODE
# ------------------------------------------------------------

vin_decode_test = build_vin_decode_index(
    equipment_index=equipment_index,
    delay_seconds=0.15,
    max_vins=5
)


# ------------------------------------------------------------
# HUMAN-READABLE RESULTS
# ------------------------------------------------------------

print()
print("=" * 70)
print("VLCP VIN DECODE TEST RESULTS")
print("=" * 70)

for vin, evidence in vin_decode_test.items():

    print()

    print(f"VIN: {vin}")

    print(
        f"Decode Status: "
        f"{evidence.get('decode_status')}"
    )

    print(
        f"Year: "
        f"{evidence.get('year') or 'Not verified'}"
    )

    print(
        f"Make: "
        f"{evidence.get('make') or 'Not verified'}"
    )

    print(
        f"Model: "
        f"{evidence.get('model') or 'Not verified'}"
    )

    print(
        f"Vehicle Type: "
        f"{evidence.get('vehicle_type') or 'Not verified'}"
    )

    print(
        f"Body Class: "
        f"{evidence.get('body_class') or 'Not verified'}"
    )

    print(
        f"Manufacturer: "
        f"{evidence.get('manufacturer') or 'Not verified'}"
    )

    print(
        "Human Identity: "
        f"{format_vin_identity(vin, vin_decode_test)}"
    )

    print(
        f"NHTSA Error Code: "
        f"{evidence.get('nhtsa_error_code') or 'None'}"
    )

    print(
        f"NHTSA Error Text: "
        f"{evidence.get('nhtsa_error_text') or 'None'}"
    )

    provenance = evidence.get(
        "provenance",
        {}
    )

    print(
        f"Evidence Source: "
        f"{provenance.get('source') or 'Unknown'}"
    )

    print("-" * 70)


# ------------------------------------------------------------
# INTEGRITY CHECK
# ------------------------------------------------------------

print()
print("CELL 27 INTEGRITY CHECK")
print("-" * 70)

print(
    "VINs requested:",
    min(5, len(candidate_vins))
)

print(
    "VIN evidence records returned:",
    len(vin_decode_test)
)

if len(vin_decode_test) == min(
    5,
    len(candidate_vins)
):

    print(
        "✓ Decode result count matches controlled test input"
    )

else:

    print(
        "⚠ Decode result count does not match expected input"
    )

print(
    "✓ Existing equipment_index was not intentionally modified"
)

print("=" * 70)

VLCP VIN DECODE VALIDATION
Unique candidate VINs in current equipment index: 194
Controlled test size: 5 VINs
Authoritative source: NHTSA vPIC
VLCP AUTHORITATIVE VIN DECODING
Candidate VINs: 5
Source: NHTSA vPIC
----------------------------------------------------------------------
[1/5] 1AJC40260T1002016 -> DECODED
[2/5] 1AJC40265T1002769 -> DECODED
[3/5] 1DW1C452X1E499580 -> DECODED
[4/5] 1FUGGHDV9LLLT8409 -> DECODED
[5/5] 1FUJA6CGX3LH34804 -> DECODED
----------------------------------------------------------------------
Decode summary:
  DECODED: 5

VLCP VIN DECODE TEST RESULTS

VIN: 1AJC40260T1002016
Decode Status: DECODED
Year: 2026
Make: HERCULES AJAX
Model: Not verified
Vehicle Type: TRAILER
Body Class: Not verified
Manufacturer: AJAX MANUFACTURING CO. INC.
Human Identity: 2026 HERCULES AJAX
NHTSA Error Code: 2,14
NHTSA Error Text: 2 - VIN corrected, error in one position; 14 - Unable to provide information for some of the characters in the VIN, based on the manufacturer submiss

In [41]:
# ============================================================
# CELL 28 — VLCP VIN EVIDENCE QUALITY LAYER
#
# PURPOSE
# -------
# Evaluate NHTSA vPIC decode quality without changing the
# original Cell 26 decoder or its evidence.
#
# PRESERVES:
# - original VIN
# - raw NHTSA response
# - NHTSA error codes
# - NHTSA vehicle type
# - NHTSA body class
# - FMCSA observations
# - conflicting FMCSA type IDs
#
# DOES NOT:
# - mutate equipment_index
# - overwrite Cell 26 results
# - classify FMCSA unit type IDs
# ============================================================


def parse_nhtsa_error_codes(error_code):
    """
    Convert NHTSA's comma-separated ErrorCode field into
    a normalized set of string codes.

    Examples:
        "0"    -> {"0"}
        "0,14" -> {"0", "14"}
        "2,14" -> {"2", "14"}
    """

    if error_code is None:
        return set()

    return {
        code.strip()
        for code in str(error_code).split(",")
        if code.strip()
    }


def assess_nhtsa_decode_quality(decode_evidence):
    """
    Add a VLCP evidence-quality assessment to an existing
    NHTSA VIN decode WITHOUT modifying the source evidence.

    Quality statuses:

        VERIFIED_CLEAN
            NHTSA decoded the VIN with code 0 and useful
            vehicle identity information exists.

        VERIFIED_PARTIAL
            VIN decoded, but NHTSA reports incomplete
            manufacturer-submitted information such as
            error code 14.

        CORRECTED_DECODE
            NHTSA reports that the submitted VIN required
            correction, such as error code 2.

        UNRESOLVED
            No useful authoritative identity was produced.

        REQUEST_FAILURE
            External lookup did not complete successfully.
    """

    decode_status = decode_evidence.get(
        "decode_status"
    )

    codes = parse_nhtsa_error_codes(
        decode_evidence.get(
            "nhtsa_error_code"
        )
    )

    year = decode_evidence.get("year")
    make = decode_evidence.get("make")
    model = decode_evidence.get("model")

    vehicle_type = decode_evidence.get(
        "vehicle_type"
    )

    body_class = decode_evidence.get(
        "body_class"
    )

    useful_identity = any([
        year,
        make,
        model,
        vehicle_type,
        body_class
    ])

    # --------------------------------------------------------
    # REQUEST / INPUT FAILURES
    # --------------------------------------------------------

    if decode_status in {
        "TIMEOUT",
        "REQUEST_ERROR",
        "UNEXPECTED_ERROR"
    }:

        quality_status = "REQUEST_FAILURE"

    elif decode_status in {
        "INVALID_VIN_FORMAT",
        "NO_RESULT",
        "UNRESOLVED",
        "NOT_ATTEMPTED"
    }:

        quality_status = "UNRESOLVED"

    # --------------------------------------------------------
    # VIN CORRECTION
    # --------------------------------------------------------
    #
    # Code 2 is important for VLCP because NHTSA is telling
    # us the submitted VIN required correction.
    #
    # We preserve the returned decode, but do not treat it
    # as equivalent to a clean VIN match.
    # --------------------------------------------------------

    elif "2" in codes:

        quality_status = "CORRECTED_DECODE"

    # --------------------------------------------------------
    # PARTIAL DECODE
    # --------------------------------------------------------

    elif "14" in codes and useful_identity:

        quality_status = "VERIFIED_PARTIAL"

    # --------------------------------------------------------
    # CLEAN DECODE
    # --------------------------------------------------------

    elif (
        "0" in codes
        and useful_identity
    ):

        quality_status = "VERIFIED_CLEAN"

    elif useful_identity:

        quality_status = "VERIFIED_PARTIAL"

    else:

        quality_status = "UNRESOLVED"

    return {
        "quality_status": quality_status,

        "nhtsa_error_codes": sorted(codes),

        "identity_fields": {
            "year": year,
            "make": make,
            "model": model,
            "vehicle_type": vehicle_type,
            "body_class": body_class
        },

        "source_decode_status":
            decode_status,

        "source_error_text":
            decode_evidence.get(
                "nhtsa_error_text"
            ),

        "provenance": {
            "source": "NHTSA vPIC",
            "assessment_layer": "VLCP",
            "source_evidence_preserved": True
        }
    }


def build_vin_quality_index(
    vin_decode_index
):
    """
    Build a separate VIN -> VLCP quality assessment index.

    The original NHTSA decode index is not modified.
    """

    quality_index = {}

    for vin, evidence in (
        vin_decode_index.items()
    ):

        quality_index[vin] = (
            assess_nhtsa_decode_quality(
                evidence
            )
        )

    return quality_index


# ============================================================
# APPLY TO CELL 27 CONTROLLED TEST
# ============================================================

vin_quality_test = build_vin_quality_index(
    vin_decode_test
)


print("=" * 70)
print("VLCP VIN EVIDENCE QUALITY TEST")
print("=" * 70)


for vin, quality in vin_quality_test.items():

    original = vin_decode_test.get(
        vin,
        {}
    )

    print()

    print(f"VIN: {vin}")

    print(
        "Quality Status:",
        quality.get(
            "quality_status"
        )
    )

    print(
        "NHTSA Codes:",
        ", ".join(
            quality.get(
                "nhtsa_error_codes",
                []
            )
        ) or "None"
    )

    print(
        "Vehicle Type:",
        original.get(
            "vehicle_type"
        ) or "Not verified"
    )

    print(
        "Body Class:",
        original.get(
            "body_class"
        ) or "Not verified"
    )

    print(
        "Human Identity:",
        format_vin_identity(
            vin,
            vin_decode_test
        )
    )

    print("-" * 70)


# ============================================================
# SUMMARY
# ============================================================

quality_counts = {}

for quality in vin_quality_test.values():

    status = quality.get(
        "quality_status",
        "UNKNOWN"
    )

    quality_counts[status] = (
        quality_counts.get(
            status,
            0
        ) + 1
    )


print()
print("QUALITY SUMMARY")
print("-" * 70)

for status, count in sorted(
    quality_counts.items()
):

    print(
        f"{status}: {count}"
    )


print()
print("INTEGRITY")
print("-" * 70)

print(
    "Source VIN evidence records:",
    len(vin_decode_test)
)

print(
    "Quality records:",
    len(vin_quality_test)
)

if len(vin_quality_test) == len(
    vin_decode_test
):

    print(
        "✓ One quality record per source VIN"
    )

else:

    print(
        "⚠ Quality/source record count mismatch"
    )

print(
    "✓ Original NHTSA evidence preserved"
)

print(
    "✓ equipment_index not modified"
)

print("=" * 70)

VLCP VIN EVIDENCE QUALITY TEST

VIN: 1AJC40260T1002016
Quality Status: CORRECTED_DECODE
NHTSA Codes: 14, 2
Vehicle Type: TRAILER
Body Class: Not verified
Human Identity: 2026 HERCULES AJAX
----------------------------------------------------------------------

VIN: 1AJC40265T1002769
Quality Status: CORRECTED_DECODE
NHTSA Codes: 14, 2
Vehicle Type: TRAILER
Body Class: Not verified
Human Identity: 2026 HERCULES AJAX
----------------------------------------------------------------------

VIN: 1DW1C452X1E499580
Quality Status: VERIFIED_PARTIAL
NHTSA Codes: 0, 14
Vehicle Type: TRAILER
Body Class: Trailer
Human Identity: 2001 STOUGHTON TRAILERS Stoughton Trailers
----------------------------------------------------------------------

VIN: 1FUGGHDV9LLLT8409
Quality Status: VERIFIED_CLEAN
NHTSA Codes: 0
Vehicle Type: TRUCK
Body Class: Truck-Tractor
Human Identity: 2020 FREIGHTLINER Cascadia
----------------------------------------------------------------------

VIN: 1FUJA6CGX3LH34804
Quality S

In [42]:
# ============================================================
# NEXT CELL — VLCP BROKER / INVESTOR EQUIPMENT SUMMARY
#
# DISPLAY CONTRACT
# ------------------------------------------------------------
# Carrier + USDOT
# Equipment | Year | Make | Model | VIN | Verification
# Fleet evidence counts
# Sources
#
# Backend evidence remains preserved but is not displayed.
# ============================================================

def clean_display_value(value, fallback="—"):
    """Return a clean human-facing value."""
    if value is None:
        return fallback

    value = str(value).strip()

    if not value:
        return fallback

    return value


def title_display(value):
    """
    Clean capitalization for broker-facing display
    without changing the stored source evidence.
    """
    if not value:
        return "—"

    return str(value).strip().title()


def get_display_equipment_type(vin_evidence):
    """
    Produce a simple human equipment category from
    authoritative NHTSA VIN evidence.

    This is DISPLAY ONLY.

    It does not overwrite or replace historical FMCSA
    equipment-type observations.
    """

    vehicle_type = str(
        vin_evidence.get("vehicle_type") or ""
    ).upper()

    body_class = str(
        vin_evidence.get("body_class") or ""
    ).upper()

    combined = f"{vehicle_type} {body_class}"

    if "TRAILER" in combined:
        return "Trailer"

    if (
        "TRUCK-TRACTOR" in combined
        or "TRUCK TRACTOR" in combined
        or "TRACTOR" in combined
    ):
        return "Tractor"

    if "TRUCK" in combined:
        return "Truck"

    return "Equipment"


def get_display_verification(vin_evidence):
    """
    Convert backend VIN evidence status into a concise
    broker-facing verification label.
    """

    status = vin_evidence.get(
        "decode_status"
    )

    error_codes = parse_nhtsa_error_codes(
        vin_evidence.get(
            "nhtsa_error_code"
        )
    )

    # NHTSA corrected the supplied VIN.
    if "2" in error_codes:
        return "Partial"

    # Useful decode but incomplete manufacturer information.
    if "14" in error_codes:
        return "Partial"

    # Clean authoritative decode.
    if status == "DECODED":
        return "Verified"

    return "Unverified"


def print_vlcp_equipment_summary(
    company_name,
    usdot,
    equipment_index,
    vin_decode_index,
    inspection_count=None,
    observation_count=None
):
    """
    Print the standard VLCP broker/investor-facing
    equipment summary.

    Does not modify source evidence.
    """

    # --------------------------------------------------------
    # BUILD DISPLAY ROWS
    # --------------------------------------------------------

    rows = []

    for vin, evidence in vin_decode_index.items():

        rows.append({
            "equipment":
                get_display_equipment_type(evidence),

            "year":
                clean_display_value(
                    evidence.get("year")
                ),

            "make":
                title_display(
                    evidence.get("make")
                ),

            "model":
                title_display(
                    evidence.get("model")
                ),

            "vin":
                vin,

            "verification":
                get_display_verification(
                    evidence
                )
        })

    # --------------------------------------------------------
    # SORT FOR CONSISTENT HUMAN OUTPUT
    # --------------------------------------------------------

    equipment_order = {
        "Tractor": 1,
        "Truck": 2,
        "Trailer": 3,
        "Equipment": 4
    }

    rows.sort(
        key=lambda row: (
            equipment_order.get(
                row["equipment"],
                99
            ),
            str(row["year"]),
            row["make"],
            row["vin"]
        )
    )

    # --------------------------------------------------------
    # HEADER
    # --------------------------------------------------------

    print()
    print("VLCP Verified Equipment Summary")
    print()

    print(
        f"Carrier: "
        f"{clean_display_value(company_name)}"
    )

    print(
        f"USDOT: "
        f"{clean_display_value(usdot)}"
    )

    print()

    # --------------------------------------------------------
    # TABLE
    # --------------------------------------------------------

    headers = [
        "Equipment",
        "Year",
        "Make",
        "Model",
        "VIN",
        "Verification"
    ]

    table_rows = [
        [
            row["equipment"],
            row["year"],
            row["make"],
            row["model"],
            row["vin"],
            row["verification"]
        ]
        for row in rows
    ]

    all_rows = [headers] + table_rows

    widths = [
        max(
            len(str(row[column]))
            for row in all_rows
        )
        for column in range(
            len(headers)
        )
    ]

    header_line = " | ".join(
        str(headers[i]).ljust(widths[i])
        for i in range(len(headers))
    )

    separator = "-+-".join(
        "-" * width
        for width in widths
    )

    print(header_line)
    print(separator)

    for row in table_rows:

        print(
            " | ".join(
                str(row[i]).ljust(widths[i])
                for i in range(len(headers))
            )
        )

    # --------------------------------------------------------
    # FLEET EVIDENCE SUMMARY
    # --------------------------------------------------------

    print()
    print("Fleet Evidence Summary")
    print()

    evidence_parts = []

    if inspection_count is not None:
        evidence_parts.append(
            f"{inspection_count:,} FMCSA inspections"
        )

    if observation_count is not None:
        evidence_parts.append(
            f"{observation_count:,} equipment observations"
        )

    evidence_parts.append(
        f"{len(equipment_index):,} likely unique equipment"
    )

    evidence_parts.append(
        f"{len(vin_decode_index):,} VIN-identified assets"
    )

    print(
        " · ".join(evidence_parts)
    )

    print()
    print(
        "Sources: FMCSA inspection records + "
        "NHTSA VIN records"
    )


print(
    "✓ VLCP broker/investor equipment summary display loaded"
)

✓ VLCP broker/investor equipment summary display loaded


In [43]:
# ============================================================
# TEST — VLCP BROKER / INVESTOR EQUIPMENT SUMMARY
# Uses the 5 VINs already decoded in Cell 27.
# No new API requests.
# ============================================================

print_vlcp_equipment_summary(
    company_name="US SERVICES LLC",
    usdot="951224",

    # Existing verified equipment index
    equipment_index=equipment_index,

    # Five-VIN controlled NHTSA test from Cell 27
    vin_decode_index=vin_decode_test,

    # Verified FMCSA counts for USDOT 951224
    inspection_count=143,
    observation_count=264
)


VLCP Verified Equipment Summary

Carrier: US SERVICES LLC
USDOT: 951224

Equipment | Year | Make               | Model              | VIN               | Verification
----------+------+--------------------+--------------------+-------------------+-------------
Tractor   | 2003 | Freightliner       | Columbia           | 1FUJA6CGX3LH34804 | Verified    
Tractor   | 2020 | Freightliner       | Cascadia           | 1FUGGHDV9LLLT8409 | Verified    
Trailer   | 2001 | Stoughton Trailers | Stoughton Trailers | 1DW1C452X1E499580 | Partial     
Trailer   | 2026 | Hercules Ajax      | —                  | 1AJC40260T1002016 | Partial     
Trailer   | 2026 | Hercules Ajax      | —                  | 1AJC40265T1002769 | Partial     

Fleet Evidence Summary

143 FMCSA inspections · 264 equipment observations · 201 likely unique equipment · 5 VIN-identified assets

Sources: FMCSA inspection records + NHTSA VIN records


In [44]:
# ============================================================
# VLCP — OPERATIONAL EQUIPMENT REPORT
# One USDOT -> finished broker-facing equipment report
# ============================================================

def run_vlcp_operational_equipment(
    usdot,
    vin_limit=10,
    decode_delay=0.15
):
    """
    Run the verified VLCP equipment workflow for any USDOT.
    """

    usdot = str(usdot).strip()

    if not usdot:
        raise ValueError("USDOT is required.")

    if not usdot.isdigit():
        raise ValueError("USDOT must contain numbers only.")

    # --------------------------------------------------------
    # VERIFY REQUIRED EXISTING COMPONENTS
    # --------------------------------------------------------

    required_functions = [
        "get_fmcsa_carrier",
        "normalize_fmcsa_carrier_result",
        "get_fmcsa_inspection_units",
        "build_equipment_index",
        "build_vin_decode_index",
        "print_vlcp_equipment_summary"
    ]

    missing = [
        name
        for name in required_functions
        if name not in globals()
    ]

    if missing:
        raise RuntimeError(
            "Required VLCP components are not loaded: "
            + ", ".join(missing)
        )

    # --------------------------------------------------------
    # CARRIER IDENTITY
    # --------------------------------------------------------

    carrier_result = get_fmcsa_carrier(
        usdot,
        webkey=FMCSA_WEBKEY
    )

    if not carrier_result.get("success"):
        raise RuntimeError(
            "FMCSA carrier lookup failed: "
            + str(carrier_result.get("error"))
        )

    carrier_evidence = normalize_fmcsa_carrier_result(
        carrier_result
    )

    carrier = carrier_evidence.get(
        "carrier",
        {}
    )

    company_name = (
        carrier.get("legal_name")
        or carrier.get("dba_name")
        or "Unknown Carrier"
    )

    # --------------------------------------------------------
    # FMCSA INSPECTION / EQUIPMENT ACQUISITION
    # --------------------------------------------------------

    inspection_result = get_fmcsa_inspection_units(
        usdot
    )

    if not inspection_result.get("success"):
        raise RuntimeError(
            "FMCSA inspection retrieval failed."
        )

    inspection_units = inspection_result.get(
        "inspection_units",
        []
    )

    inspection_count = inspection_result.get(
        "inspection_count",
        0
    )

    observation_count = len(
        inspection_units
    )

    # --------------------------------------------------------
    # EXISTING VERIFIED EQUIPMENT NORMALIZATION
    # --------------------------------------------------------

    current_equipment_index = build_equipment_index(
        inspection_units
    )

    # --------------------------------------------------------
    # NHTSA VIN DECODING
    # --------------------------------------------------------

    current_vin_decode_index = build_vin_decode_index(
        equipment_index=current_equipment_index,
        delay_seconds=decode_delay,
        max_vins=vin_limit
    )

    # --------------------------------------------------------
    # BROKER / INVESTOR OUTPUT
    # --------------------------------------------------------

    print_vlcp_equipment_summary(
        company_name=company_name,
        usdot=usdot,
        equipment_index=current_equipment_index,
        vin_decode_index=current_vin_decode_index,
        inspection_count=inspection_count,
        observation_count=observation_count
    )

    # --------------------------------------------------------
    # RETURN DATA FOR FUTURE API / MCP USE
    # --------------------------------------------------------

    return {
        "company_name": company_name,
        "usdot": usdot,
        "inspection_count": inspection_count,
        "equipment_observation_count": observation_count,
        "likely_unique_equipment": len(
            current_equipment_index
        ),
        "vin_identified_assets": len(
            current_vin_decode_index
        ),
        "equipment_index": current_equipment_index,
        "vin_decode_index": current_vin_decode_index
    }


# ------------------------------------------------------------
# CONFIRM FUNCTION ACTUALLY EXISTS IN CURRENT RUNTIME
# ------------------------------------------------------------

assert "run_vlcp_operational_equipment" in globals()

print(
    "✓ run_vlcp_operational_equipment() "
    "is loaded in the current runtime"
)

✓ run_vlcp_operational_equipment() is loaded in the current runtime


In [46]:
# ============================================================
# VLCP — FIND EXISTING EQUIPMENT INDEX FUNCTION
# Read-only diagnostic. Changes nothing.
# ============================================================

print("VLCP EQUIPMENT FUNCTION DISCOVERY")
print("=" * 55)

keywords = [
    "equipment",
    "index",
    "normalize",
    "dedup",
    "inspection"
]

matches = []

for name, obj in globals().items():

    if not callable(obj):
        continue

    name_lower = name.lower()

    if any(
        keyword in name_lower
        for keyword in keywords
    ):
        matches.append(name)

for name in sorted(set(matches)):
    print(name)

print("=" * 55)

VLCP EQUIPMENT FUNCTION DISCOVERY
analyze_equipment_type_evidence
build_equipment_display
build_equipment_display_report
build_equipment_evidence_from_inspections
build_unique_equipment_index
build_vin_decode_index
build_vin_quality_index
collect_vins_from_equipment_index
convert_inspection_units_to_vlcp_evidence
get_display_equipment_type
get_fmcsa_inspection_units
normalize_bol_to_vlcp
normalize_equipment_value
normalize_fmcsa_authority_result
normalize_fmcsa_carrier_result
normalize_fmcsa_dot_for_actpend
normalize_fmcsa_inspection_units
normalize_fmcsa_insurance_lifecycle
normalize_fmcsa_insurance_result
normalize_name
normalize_vin
print_equipment_display_report
print_vlcp_equipment_summary
profile_fmcsa_inspection_units
run_equipment_pipeline_test
run_vlcp_equipment_test_suite
run_vlcp_operational_equipment
summarize_equipment_evidence
verify_equipment_claim


In [47]:
# ============================================================
# VLCP — OPERATIONAL COMPATIBILITY FIX
#
# Maps the operational wrapper to the already-verified
# equipment index function.
#
# No verified equipment logic is changed.
# ============================================================

if "build_unique_equipment_index" not in globals():
    raise RuntimeError(
        "Verified build_unique_equipment_index() is not loaded."
    )

# Compatibility alias for the operational wrapper
build_equipment_index = build_unique_equipment_index

# Verify the alias
assert build_equipment_index is build_unique_equipment_index

print(
    "✓ Operational pipeline connected to "
    "verified build_unique_equipment_index()"
)

✓ Operational pipeline connected to verified build_unique_equipment_index()


In [48]:
vlcp_result = run_vlcp_operational_equipment(
    usdot="951224",
    vin_limit=10,
    decode_delay=0.15
)

VLCP AUTHORITATIVE VIN DECODING
Candidate VINs: 10
Source: NHTSA vPIC
----------------------------------------------------------------------
[1/10] 1AJC40260T1002016 -> DECODED
[2/10] 1AJC40265T1002769 -> DECODED
[3/10] 1DW1C452X1E499580 -> DECODED
[4/10] 1FUGGHDV9LLLT8409 -> DECODED
[5/10] 1FUJA6CGX3LH34804 -> DECODED
[6/10] 1FUJA6CK28DY52591 -> DECODED
[7/10] 1FUJA6CK36LV99316 -> DECODED
[8/10] 1FUJA6CK47LX77848 -> DECODED
[9/10] 1FUJA6CK57LW40286 -> DECODED
[10/10] 1FUJA6CK65LN64042 -> DECODED
----------------------------------------------------------------------
Decode summary:
  DECODED: 10

VLCP Verified Equipment Summary

Carrier: US SERVICES LLC
USDOT: 951224

Equipment | Year | Make               | Model              | VIN               | Verification
----------+------+--------------------+--------------------+-------------------+-------------
Tractor   | 2003 | Freightliner       | Columbia           | 1FUJA6CGX3LH34804 | Verified    
Tractor   | 2005 | Freightliner       | C

In [49]:
# ============================================================
# VLCP — VERIFIED EQUIPMENT INDEX CONTRACT INSPECTION
# Read-only. Does not modify any functions or data.
# ============================================================

import inspect

print("VLCP VERIFIED EQUIPMENT INDEX CONTRACT")
print("=" * 60)

print("\nFUNCTION SIGNATURE")
print("-" * 60)

print(
    inspect.signature(
        build_unique_equipment_index
    )
)

print("\nFUNCTION SOURCE")
print("-" * 60)

print(
    inspect.getsource(
        build_unique_equipment_index
    )
)

print("=" * 60)

VLCP VERIFIED EQUIPMENT INDEX CONTRACT

FUNCTION SIGNATURE
------------------------------------------------------------
(normalized_rows)

FUNCTION SOURCE
------------------------------------------------------------
def build_unique_equipment_index(normalized_rows):
    """
    Group repeated FMCSA inspection observations into likely
    unique equipment records.

    VIN is used as the primary identity key when available.

    If VIN is unavailable, state + license is used as a
    secondary evidence key.

    Records lacking both are preserved rather than discarded.
    """

    equipment_index = defaultdict(list)

    unidentified_counter = 0

    for row in normalized_rows:

        vin = row.get("vin_normalized")
        plate = row.get("license_normalized")
        state = row.get("license_state")

        if vin:
            identity_key = f"VIN:{vin}"

        elif plate and state:
            identity_key = (
                f"PLATE:{state}:{plate}"
            )

        else

In [50]:
# ============================================================
# VLCP — CORRECTED OPERATIONAL EQUIPMENT PIPELINE
#
# Correct flow:
#
# Raw FMCSA observations
#       ↓
# normalize_fmcsa_inspection_units()
#       ↓
# build_unique_equipment_index()
#       ↓
# NHTSA VIN decoding
#       ↓
# Broker-facing VLCP report
#
# Existing verified functions are NOT modified.
# ============================================================

def run_vlcp_operational_equipment(
    usdot,
    vin_limit=10,
    decode_delay=0.15
):
    """
    Operational VLCP equipment workflow for any USDOT.
    """

    usdot = str(usdot).strip()

    if not usdot:
        raise ValueError("USDOT is required.")

    if not usdot.isdigit():
        raise ValueError(
            "USDOT must contain numbers only."
        )

    # --------------------------------------------------------
    # REQUIRED VERIFIED COMPONENTS
    # --------------------------------------------------------

    required_functions = [
        "get_fmcsa_carrier",
        "normalize_fmcsa_carrier_result",
        "get_fmcsa_inspection_units",
        "normalize_fmcsa_inspection_units",
        "build_unique_equipment_index",
        "build_vin_decode_index",
        "print_vlcp_equipment_summary"
    ]

    missing = [
        name
        for name in required_functions
        if name not in globals()
    ]

    if missing:
        raise RuntimeError(
            "Required VLCP components are not loaded: "
            + ", ".join(missing)
        )

    # ========================================================
    # 1. CARRIER IDENTITY
    # ========================================================

    carrier_result = get_fmcsa_carrier(
        usdot,
        webkey=FMCSA_WEBKEY
    )

    if not carrier_result.get("success"):

        raise RuntimeError(
            "FMCSA carrier lookup failed: "
            + str(
                carrier_result.get(
                    "error",
                    "Unknown FMCSA error"
                )
            )
        )

    carrier_evidence = (
        normalize_fmcsa_carrier_result(
            carrier_result
        )
    )

    carrier = carrier_evidence.get(
        "carrier",
        {}
    )

    company_name = (
        carrier.get("legal_name")
        or carrier.get("dba_name")
        or "Unknown Carrier"
    )

    # ========================================================
    # 2. FMCSA INSPECTION ACQUISITION
    # ========================================================

    inspection_result = (
        get_fmcsa_inspection_units(
            usdot
        )
    )

    if not inspection_result.get(
        "success"
    ):

        raise RuntimeError(
            "FMCSA inspection retrieval failed."
        )

    raw_inspection_units = (
        inspection_result.get(
            "inspection_units",
            []
        )
    )

    inspection_count = (
        inspection_result.get(
            "inspection_count",
            0
        )
    )

    observation_count = len(
        raw_inspection_units
    )

    # ========================================================
    # 3. VERIFIED EQUIPMENT NORMALIZATION
    # ========================================================

    normalized_equipment = (
        normalize_fmcsa_inspection_units(
            raw_inspection_units
        )
    )

    # ========================================================
    # 4. VERIFIED EQUIPMENT DEDUPLICATION
    # ========================================================

    current_equipment_index = (
        build_unique_equipment_index(
            normalized_equipment
        )
    )

    unique_equipment_count = len(
        current_equipment_index
    )

    # ========================================================
    # 5. NHTSA VIN DECODING
    # ========================================================

    current_vin_decode_index = (
        build_vin_decode_index(
            equipment_index=
                current_equipment_index,

            delay_seconds=
                decode_delay,

            max_vins=
                vin_limit
        )
    )

    # ========================================================
    # 6. BROKER / INVESTOR OUTPUT
    # ========================================================

    print_vlcp_equipment_summary(
        company_name=
            company_name,

        usdot=
            usdot,

        equipment_index=
            current_equipment_index,

        vin_decode_index=
            current_vin_decode_index,

        inspection_count=
            inspection_count,

        observation_count=
            observation_count
    )

    # ========================================================
    # 7. STRUCTURED OPERATIONAL RESULT
    # ========================================================

    return {

        "company_name":
            company_name,

        "usdot":
            usdot,

        "inspection_count":
            inspection_count,

        "equipment_observation_count":
            observation_count,

        "likely_unique_equipment":
            unique_equipment_count,

        "vin_identified_assets":
            len(
                current_vin_decode_index
            ),

        "equipment_index":
            current_equipment_index,

        "vin_decode_index":
            current_vin_decode_index,

        "sources": [
            "FMCSA inspection records",
            "NHTSA vPIC"
        ]
    }


print(
    "✓ Corrected VLCP operational equipment pipeline loaded"
)

✓ Corrected VLCP operational equipment pipeline loaded


In [51]:
# ============================================================
# VLCP — OPERATIONAL EQUIPMENT PIPELINE v2
# Fix: pass complete inspection_result into the VERIFIED
# normalize_fmcsa_inspection_units() function.
# ============================================================

def run_vlcp_operational_equipment(
    usdot,
    vin_limit=10,
    decode_delay=0.15
):
    """
    One USDOT -> finished VLCP equipment report.

    Reuses existing verified VLCP components.
    """

    usdot = str(usdot).strip()

    if not usdot:
        raise ValueError("USDOT is required.")

    if not usdot.isdigit():
        raise ValueError(
            "USDOT must contain numbers only."
        )

    # --------------------------------------------------------
    # 1. CARRIER IDENTITY
    # --------------------------------------------------------

    carrier_result = get_fmcsa_carrier(
        usdot,
        webkey=FMCSA_WEBKEY
    )

    if not carrier_result.get("success"):
        raise RuntimeError(
            "FMCSA carrier lookup failed: "
            + str(
                carrier_result.get(
                    "error",
                    "Unknown error"
                )
            )
        )

    carrier_evidence = (
        normalize_fmcsa_carrier_result(
            carrier_result
        )
    )

    carrier = carrier_evidence.get(
        "carrier",
        {}
    )

    company_name = (
        carrier.get("legal_name")
        or carrier.get("dba_name")
        or "Unknown Carrier"
    )

    # --------------------------------------------------------
    # 2. FMCSA INSPECTION ACQUISITION
    # --------------------------------------------------------

    inspection_result = (
        get_fmcsa_inspection_units(
            usdot
        )
    )

    if not inspection_result.get("success"):
        raise RuntimeError(
            "FMCSA inspection retrieval failed."
        )

    raw_inspection_units = (
        inspection_result.get(
            "inspection_units",
            []
        )
    )

    inspection_count = (
        inspection_result.get(
            "inspection_count",
            0
        )
    )

    observation_count = len(
        raw_inspection_units
    )

    # --------------------------------------------------------
    # 3. VERIFIED NORMALIZATION
    #
    # IMPORTANT:
    # normalize_fmcsa_inspection_units expects the COMPLETE
    # inspection_result dictionary.
    # --------------------------------------------------------

    normalized_equipment = (
        normalize_fmcsa_inspection_units(
            inspection_result
        )
    )

    # --------------------------------------------------------
    # 4. VERIFIED DEDUPLICATION
    # --------------------------------------------------------

    current_equipment_index = (
        build_unique_equipment_index(
            normalized_equipment
        )
    )

    # --------------------------------------------------------
    # 5. NHTSA VIN DECODING
    # --------------------------------------------------------

    current_vin_decode_index = (
        build_vin_decode_index(
            equipment_index=
                current_equipment_index,

            delay_seconds=
                decode_delay,

            max_vins=
                vin_limit
        )
    )

    # --------------------------------------------------------
    # 6. CLEAN VLCP OUTPUT
    # --------------------------------------------------------

    print_vlcp_equipment_summary(
        company_name=
            company_name,

        usdot=
            usdot,

        equipment_index=
            current_equipment_index,

        vin_decode_index=
            current_vin_decode_index,

        inspection_count=
            inspection_count,

        observation_count=
            observation_count
    )

    # --------------------------------------------------------
    # 7. OPERATIONAL RESULT
    # --------------------------------------------------------

    return {
        "company_name":
            company_name,

        "usdot":
            usdot,

        "inspection_count":
            inspection_count,

        "equipment_observation_count":
            observation_count,

        "likely_unique_equipment":
            len(current_equipment_index),

        "vin_identified_assets":
            len(current_vin_decode_index),

        "equipment_index":
            current_equipment_index,

        "vin_decode_index":
            current_vin_decode_index,

        "sources": [
            "FMCSA inspection records",
            "NHTSA vPIC"
        ]
    }


print(
    "✓ VLCP operational equipment pipeline v2 loaded"
)

✓ VLCP operational equipment pipeline v2 loaded


In [52]:
vlcp_result = run_vlcp_operational_equipment(
    usdot="951224",
    vin_limit=10,
    decode_delay=0.15
)

VLCP AUTHORITATIVE VIN DECODING
Candidate VINs: 10
Source: NHTSA vPIC
----------------------------------------------------------------------
[1/10] 1AJC40260T1002016 -> DECODED
[2/10] 1AJC40265T1002769 -> DECODED
[3/10] 1DW1C452X1E499580 -> DECODED
[4/10] 1FUGGHDV9LLLT8409 -> DECODED
[5/10] 1FUJA6CGX3LH34804 -> DECODED
[6/10] 1FUJA6CK28DY52591 -> DECODED
[7/10] 1FUJA6CK36LV99316 -> DECODED
[8/10] 1FUJA6CK47LX77848 -> DECODED
[9/10] 1FUJA6CK57LW40286 -> DECODED
[10/10] 1FUJA6CK65LN64042 -> DECODED
----------------------------------------------------------------------
Decode summary:
  DECODED: 10

VLCP Verified Equipment Summary

Carrier: US SERVICES LLC
USDOT: 951224

Equipment | Year | Make               | Model              | VIN               | Verification
----------+------+--------------------+--------------------+-------------------+-------------
Tractor   | 2003 | Freightliner       | Columbia           | 1FUJA6CGX3LH34804 | Verified    
Tractor   | 2005 | Freightliner       | C

In [53]:
vlcp_result_80806 = run_vlcp_operational_equipment(
    usdot="80806",
    vin_limit=10,
    decode_delay=0.15
)

VLCP AUTHORITATIVE VIN DECODING
Candidate VINs: 10
Source: NHTSA vPIC
----------------------------------------------------------------------
[1/10] 00000000000000000 -> UNRESOLVED
[2/10] 00000000HFT003497 -> UNRESOLVED
[3/10] 109FA4022D1257509 -> UNRESOLVED
[4/10] 13N148200C1554153 -> DECODED
[5/10] 13N148201C1554132 -> DECODED
[6/10] 13N148202C1554137 -> DECODED
[7/10] 13N148202C1554140 -> DECODED
[8/10] 13N148202N1550818 -> DECODED
[9/10] 13N148202N1550821 -> DECODED
[10/10] 13N148203L1541848 -> DECODED
----------------------------------------------------------------------
Decode summary:
  DECODED: 7
  UNRESOLVED: 3

VLCP Verified Equipment Summary

Carrier: J B HUNT TRANSPORT INC
USDOT: 80806

Equipment | Year | Make                 | Model                | VIN               | Verification
----------+------+----------------------+----------------------+-------------------+-------------
Trailer   | 2012 | Fontaine Trailer Co. | Fontaine Trailer Co. | 13N148200C1554153 | Verified    

In [ ]:
# ============================================================
# VLCP — VIN CANDIDATE QUALITY DIAGNOSTIC
# Read-only. Does not modify equipment evidence.
# ============================================================

import inspect

print("VLCP VIN CANDIDATE CONTRACT")
print("=" * 60)

print("\nis_candidate_vin()")
print("-" * 60)
print(inspect.getsource(is_candidate_vin))

print("\ncollect_vins_from_equipment_index()")
print("-" * 60)
print(inspect.getsource(collect_vins_from_equipment_index))

print("=" * 60)

In [54]:
# ============================================================
# VLCP — VIN CANDIDATE QUALITY GATE v2
#
# Purpose:
# Prevent obvious placeholder / malformed FMCSA VIN values
# from consuming NHTSA decode requests.
#
# IMPORTANT:
# - Does NOT delete FMCSA evidence.
# - Does NOT change equipment deduplication.
# - Does NOT claim a VIN is valid.
# - NHTSA remains the authoritative decoder.
# ============================================================

import re


def is_candidate_vin(vin):
    """
    Determine whether a VIN-like value is suitable for
    submission to NHTSA vPIC.

    This is an input-quality gate only.
    It is NOT authoritative VIN validation.
    """

    vin = normalize_vin(vin)

    if not vin:
        return False

    # Standard VIN length
    if len(vin) != 17:
        return False

    # VIN must be alphanumeric
    if not re.fullmatch(r"[A-Z0-9]{17}", vin):
        return False

    # Standard VINs do not use I, O, or Q
    if any(char in vin for char in ("I", "O", "Q")):
        return False

    # Reject single-character placeholders:
    # 00000000000000000
    # 11111111111111111
    # AAAAAAAAAAAAAAAAA
    if len(set(vin)) == 1:
        return False

    # Reject overwhelmingly zero-filled placeholder values.
    #
    # Conservative threshold:
    # 10+ zeros in a 17-character value.
    if vin.count("0") >= 10:
        return False

    return True


print("✓ VLCP VIN candidate quality gate v2 loaded")

✓ VLCP VIN candidate quality gate v2 loaded


In [55]:
# ============================================================
# VLCP — J.B. HUNT VIN GATE REGRESSION
# Uses existing equipment evidence. No FMCSA re-download.
# ============================================================

jb_vins = collect_vins_from_equipment_index(
    vlcp_result_80806["equipment_index"]
)

print("VLCP VIN GATE REGRESSION")
print("=" * 60)

print(
    f"Qualified VIN candidates: {len(jb_vins):,}"
)

print("\nFirst 10 candidates:")
print("-" * 60)

for i, vin in enumerate(
    jb_vins[:10],
    start=1
):
    print(f"{i:>2}. {vin}")

print("=" * 60)

VLCP VIN GATE REGRESSION
Qualified VIN candidates: 7,980

First 10 candidates:
------------------------------------------------------------
 1. 109FA4022D1257509
 2. 13N148200C1554153
 3. 13N148201C1554132
 4. 13N148202C1554137
 5. 13N148202C1554140
 6. 13N148202N1550818
 7. 13N148202N1550821
 8. 13N148203L1541848
 9. 13N148203N1550813
10. 13N148204K1537418


In [56]:
# ============================================================
# VLCP — VIN CANDIDATE QUALITY GATE v3
#
# Filters obvious malformed/placeholder VIN evidence before
# NHTSA submission.
#
# FMCSA source evidence remains preserved unchanged.
# NHTSA remains the authoritative decoder.
# ============================================================

import re


def is_candidate_vin(vin):
    """
    Input-quality gate for NHTSA vPIC submission.

    Passing this function means only:
        "reasonable candidate for authoritative decoding"

    It does NOT mean:
        "verified VIN"
    """

    vin = normalize_vin(vin)

    if not vin:
        return False

    # Standard VIN length
    if len(vin) != 17:
        return False

    # Alphanumeric only
    if not re.fullmatch(r"[A-Z0-9]{17}", vin):
        return False

    # Standard VIN alphabet excludes I, O and Q
    if any(char in vin for char in ("I", "O", "Q")):
        return False

    # Reject repeated-character placeholders
    if len(set(vin)) == 1:
        return False

    # Reject suspicious long leading-zero sequences.
    #
    # Examples rejected:
    # 0000000JBHU327948
    # 00000XCSE20001352
    # 0000000000P616894
    #
    # Legitimate values beginning 10..., 11..., etc.
    # remain eligible.
    if re.match(r"^0{5,}", vin):
        return False

    # Reject overwhelmingly zero-filled values regardless
    # of where the zeros occur.
    if vin.count("0") >= 10:
        return False

    return True


print("✓ VLCP VIN candidate quality gate v3 loaded")

✓ VLCP VIN candidate quality gate v3 loaded


In [57]:
jb_vins = collect_vins_from_equipment_index(
    vlcp_result_80806["equipment_index"]
)

print("VLCP VIN GATE REGRESSION v3")
print("=" * 60)

print(
    f"Qualified VIN candidates: {len(jb_vins):,}"
)

print("\nFirst 10 candidates:")
print("-" * 60)

for i, vin in enumerate(jb_vins[:10], start=1):
    print(f"{i:>2}. {vin}")

print("=" * 60)

VLCP VIN GATE REGRESSION v3
Qualified VIN candidates: 7,980

First 10 candidates:
------------------------------------------------------------
 1. 109FA4022D1257509
 2. 13N148200C1554153
 3. 13N148201C1554132
 4. 13N148202C1554137
 5. 13N148202C1554140
 6. 13N148202N1550818
 7. 13N148202N1550821
 8. 13N148203L1541848
 9. 13N148203N1550813
10. 13N148204K1537418


In [58]:
# ============================================================
# VLCP — FINAL VIN GATE / NHTSA VALIDATION
#
# Uses existing J.B. Hunt equipment evidence.
# Does NOT re-download FMCSA data.
# ============================================================

jb_vin_decode_test = build_vin_decode_index(
    equipment_index=
        vlcp_result_80806["equipment_index"],

    delay_seconds=
        0.15,

    max_vins=
        10
)

print_vlcp_equipment_summary(
    company_name=
        vlcp_result_80806["company_name"],

    usdot=
        vlcp_result_80806["usdot"],

    equipment_index=
        vlcp_result_80806["equipment_index"],

    vin_decode_index=
        jb_vin_decode_test,

    inspection_count=
        vlcp_result_80806["inspection_count"],

    observation_count=
        vlcp_result_80806[
            "equipment_observation_count"
        ]
)

VLCP AUTHORITATIVE VIN DECODING
Candidate VINs: 10
Source: NHTSA vPIC
----------------------------------------------------------------------
[1/10] 109FA4022D1257509 -> UNRESOLVED
[2/10] 13N148200C1554153 -> DECODED
[3/10] 13N148201C1554132 -> DECODED
[4/10] 13N148202C1554137 -> DECODED
[5/10] 13N148202C1554140 -> DECODED
[6/10] 13N148202N1550818 -> DECODED
[7/10] 13N148202N1550821 -> DECODED
[8/10] 13N148203L1541848 -> DECODED
[9/10] 13N148203N1550813 -> DECODED
[10/10] 13N148204K1537418 -> DECODED
----------------------------------------------------------------------
Decode summary:
  DECODED: 9
  UNRESOLVED: 1

VLCP Verified Equipment Summary

Carrier: J B HUNT TRANSPORT INC
USDOT: 80806

Equipment | Year | Make                 | Model                | VIN               | Verification
----------+------+----------------------+----------------------+-------------------+-------------
Trailer   | 2012 | Fontaine Trailer Co. | Fontaine Trailer Co. | 13N148200C1554153 | Verified    
Trail

In [59]:
# ============================================================
# VLCP — LOCATE EQUIPMENT SUMMARY LABEL
# Read-only. No functional changes.
# ============================================================

import inspect

source = inspect.getsource(
    print_vlcp_equipment_summary
)

for line_number, line in enumerate(
    source.splitlines(),
    start=1
):
    if (
        "likely unique" in line.lower()
        or "unique equipment" in line.lower()
    ):
        print(
            f"{line_number}: {line}"
        )

175:         f"{len(equipment_index):,} likely unique equipment"


# New Section

In [60]:
# ============================================================
# VLCP — CARRIER VERIFICATION COMPONENT DISCOVERY
#
# Purpose:
# Inventory existing carrier / authority / insurance /
# verification functions before building the unified
# verify_carrier(USDOT) operational workflow.
#
# READ-ONLY — modifies nothing.
# ============================================================

import inspect

print("VLCP CARRIER VERIFICATION COMPONENT DISCOVERY")
print("=" * 70)

keywords = [
    "carrier",
    "authority",
    "insurance",
    "verify",
    "verification",
    "fmcsa",
    "docket",
    "operating",
    "liability",
    "coverage",
    "report",
]

matches = []

for name, obj in globals().items():

    if not callable(obj):
        continue

    name_lower = name.lower()

    if any(
        keyword in name_lower
        for keyword in keywords
    ):
        matches.append(name)


matches = sorted(set(matches))

for name in matches:

    obj = globals()[name]

    try:
        signature = str(
            inspect.signature(obj)
        )
    except Exception:
        signature = "(signature unavailable)"

    print(
        f"{name}{signature}"
    )

print("-" * 70)

print(
    f"Functions discovered: {len(matches)}"
)

print("=" * 70)

VLCP CARRIER VERIFICATION COMPONENT DISCOVERY
analyze_fmcsa_unit_types(normalized_rows)
apply_carrier_verification_to_shipment(vlcp_shipment, carrier_verification)
build_equipment_display_report(equipment_records)
compare_carrier_claim_to_fmcsa(vlcp_shipment, carrier_evidence)
get_carrier_usdot(record, validation_result=None)
get_display_verification(vin_evidence)
get_fmcsa_authority(usdot, timeout=30)
get_fmcsa_carrier(usdot, webkey=None)
get_fmcsa_inspection_units(usdot)
get_fmcsa_insurance(usdot, timeout=30)
get_fmcsa_insurance_lifecycle(usdot, timeout=30)
normalize_fmcsa_authority_result(authority_result)
normalize_fmcsa_carrier_result(fmcsa_result)
normalize_fmcsa_dot_for_actpend(usdot)
normalize_fmcsa_inspection_units(inspection_result)
normalize_fmcsa_insurance_lifecycle(lifecycle_result)
normalize_fmcsa_insurance_result(insurance_result)
print_equipment_display_report(equipment_records)
print_vlcp_carrier_verification(result)
print_vlcp_shipment_report(vlcp_shipment)
profile_fm

In [61]:
# ============================================================
# VLCP — CARRIER IDENTITY CONTRACT INSPECTION
#
# Purpose:
# Inspect the existing carrier identity resolver before
# connecting Authority + Insurance to the unified workflow.
#
# READ-ONLY — modifies nothing.
# ============================================================

import inspect

print("VLCP CARRIER IDENTITY CONTRACT")
print("=" * 70)

# ------------------------------------------------------------
# 1. resolve_carrier_identity
# ------------------------------------------------------------

print("\nresolve_carrier_identity()")
print("-" * 70)

print(
    "Signature:",
    inspect.signature(
        resolve_carrier_identity
    )
)

print("\nSource:")
print(
    inspect.getsource(
        resolve_carrier_identity
    )
)

# ------------------------------------------------------------
# 2. normalize_fmcsa_carrier_result
# ------------------------------------------------------------

print("\n" + "=" * 70)

print("\nnormalize_fmcsa_carrier_result()")
print("-" * 70)

print(
    "Signature:",
    inspect.signature(
        normalize_fmcsa_carrier_result
    )
)

print("\nSource:")
print(
    inspect.getsource(
        normalize_fmcsa_carrier_result
    )
)

print("\n" + "=" * 70)

VLCP CARRIER IDENTITY CONTRACT

resolve_carrier_identity()
----------------------------------------------------------------------
Signature: (usdot)

Source:
def resolve_carrier_identity(usdot):
    """
    Retrieve and normalize FMCSA carrier identity.

    Source failures remain distinguishable from NOT_FOUND.
    """

    usdot = str(usdot).strip()

    identity = {
        "usdot": usdot,
        "legal_name": None,
        "dba_name": None,
        "display_name": None,
        "identity_status": "UNKNOWN",
        "identity_source": "FMCSA QCMobile",
        "error": None,
    }

    try:

        result = get_fmcsa_carrier(
            usdot,
            webkey=FMCSA_WEBKEY
        )

        identity["identity_status"] = (
            result.get(
                "lookup_status",
                "UNKNOWN"
            )
        )

        if not result.get("success"):

            identity["error"] = (
                result.get("error")
            )

            identity["displ

In [62]:
# ============================================================
# VLCP — FMCSA AUTHORITY / INSURANCE DATASET DISCOVERY
#
# Purpose:
# Search the USDOT public-data catalog for authoritative
# FMCSA datasets related to:
#
#   1. Operating Authority
#   2. Insurance
#
# We discover datasets BEFORE writing retrieval logic.
#
# No VLCP production functions are modified.
# ============================================================

import requests


CATALOG_URL = (
    "https://api.us.socrata.com/api/catalog/v1"
)


def search_dot_catalog(search_term, limit=20):
    """
    Search the U.S. DOT Socrata catalog and return
    compact dataset metadata.
    """

    params = {
        "search_context": "data.transportation.gov",
        "q": search_term,
        "limit": limit
    }

    response = requests.get(
        CATALOG_URL,
        params=params,
        timeout=30
    )

    response.raise_for_status()

    payload = response.json()

    results = []

    for item in payload.get("results", []):

        resource = item.get("resource", {})
        metadata = item.get("metadata", {})

        results.append({
            "name": resource.get("name"),
            "dataset_id": resource.get("id"),
            "type": resource.get("type"),
            "description": resource.get(
                "description"
            ),
            "domain": metadata.get("domain"),
            "permalink": item.get("permalink")
        })

    return results


def print_catalog_results(title, results):

    print("\n" + "=" * 78)
    print(title)
    print("=" * 78)

    if not results:
        print("No datasets found.")
        return

    for i, item in enumerate(
        results,
        start=1
    ):

        print(
            f"\n[{i}] {item['name']}"
        )

        print(
            f"    Dataset ID: "
            f"{item['dataset_id']}"
        )

        print(
            f"    Domain: "
            f"{item['domain']}"
        )

        description = (
            item.get("description")
            or ""
        )

        description = " ".join(
            description.split()
        )

        if len(description) > 300:
            description = (
                description[:300] + "..."
            )

        if description:
            print(
                f"    Description: "
                f"{description}"
            )


# ------------------------------------------------------------
# AUTHORITY SEARCH
# ------------------------------------------------------------

authority_results = search_dot_catalog(
    "FMCSA operating authority"
)

print_catalog_results(
    "FMCSA OPERATING AUTHORITY DATASET CANDIDATES",
    authority_results
)


# ------------------------------------------------------------
# INSURANCE SEARCH
# ------------------------------------------------------------

insurance_results = search_dot_catalog(
    "FMCSA insurance"
)

print_catalog_results(
    "FMCSA INSURANCE DATASET CANDIDATES",
    insurance_results
)


print("\n" + "=" * 78)
print(
    "✓ Authority / insurance dataset discovery complete"
)
print("=" * 78)


FMCSA OPERATING AUTHORITY DATASET CANDIDATES

[1] Revocation - All With History
    Dataset ID: sa6p-acbp
    Domain: data.transportation.gov
    Description: *Dataset* Information on carrier/broker/freight forwarder authorities that have been revoked by FMCSA. The dataset includes the DOT number and docket number of the entity, the type of authority revoked, and the reason. Note: This dataset was last refreshed on 05/14/2026 and will no longer be updated...

[2] AuthHist - All With History
    Dataset ID: 9mw4-x3tu
    Domain: data.transportation.gov
    Description: *Dataset* Records showing the history of each authority granted to a carrier/broker/freight forwarder, along with the dates of the original authority action (e.g., “granted”) and the final authority action (e.g., “revoked”). The dataset contains the DOT number and docket number of the entity that ho...

[3] BOC3 - All With History
    Dataset ID: 2emp-mxtb
    Domain: data.transportation.gov
    Description: *Dataset* Re

In [ ]:
# ============================================================
# VLCP — FMCSA AUTHORITY / INSURANCE SCHEMA INSPECTION
#
# Purpose:
# Inspect actual Socrata columns before selecting production
# datasets or writing retrieval functions.
#
# READ-ONLY — modifies no VLCP functions.
# ============================================================

import requests


SOCRATA_DOMAIN = "https://data.transportation.gov"

DATASETS = {
    # Authority
    "Motus Carrier - All With History": "inys-ebih",
    "AuthHist - All With History": "9mw4-x3tu",

    # Insurance
    "Motus Insur - All With History": "c5y8-a4uz",
    "ActPendInsur - All With History": "qh9u-swkp",
}


def inspect_socrata_schema(name, dataset_id):

    metadata_url = (
        f"{SOCRATA_DOMAIN}/api/views/{dataset_id}"
    )

    response = requests.get(
        metadata_url,
        timeout=30
    )

    response.raise_for_status()

    metadata = response.json()

    print("\n" + "=" * 78)
    print(name)
    print(f"Dataset ID: {dataset_id}")
    print("=" * 78)

    print(
        "Updated:",
        metadata.get("rowsUpdatedAt")
    )

    print(
        "Rows:",
        metadata.get("rowsUpdatedBy")
        or metadata.get("viewLastModified")
        or "metadata available"
    )

    print("\nCOLUMNS")
    print("-" * 78)

    columns = metadata.get("columns", [])

    for column in columns:

        print(
            f"{column.get('name', ''):<35} "
            f"| field: "
            f"{column.get('fieldName', ''):<30} "
            f"| type: "
            f"{column.get('dataTypeName', '')}"
        )

    print("\nUSDOT / DOCKET / STATUS-RELATED FIELDS")
    print("-" * 78)

    important = []

    terms = [
        "dot",
        "docket",
        "authority",
        "status",
        "active",
        "insurance",
        "policy",
        "coverage",
        "limit",
        "form",
        "effective",
        "cancel",
        "carrier"
    ]

    for column in columns:

        combined = (
            str(column.get("name", ""))
            + " "
            + str(column.get("fieldName", ""))
        ).lower()

        if any(term in combined for term in terms):

            important.append(
                (
                    column.get("name"),
                    column.get("fieldName"),
                    column.get("dataTypeName")
                )
            )

    for display_name, field_name, data_type in important:

        print(
            f"{display_name:<35} "
            f"| {field_name:<30} "
            f"| {data_type}"
        )


for dataset_name, dataset_id in DATASETS.items():

    try:

        inspect_socrata_schema(
            dataset_name,
            dataset_id
        )

    except Exception as exc:

        print("\n" + "=" * 78)
        print(dataset_name)
        print("=" * 78)

        print(
            "SCHEMA ERROR:",
            type(exc).__name__,
            str(exc)
        )


print("\n" + "=" * 78)
print("✓ Authority / insurance schema inspection complete")
print("=" * 78)

In [ ]:
# ============================================================
# VLCP — FMCSA OPERATING AUTHORITY MODULE v1
#
# Primary source:
#   FMCSA Motus Carrier - All With History
#   Dataset: inys-ebih
#
# Input:
#   Any USDOT number
#
# Output:
#   Structured VLCP authority evidence
#
# IMPORTANT:
# - Dynamic USDOT
# - No carrier-specific production logic
# - Preserves all authority records
# - Does not convert source failure into NOT_FOUND
# ============================================================

import requests
from datetime import datetime, UTC


FMCSA_MOTUS_CARRIER_DATASET = "inys-ebih"

FMCSA_MOTUS_CARRIER_URL = (
    "https://data.transportation.gov/"
    f"resource/{FMCSA_MOTUS_CARRIER_DATASET}.json"
)


def get_fmcsa_authority(usdot, timeout=30):
    """
    Retrieve FMCSA operating-authority evidence for a USDOT.

    Source:
        FMCSA Motus Carrier - All With History

    Returns source evidence only.
    It does NOT make a VLCP policy/qualification decision.
    """

    usdot = str(usdot).strip()

    result = {
        "usdot": usdot,
        "success": False,
        "lookup_status": "UNKNOWN",
        "source": {
            "name": "FMCSA Motus Carrier - All With History",
            "dataset_id": FMCSA_MOTUS_CARRIER_DATASET,
            "request_url": FMCSA_MOTUS_CARRIER_URL
        },
        "record_count": 0,
        "records": [],
        "error": None,
        "retrieved_at_utc": None
    }

    # --------------------------------------------------------
    # INPUT VALIDATION
    # --------------------------------------------------------

    if not usdot:

        result["lookup_status"] = "INVALID_INPUT"
        result["error"] = "USDOT is required."

        return result


    if not usdot.isdigit():

        result["lookup_status"] = "INVALID_INPUT"
        result["error"] = (
            "USDOT must contain numbers only."
        )

        return result


    # --------------------------------------------------------
    # SOCRATA QUERY
    # --------------------------------------------------------

    params = {
        "$where": f"usdot_number='{usdot}'",
        "$limit": 5000
    }

    try:

        response = requests.get(
            FMCSA_MOTUS_CARRIER_URL,
            params=params,
            timeout=timeout
        )

        result["source"]["http_status"] = (
            response.status_code
        )

        result["source"]["resolved_url"] = (
            response.url
        )

        response.raise_for_status()

        rows = response.json()

        if not isinstance(rows, list):

            raise TypeError(
                "FMCSA authority response "
                "was not a list."
            )

        result["success"] = True

        result["record_count"] = len(rows)

        result["records"] = rows

        result["retrieved_at_utc"] = (
            datetime.now(UTC).isoformat()
        )

        if rows:

            result["lookup_status"] = "FOUND"

        else:

            result["lookup_status"] = "NOT_FOUND"

        return result


    except Exception as exc:

        result["lookup_status"] = "SOURCE_ERROR"

        result["error"] = (
            f"{type(exc).__name__}: {exc}"
        )

        result["retrieved_at_utc"] = (
            datetime.now(UTC).isoformat()
        )

        return result


# ============================================================
# NORMALIZATION
# ============================================================

def normalize_fmcsa_authority_result(authority_result):
    """
    Convert raw Motus Carrier records into stable VLCP
    authority evidence.

    Every source record is preserved.
    """

    evidence = {
        "evidence_type": "carrier_authority_fmcsa",
        "evidence_status": "UNAVAILABLE",

        "usdot": authority_result.get("usdot"),

        "source": authority_result.get("source"),

        "lookup_status": authority_result.get(
            "lookup_status"
        ),

        "authority_records": [],

        "retrieved_at_utc": authority_result.get(
            "retrieved_at_utc"
        ),

        "error": authority_result.get("error"),

        "raw_records": authority_result.get(
            "records",
            []
        )
    }


    if not authority_result.get("success"):

        return evidence


    rows = authority_result.get(
        "records",
        []
    )


    if not rows:

        evidence["evidence_status"] = "NOT_FOUND"

        return evidence


    normalized_records = []


    for row in rows:

        normalized_records.append({

            "usdot": str(
                row.get("usdot_number")
                or ""
            ).strip(),

            "docket_number": row.get(
                "docket_number"
            ),

            "authority_type": row.get(
                "op_auth_type"
            ),

            "authority_status": row.get(
                "op_auth_status"
            ),

            "minimum_coverage_amount": row.get(
                "min_cov_amount"
            ),

            "cargo_required": row.get(
                "cargo_req"
            ),

            "bond_required": row.get(
                "bond_req"
            ),

            "bipd_on_file": row.get(
                "bipd_file"
            ),

            "cargo_on_file": row.get(
                "cargo_file"
            ),

            "bond_on_file": row.get(
                "bond_file"
            ),

            "legal_name": row.get(
                "legal_name"
            ),

            "dba_name": row.get(
                "dba_name"
            )
        })


    evidence["authority_records"] = (
        normalized_records
    )

    evidence["evidence_status"] = "RETRIEVED"

    return evidence


print(
    "✓ VLCP FMCSA authority module v1 loaded"
)

In [ ]:
# ============================================================
# VLCP — AUTHORITY MODULE REGRESSION
#
# Tests the same authority module against two different
# carriers without changing production logic.
# ============================================================

for test_usdot in ["951224", "80806"]:

    raw = get_fmcsa_authority(
        test_usdot
    )

    evidence = (
        normalize_fmcsa_authority_result(
            raw
        )
    )

    print("\n" + "=" * 70)
    print(f"USDOT: {test_usdot}")

    print(
        "Lookup:",
        evidence["lookup_status"]
    )

    print(
        "Evidence:",
        evidence["evidence_status"]
    )

    print(
        "Authority records:",
        len(
            evidence["authority_records"]
        )
    )

    print("-" * 70)

    for record in evidence[
        "authority_records"
    ]:

        print(
            "Docket:",
            record["docket_number"],
            "| Type:",
            record["authority_type"],
            "| Status:",
            record["authority_status"],
            "| Legal Name:",
            record["legal_name"]
        )

print("\n" + "=" * 70)
print("✓ Authority regression complete")
print("=" * 70)

In [ ]:
# ============================================================
# VLCP — FMCSA INSURANCE MODULE v1
#
# Primary source:
#   FMCSA Motus Insur - All With History
#   Dataset: c5y8-a4uz
#
# Input:
#   Any USDOT number
#
# Output:
#   Structured VLCP insurance evidence
#
# IMPORTANT:
# - Dynamic USDOT
# - No carrier-specific production logic
# - Preserves all returned insurance records
# - Source failure != NOT_FOUND
# - No policy qualification decision is made here
# ============================================================

import requests
from datetime import datetime, UTC


FMCSA_MOTUS_INSURANCE_DATASET = "c5y8-a4uz"

FMCSA_MOTUS_INSURANCE_URL = (
    "https://data.transportation.gov/"
    f"resource/{FMCSA_MOTUS_INSURANCE_DATASET}.json"
)


def get_fmcsa_insurance(usdot, timeout=30):
    """
    Retrieve FMCSA insurance evidence for a USDOT.

    Source:
        FMCSA Motus Insur - All With History

    This function retrieves evidence only.
    It does NOT decide whether coverage satisfies
    a broker, shipper, or VLCP policy.
    """

    usdot = str(usdot).strip()

    result = {
        "usdot": usdot,
        "success": False,
        "lookup_status": "UNKNOWN",

        "source": {
            "name": "FMCSA Motus Insur - All With History",
            "dataset_id": FMCSA_MOTUS_INSURANCE_DATASET,
            "request_url": FMCSA_MOTUS_INSURANCE_URL
        },

        "record_count": 0,
        "records": [],
        "error": None,
        "retrieved_at_utc": None
    }

    # --------------------------------------------------------
    # INPUT VALIDATION
    # --------------------------------------------------------

    if not usdot:

        result["lookup_status"] = "INVALID_INPUT"
        result["error"] = "USDOT is required."

        return result


    if not usdot.isdigit():

        result["lookup_status"] = "INVALID_INPUT"

        result["error"] = (
            "USDOT must contain numbers only."
        )

        return result


    # --------------------------------------------------------
    # SOCRATA QUERY
    # --------------------------------------------------------

    params = {
        "$where": f"usdot_number='{usdot}'",
        "$limit": 5000
    }


    try:

        response = requests.get(
            FMCSA_MOTUS_INSURANCE_URL,
            params=params,
            timeout=timeout
        )

        result["source"]["http_status"] = (
            response.status_code
        )

        result["source"]["resolved_url"] = (
            response.url
        )

        response.raise_for_status()

        rows = response.json()

        if not isinstance(rows, list):

            raise TypeError(
                "FMCSA insurance response "
                "was not a list."
            )


        result["success"] = True

        result["record_count"] = len(rows)

        result["records"] = rows

        result["retrieved_at_utc"] = (
            datetime.now(UTC).isoformat()
        )


        if rows:

            result["lookup_status"] = "FOUND"

        else:

            result["lookup_status"] = "NOT_FOUND"


        return result


    except Exception as exc:

        result["lookup_status"] = "SOURCE_ERROR"

        result["error"] = (
            f"{type(exc).__name__}: {exc}"
        )

        result["retrieved_at_utc"] = (
            datetime.now(UTC).isoformat()
        )

        return result


# ============================================================
# NORMALIZATION
# ============================================================

def normalize_fmcsa_insurance_result(
    insurance_result
):
    """
    Normalize FMCSA Motus insurance records into stable
    VLCP evidence.

    All source records remain preserved.
    """

    evidence = {
        "evidence_type":
            "carrier_insurance_fmcsa",

        "evidence_status":
            "UNAVAILABLE",

        "usdot":
            insurance_result.get("usdot"),

        "source":
            insurance_result.get("source"),

        "lookup_status":
            insurance_result.get(
                "lookup_status"
            ),

        "insurance_records": [],

        "retrieved_at_utc":
            insurance_result.get(
                "retrieved_at_utc"
            ),

        "error":
            insurance_result.get("error"),

        "raw_records":
            insurance_result.get(
                "records",
                []
            )
    }


    if not insurance_result.get("success"):

        return evidence


    rows = insurance_result.get(
        "records",
        []
    )


    if not rows:

        evidence["evidence_status"] = (
            "NOT_FOUND"
        )

        return evidence


    normalized_records = []


    for row in rows:

        normalized_records.append({

            "usdot": str(
                row.get("usdot_number")
                or ""
            ).strip(),

            "docket_number":
                row.get("docket_number"),

            "form_code":
                row.get("ins_form_code"),

            "insurance_type":
                row.get("ins_type_code"),

            "insurance_class":
                row.get("ins_class_code"),

            "maximum_coverage_amount":
                row.get("max_cov_amount"),

            "underlying_limit_amount":
                row.get("underl_lim_amount"),

            "policy_number":
                row.get("policy_no"),

            "effective_date":
                row.get("effective_date"),

            "insurance_company":
                row.get(
                    "insurance_company_name"
                ),

            "transaction_date":
                row.get("trans_date")
        })


    evidence["insurance_records"] = (
        normalized_records
    )

    evidence["evidence_status"] = (
        "RETRIEVED"
    )

    return evidence


print(
    "✓ VLCP FMCSA insurance module v1 loaded"
)

In [ ]:
# ============================================================
# VLCP — INSURANCE MODULE MULTI-USDOT REGRESSION
#
# Tests:
#   951224 — US SERVICES LLC
#   80806  — J B HUNT TRANSPORT INC
#
# Evidence only — no policy qualification decisions.
# ============================================================

for test_usdot in ["951224", "80806"]:

    raw = get_fmcsa_insurance(
        test_usdot
    )

    evidence = (
        normalize_fmcsa_insurance_result(
            raw
        )
    )

    print("\n" + "=" * 78)
    print(f"USDOT: {test_usdot}")

    print(
        "Lookup:",
        evidence["lookup_status"]
    )

    print(
        "Evidence:",
        evidence["evidence_status"]
    )

    print(
        "Insurance records:",
        len(
            evidence["insurance_records"]
        )
    )

    print("-" * 78)

    for record in evidence[
        "insurance_records"
    ]:

        print(
            "Docket:",
            record["docket_number"],
            "| Form:",
            record["form_code"],
            "| Type:",
            record["insurance_type"],
            "| Class:",
            record["insurance_class"]
        )

        print(
            "  Company:",
            record["insurance_company"]
        )

        print(
            "  Policy:",
            record["policy_number"],
            "| Effective:",
            record["effective_date"]
        )

        print(
            "  Underlying:",
            record["underlying_limit_amount"],
            "| Maximum:",
            record["maximum_coverage_amount"]
        )

print("\n" + "=" * 78)
print("✓ Insurance regression complete")
print("=" * 78)

In [ ]:
# ============================================================
# VLCP — INSURANCE DUPLICATE DIAGNOSTIC
#
# Purpose:
# Determine whether apparent duplicate insurance records
# are truly identical source rows or differ in fields that
# our current normalized view does not display.
#
# READ-ONLY.
# ============================================================

import json
from collections import Counter


def inspect_insurance_duplicates(usdot):

    raw = get_fmcsa_insurance(usdot)

    rows = raw.get("records", [])

    print("\n" + "=" * 78)
    print(f"VLCP INSURANCE DUPLICATE DIAGNOSTIC — USDOT {usdot}")
    print("=" * 78)

    print(
        f"Raw source rows: {len(rows)}"
    )

    # --------------------------------------------------------
    # FULL-ROW SIGNATURE
    # --------------------------------------------------------

    signatures = []

    for row in rows:

        signature = json.dumps(
            row,
            sort_keys=True,
            default=str
        )

        signatures.append(signature)


    counts = Counter(signatures)

    print(
        f"Unique full source rows: {len(counts)}"
    )

    print(
        f"Exact duplicate rows: "
        f"{len(rows) - len(counts)}"
    )

    # --------------------------------------------------------
    # DISPLAY DUPLICATE GROUPS
    # --------------------------------------------------------

    duplicate_number = 0

    for signature, count in counts.items():

        if count <= 1:
            continue

        duplicate_number += 1

        row = json.loads(signature)

        print("\n" + "-" * 78)

        print(
            f"Duplicate group {duplicate_number} "
            f"— repeated {count} times"
        )

        print("-" * 78)

        for key, value in row.items():

            print(
                f"{key}: {value}"
            )

    if duplicate_number == 0:

        print(
            "\nNo exact full-row duplicates detected."
        )

        print(
            "The apparent duplicates differ in "
            "one or more source fields."
        )


# ------------------------------------------------------------
# MULTI-USDOT DIAGNOSTIC
# ------------------------------------------------------------

inspect_insurance_duplicates("951224")
inspect_insurance_duplicates("80806")

print("\n" + "=" * 78)
print("✓ Insurance duplicate diagnostic complete")
print("=" * 78)

In [66]:
# ============================================================
# VLCP — FMCSA INSURANCE NORMALIZER v2
#
# Improvement:
# - Preserves ALL raw FMCSA source rows
# - Removes exact duplicate rows from operational evidence
# - Records raw / unique / duplicate counts
# - Makes no insurance-policy qualification decision
#
# Replaces:
#   normalize_fmcsa_insurance_result()
# ============================================================

import json


def normalize_fmcsa_insurance_result(
    insurance_result
):
    """
    Normalize FMCSA Motus insurance records into stable
    VLCP operational evidence.

    Exact duplicate source rows are removed from the
    operational insurance_records list.

    ALL original source rows remain preserved in raw_records.
    """

    raw_rows = insurance_result.get(
        "records",
        []
    )

    evidence = {

        "evidence_type":
            "carrier_insurance_fmcsa",

        "evidence_status":
            "UNAVAILABLE",

        "usdot":
            insurance_result.get("usdot"),

        "source":
            insurance_result.get("source"),

        "lookup_status":
            insurance_result.get(
                "lookup_status"
            ),

        "insurance_records": [],

        # --------------------------------------------
        # Evidence accounting
        # --------------------------------------------

        "raw_record_count":
            len(raw_rows),

        "unique_record_count":
            0,

        "duplicate_rows_removed":
            0,

        "retrieved_at_utc":
            insurance_result.get(
                "retrieved_at_utc"
            ),

        "error":
            insurance_result.get("error"),

        # Preserve original FMCSA evidence unchanged
        "raw_records":
            raw_rows
    }


    # --------------------------------------------------------
    # SOURCE FAILURE
    # --------------------------------------------------------

    if not insurance_result.get("success"):

        return evidence


    # --------------------------------------------------------
    # VALID EMPTY RESPONSE
    # --------------------------------------------------------

    if not raw_rows:

        evidence["evidence_status"] = (
            "NOT_FOUND"
        )

        return evidence


    # --------------------------------------------------------
    # EXACT FULL-ROW DEDUPLICATION
    # --------------------------------------------------------

    unique_rows = []

    seen = set()


    for row in raw_rows:

        signature = json.dumps(
            row,
            sort_keys=True,
            default=str
        )


        if signature in seen:

            continue


        seen.add(signature)

        unique_rows.append(row)


    evidence["unique_record_count"] = (
        len(unique_rows)
    )

    evidence["duplicate_rows_removed"] = (
        len(raw_rows) - len(unique_rows)
    )


    # --------------------------------------------------------
    # NORMALIZE UNIQUE SOURCE RECORDS
    # --------------------------------------------------------

    normalized_records = []


    for row in unique_rows:

        normalized_records.append({

            "usdot": str(
                row.get("usdot_number")
                or ""
            ).strip(),

            "docket_number":
                row.get("docket_number"),

            "form_code":
                row.get("ins_form_code"),

            "insurance_type":
                row.get("ins_type_code"),

            "insurance_class":
                row.get("ins_class_code"),

            "maximum_coverage_amount":
                row.get("max_cov_amount"),

            "underlying_limit_amount":
                row.get("underl_lim_amount"),

            "policy_number":
                row.get("policy_no"),

            "effective_date":
                row.get("effective_date"),

            "insurance_company":
                row.get(
                    "insurance_company_name"
                ),

            "transaction_date":
                row.get("trans_date")
        })


    evidence["insurance_records"] = (
        normalized_records
    )

    evidence["evidence_status"] = (
        "RETRIEVED"
    )


    return evidence


print(
    "✓ VLCP FMCSA insurance normalizer v2 loaded"
)

✓ VLCP FMCSA insurance normalizer v2 loaded


In [65]:
# ============================================================
# VLCP — INSURANCE NORMALIZER v2 INTEGRITY CHECK
# ============================================================

for test_usdot in ["951224", "80806"]:

    raw = get_fmcsa_insurance(test_usdot)

    evidence = normalize_fmcsa_insurance_result(raw)

    print("\n" + "=" * 70)
    print(f"USDOT: {test_usdot}")
    print("Status:", evidence["evidence_status"])
    print("Raw rows:", evidence["raw_record_count"])
    print("Unique filings:", evidence["unique_record_count"])
    print(
        "Exact duplicates removed:",
        evidence["duplicate_rows_removed"]
    )

    print("-" * 70)

    for record in evidence["insurance_records"]:

        print(
            record["form_code"],
            "|",
            record["insurance_company"],
            "|",
            record["policy_number"],
            "|",
            record["maximum_coverage_amount"],
            "|",
            record["effective_date"]
        )


print("\n" + "=" * 70)
print("✓ Insurance v2 integrity check complete")
print("=" * 70)


USDOT: 951224
Status: RETRIEVED
Raw rows: 4
Unique filings: 2
Exact duplicates removed: 2
----------------------------------------------------------------------
BMC-91X | Northland Insurance Company | WF003597 | 1000000.00 | 20250501
BMC-84 | American Alternative Insurance Corporation | MC407574 | 75000.00 | 20131001

USDOT: 80806
Status: RETRIEVED
Raw rows: 16
Unique filings: 5
Exact duplicates removed: 11
----------------------------------------------------------------------
BMC-91X | Ace American Insurance Company | XSAH11347472 | 2500000.00 | 20251231
BMC-91X | Self-Insured | 0 | 1000000.00 | 19930301
BMC-83 | Travelers Casualty and Surety Company of America | 105917743 | 5000.00 | 20231023
BMC-84 | Travelers Casualty and Surety Company of America | 105917743 | 75000.00 | 20140113
BMC-91X | Palisades Insurance Company | NONE | 1000000.00 | 19930301

✓ Insurance v2 integrity check complete


In [64]:
# ============================================================
# VLCP — ACTIVE / PENDING INSURANCE EVIDENCE INSPECTION
#
# Source:
#   FMCSA ActPendInsur - All With History
#   Dataset: qh9u-swkp
#
# Purpose:
# Inspect effective + cancellation dates before VLCP creates
# any "current insurance" logic.
#
# READ-ONLY.
# ============================================================

import requests
import json
from collections import Counter


FMCSA_ACTPEND_INSURANCE_DATASET = "qh9u-swkp"

FMCSA_ACTPEND_INSURANCE_URL = (
    "https://data.transportation.gov/"
    f"resource/{FMCSA_ACTPEND_INSURANCE_DATASET}.json"
)


def inspect_active_pending_insurance(usdot):

    usdot = str(usdot).strip()

    params = {
        "$where": f"dot_number='{usdot}'",
        "$limit": 5000
    }

    print("\n" + "=" * 78)
    print(
        f"FMCSA ACTIVE/PENDING INSURANCE — USDOT {usdot}"
    )
    print("=" * 78)

    try:

        response = requests.get(
            FMCSA_ACTPEND_INSURANCE_URL,
            params=params,
            timeout=30
        )

        print(
            "HTTP:",
            response.status_code
        )

        response.raise_for_status()

        rows = response.json()

        print(
            "Raw rows:",
            len(rows)
        )

        # ----------------------------------------------------
        # Count exact duplicates
        # ----------------------------------------------------

        signatures = [
            json.dumps(
                row,
                sort_keys=True,
                default=str
            )
            for row in rows
        ]

        counts = Counter(signatures)

        print(
            "Unique full rows:",
            len(counts)
        )

        print(
            "Exact duplicates:",
            len(rows) - len(counts)
        )

        # ----------------------------------------------------
        # Print unique evidence rows
        # ----------------------------------------------------

        print("\nUNIQUE SOURCE RECORDS")
        print("-" * 78)

        for signature in counts:

            row = json.loads(signature)

            print(
                "\nDocket:",
                row.get("docket_number")
            )

            print(
                "Form:",
                row.get("ins_form_code")
            )

            print(
                "Type:",
                row.get("mod_col_1")
            )

            print(
                "Company:",
                row.get("name_company")
            )

            print(
                "Policy:",
                row.get("policy_no")
            )

            print(
                "Effective:",
                row.get("effective_date")
            )

            print(
                "Cancel Effective:",
                row.get("cancl_effective_date")
            )

            print(
                "Underlying:",
                row.get("underl_lim_amount")
            )

            print(
                "Maximum:",
                row.get("max_cov_amount")
            )

            print(
                "Transaction:",
                row.get("trans_date")
            )

    except Exception as exc:

        print(
            "SOURCE ERROR:",
            type(exc).__name__,
            str(exc)
        )


# ============================================================
# MULTI-USDOT INSPECTION
# ============================================================

inspect_active_pending_insurance("951224")
inspect_active_pending_insurance("80806")

print("\n" + "=" * 78)
print(
    "✓ Active/pending insurance evidence inspection complete"
)
print("=" * 78)


FMCSA ACTIVE/PENDING INSURANCE — USDOT 951224
HTTP: 200
Raw rows: 0
Unique full rows: 0
Exact duplicates: 0

UNIQUE SOURCE RECORDS
------------------------------------------------------------------------------

FMCSA ACTIVE/PENDING INSURANCE — USDOT 80806
HTTP: 200
Raw rows: 0
Unique full rows: 0
Exact duplicates: 0

UNIQUE SOURCE RECORDS
------------------------------------------------------------------------------

✓ Active/pending insurance evidence inspection complete


In [63]:
# ============================================================
# VLCP — ActPendInsur LOOKUP DIAGNOSTIC
#
# Purpose:
# Determine why valid carriers returned zero rows when queried
# by DOT_NUMBER.
#
# Tests:
#   1. USDOT lookup
#   2. Known docket lookup
#
# READ-ONLY — no VLCP production functions changed.
# ============================================================

import requests


ACTPEND_URL = (
    "https://data.transportation.gov/"
    "resource/qh9u-swkp.json"
)


TEST_CASES = [
    {
        "usdot": "951224",
        "docket": "MC407574",
        "name": "US SERVICES LLC"
    },
    {
        "usdot": "80806",
        "docket": "MC135797",
        "name": "J B HUNT TRANSPORT INC"
    }
]


def query_actpend(where_clause):

    response = requests.get(
        ACTPEND_URL,
        params={
            "$where": where_clause,
            "$limit": 100
        },
        timeout=30
    )

    response.raise_for_status()

    return response.json(), response.url


for carrier in TEST_CASES:

    print("\n" + "=" * 78)
    print(
        carrier["name"],
        "| USDOT:",
        carrier["usdot"],
        "| Docket:",
        carrier["docket"]
    )
    print("=" * 78)

    # --------------------------------------------------------
    # TEST 1 — USDOT
    # --------------------------------------------------------

    dot_rows, dot_url = query_actpend(
        f"dot_number='{carrier['usdot']}'"
    )

    print("\nUSDOT QUERY")
    print("-" * 78)

    print(
        "Rows:",
        len(dot_rows)
    )

    print(
        "Resolved URL:",
        dot_url
    )

    # --------------------------------------------------------
    # TEST 2 — DOCKET
    # --------------------------------------------------------

    docket_rows, docket_url = query_actpend(
        f"docket_number='{carrier['docket']}'"
    )

    print("\nDOCKET QUERY")
    print("-" * 78)

    print(
        "Rows:",
        len(docket_rows)
    )

    print(
        "Resolved URL:",
        docket_url
    )

    # --------------------------------------------------------
    # PRINT RECORDS IF FOUND
    # --------------------------------------------------------

    if docket_rows:

        print("\nDOCKET RECORDS")
        print("-" * 78)

        for row in docket_rows:

            print(
                "DOT:",
                row.get("dot_number"),
                "| Docket:",
                row.get("docket_number"),
                "| Form:",
                row.get("ins_form_code")
            )

            print(
                "  Company:",
                row.get("name_company")
            )

            print(
                "  Effective:",
                row.get("effective_date"),
                "| Cancel:",
                row.get("cancl_effective_date")
            )


print("\n" + "=" * 78)
print("✓ ActPendInsur lookup diagnostic complete")
print("=" * 78)


US SERVICES LLC | USDOT: 951224 | Docket: MC407574

USDOT QUERY
------------------------------------------------------------------------------
Rows: 0
Resolved URL: https://data.transportation.gov/resource/qh9u-swkp.json?%24where=dot_number%3D%27951224%27&%24limit=100

DOCKET QUERY
------------------------------------------------------------------------------
Rows: 2
Resolved URL: https://data.transportation.gov/resource/qh9u-swkp.json?%24where=docket_number%3D%27MC407574%27&%24limit=100

DOCKET RECORDS
------------------------------------------------------------------------------
DOT: 00951224 | Docket: MC407574 | Form: 91X
  Company: NORTHLAND INSURANCE COMPANY
  Effective: 05/01/2025 | Cancel: None
DOT: 00951224 | Docket: MC407574 | Form: 84
  Company: AMERICAN ALTERNATIVE INSURANCE CORPORATION
  Effective: 10/01/2013 | Cancel: None

J B HUNT TRANSPORT INC | USDOT: 80806 | Docket: MC135797

USDOT QUERY
------------------------------------------------------------------------------
R

In [28]:
# ============================================================
# VLCP — FMCSA INSURANCE LIFECYCLE MODULE v1
#
# Source:
#   FMCSA ActPendInsur - All With History
#   Dataset: qh9u-swkp
#
# Important source behavior discovered:
#   DOT_NUMBER is stored as an 8-character zero-padded value.
#
# Examples:
#   951224 -> 00951224
#   80806  -> 00080806
#
# Purpose:
#   Retrieve active/pending insurance filing evidence and
#   preserve effective/cancellation lifecycle information.
#
# This is EVIDENCE, not a broker qualification decision.
# ============================================================

import requests
import json
from datetime import datetime, UTC


FMCSA_ACTPEND_INSURANCE_DATASET = "qh9u-swkp"

FMCSA_ACTPEND_INSURANCE_URL = (
    "https://data.transportation.gov/"
    f"resource/{FMCSA_ACTPEND_INSURANCE_DATASET}.json"
)


def normalize_fmcsa_dot_for_actpend(usdot):
    """
    Normalize USDOT to the 8-character representation used
    by the ActPendInsur dataset.

    Example:
        951224 -> 00951224
        80806  -> 00080806
    """

    usdot = str(usdot).strip()

    if not usdot:
        return None

    if not usdot.isdigit():
        return None

    return usdot.zfill(8)


def get_fmcsa_insurance_lifecycle(
    usdot,
    timeout=30
):
    """
    Retrieve FMCSA active/pending insurance filing evidence
    using the dataset's zero-padded USDOT representation.
    """

    original_usdot = str(usdot).strip()

    source_usdot = (
        normalize_fmcsa_dot_for_actpend(
            original_usdot
        )
    )

    result = {

        "usdot": original_usdot,

        "source_usdot": source_usdot,

        "success": False,

        "lookup_status": "UNKNOWN",

        "source": {
            "name":
                "FMCSA ActPendInsur - All With History",

            "dataset_id":
                FMCSA_ACTPEND_INSURANCE_DATASET,

            "request_url":
                FMCSA_ACTPEND_INSURANCE_URL
        },

        "record_count": 0,

        "records": [],

        "error": None,

        "retrieved_at_utc": None
    }


    # --------------------------------------------------------
    # INPUT VALIDATION
    # --------------------------------------------------------

    if source_usdot is None:

        result["lookup_status"] = (
            "INVALID_INPUT"
        )

        result["error"] = (
            "USDOT must contain numbers only."
        )

        return result


    # --------------------------------------------------------
    # QUERY
    # --------------------------------------------------------

    params = {

        "$where":
            f"dot_number='{source_usdot}'",

        "$limit":
            5000
    }


    try:

        response = requests.get(
            FMCSA_ACTPEND_INSURANCE_URL,
            params=params,
            timeout=timeout
        )

        result["source"]["http_status"] = (
            response.status_code
        )

        result["source"]["resolved_url"] = (
            response.url
        )

        response.raise_for_status()

        rows = response.json()


        if not isinstance(rows, list):

            raise TypeError(
                "FMCSA insurance lifecycle "
                "response was not a list."
            )


        result["success"] = True

        result["record_count"] = len(rows)

        result["records"] = rows

        result["retrieved_at_utc"] = (
            datetime.now(UTC).isoformat()
        )


        if rows:

            result["lookup_status"] = "FOUND"

        else:

            result["lookup_status"] = (
                "NOT_FOUND"
            )


        return result


    except Exception as exc:

        result["lookup_status"] = (
            "SOURCE_ERROR"
        )

        result["error"] = (
            f"{type(exc).__name__}: {exc}"
        )

        result["retrieved_at_utc"] = (
            datetime.now(UTC).isoformat()
        )

        return result


# ============================================================
# NORMALIZATION
# ============================================================

def normalize_fmcsa_insurance_lifecycle(
    lifecycle_result
):
    """
    Normalize lifecycle evidence while preserving all raw
    FMCSA source rows.

    Exact duplicate source rows are removed only from the
    operational evidence view.
    """

    raw_rows = lifecycle_result.get(
        "records",
        []
    )


    evidence = {

        "evidence_type":
            "carrier_insurance_lifecycle_fmcsa",

        "evidence_status":
            "UNAVAILABLE",

        "usdot":
            lifecycle_result.get("usdot"),

        "source_usdot":
            lifecycle_result.get(
                "source_usdot"
            ),

        "source":
            lifecycle_result.get("source"),

        "lookup_status":
            lifecycle_result.get(
                "lookup_status"
            ),

        "filings": [],

        "raw_record_count":
            len(raw_rows),

        "unique_record_count":
            0,

        "duplicate_rows_removed":
            0,

        "retrieved_at_utc":
            lifecycle_result.get(
                "retrieved_at_utc"
            ),

        "error":
            lifecycle_result.get("error"),

        "raw_records":
            raw_rows
    }


    if not lifecycle_result.get("success"):

        return evidence


    if not raw_rows:

        evidence["evidence_status"] = (
            "NOT_FOUND"
        )

        return evidence


    # --------------------------------------------------------
    # EXACT DEDUPLICATION
    # --------------------------------------------------------

    unique_rows = []

    seen = set()


    for row in raw_rows:

        signature = json.dumps(
            row,
            sort_keys=True,
            default=str
        )


        if signature in seen:
            continue


        seen.add(signature)

        unique_rows.append(row)


    evidence["unique_record_count"] = (
        len(unique_rows)
    )

    evidence["duplicate_rows_removed"] = (
        len(raw_rows) - len(unique_rows)
    )


    # --------------------------------------------------------
    # NORMALIZE
    # --------------------------------------------------------

    filings = []


    for row in unique_rows:

        cancellation_date = (
            row.get("cancl_effective_date")
        )


        # This describes the source evidence only.
        # It is NOT a VLCP insurance qualification decision.

        if cancellation_date:

            lifecycle_status = (
                "CANCELLATION_DATE_ON_FILE"
            )

        else:

            lifecycle_status = (
                "NO_CANCELLATION_DATE_RETURNED"
            )


        filings.append({

            "usdot":
                str(
                    row.get("dot_number")
                    or ""
                ).lstrip("0")
                or "0",

            "source_usdot":
                row.get("dot_number"),

            "docket_number":
                row.get("docket_number"),

            "form_code":
                row.get("ins_form_code"),

            "insurance_type_description":
                row.get("mod_col_1"),

            "insurance_company":
                row.get("name_company"),

            "policy_number":
                row.get("policy_no"),

            "transaction_date":
                row.get("trans_date"),

            "effective_date":
                row.get("effective_date"),

            "cancellation_effective_date":
                cancellation_date,

            "underlying_limit_amount":
                row.get("underl_lim_amount"),

            "maximum_coverage_amount":
                row.get("max_cov_amount"),

            "lifecycle_status":
                lifecycle_status
        })


    evidence["filings"] = filings

    evidence["evidence_status"] = (
        "RETRIEVED"
    )


    return evidence


print(
    "✓ VLCP FMCSA insurance lifecycle module v1 loaded"
)

✓ VLCP FMCSA insurance lifecycle module v1 loaded


In [29]:
# ============================================================
# VLCP — INSURANCE LIFECYCLE MULTI-USDOT REGRESSION
#
# Tests dynamic USDOT-only retrieval.
#
# No docket numbers are supplied.
# No broker qualification decisions are made.
# ============================================================

for test_usdot in ["951224", "80806"]:

    raw = get_fmcsa_insurance_lifecycle(
        test_usdot
    )

    evidence = (
        normalize_fmcsa_insurance_lifecycle(
            raw
        )
    )

    print("\n" + "=" * 78)
    print(f"USDOT: {test_usdot}")
    print(
        "Source USDOT:",
        evidence["source_usdot"]
    )
    print(
        "Lookup:",
        evidence["lookup_status"]
    )
    print(
        "Evidence:",
        evidence["evidence_status"]
    )
    print(
        "Raw rows:",
        evidence["raw_record_count"]
    )
    print(
        "Unique filings:",
        evidence["unique_record_count"]
    )
    print(
        "Duplicates removed:",
        evidence["duplicate_rows_removed"]
    )

    print("-" * 78)

    for filing in evidence["filings"]:

        print(
            "Docket:",
            filing["docket_number"],
            "| Form:",
            filing["form_code"]
        )

        print(
            "  Type:",
            filing["insurance_type_description"]
        )

        print(
            "  Company:",
            filing["insurance_company"]
        )

        print(
            "  Policy:",
            filing["policy_number"]
        )

        print(
            "  Effective:",
            filing["effective_date"],
            "| Cancel:",
            filing[
                "cancellation_effective_date"
            ]
        )

        print(
            "  Maximum:",
            filing[
                "maximum_coverage_amount"
            ]
        )

        print(
            "  Lifecycle:",
            filing["lifecycle_status"]
        )


print("\n" + "=" * 78)
print(
    "✓ Insurance lifecycle regression complete"
)
print("=" * 78)


USDOT: 951224
Source USDOT: 00951224
Lookup: FOUND
Evidence: RETRIEVED
Raw rows: 2
Unique filings: 2
Duplicates removed: 0
------------------------------------------------------------------------------
Docket: MC407574 | Form: 91X
  Type: BIPD/Primary
  Company: NORTHLAND INSURANCE COMPANY
  Policy: WF003597
  Effective: 05/01/2025 | Cancel: None
  Maximum: 1000
  Lifecycle: NO_CANCELLATION_DATE_RETURNED
Docket: MC407574 | Form: 84
  Type: SURETY
  Company: AMERICAN ALTERNATIVE INSURANCE CORPORATION
  Policy: MC407574
  Effective: 10/01/2013 | Cancel: None
  Maximum: 0
  Lifecycle: NO_CANCELLATION_DATE_RETURNED

USDOT: 80806
Source USDOT: 00080806
Lookup: FOUND
Evidence: RETRIEVED
Raw rows: 6
Unique filings: 6
Duplicates removed: 0
------------------------------------------------------------------------------
Docket: MC135797 | Form: 91X
  Type: BIPD/Excess
  Company: ACE AMERICAN INSURANCE COMPANY
  Policy: XSAH11347472
  Effective: 12/31/2025 | Cancel: None
  Maximum: 2500
  Lifecyc

# New Section

In [30]:
# ============================================================
# VLCP — UNIFIED CARRIER VERIFICATION ORCHESTRATOR v1
#
# Entry point:
#     verify_carrier("951224")
#
# Combines:
#   1. Carrier Identity
#   2. Operating Authority
#   3. Insurance Filing Evidence
#   4. Insurance Lifecycle Evidence
#
# Equipment remains a frozen module and will be attached after
# this core orchestrator passes regression.
#
# IMPORTANT:
# - Dynamic USDOT
# - Evidence != Policy != Decision
# - Source failures remain distinguishable from NOT_FOUND
# - No arbitrary score or broker qualification decision
# ============================================================

from datetime import datetime, UTC


def verify_carrier(usdot):
    """
    Run the core VLCP carrier verification workflow.

    Returns structured evidence from independent FMCSA
    verification modules.

    This function does NOT decide whether the carrier meets
    a particular broker/shipper policy.
    """

    usdot = str(usdot).strip()

    result = {

        "vlcp_verification_version": "1.0",

        "usdot": usdot,

        "verification_status": "UNKNOWN",

        "identity": None,

        "authority": None,

        "insurance": None,

        "insurance_lifecycle": None,

        "module_status": {},

        "sources": [],

        "errors": [],

        "verified_at_utc":
            datetime.now(UTC).isoformat()
    }


    # ========================================================
    # INPUT VALIDATION
    # ========================================================

    if not usdot:

        result["verification_status"] = (
            "INVALID_INPUT"
        )

        result["errors"].append(
            "USDOT is required."
        )

        return result


    if not usdot.isdigit():

        result["verification_status"] = (
            "INVALID_INPUT"
        )

        result["errors"].append(
            "USDOT must contain numbers only."
        )

        return result


    # ========================================================
    # 1. CARRIER IDENTITY
    # ========================================================

    try:

        identity = resolve_carrier_identity(
            usdot
        )

        result["identity"] = identity

        identity_status = identity.get(
            "identity_status",
            "UNKNOWN"
        )

        result["module_status"][
            "identity"
        ] = identity_status


        if identity.get("identity_source"):

            result["sources"].append(
                identity["identity_source"]
            )


        if identity.get("error"):

            result["errors"].append(
                "Identity: "
                + str(identity["error"])
            )


    except Exception as exc:

        result["module_status"][
            "identity"
        ] = "ERROR"

        result["errors"].append(
            "Identity: "
            f"{type(exc).__name__}: {exc}"
        )


    # ========================================================
    # 2. OPERATING AUTHORITY
    # ========================================================

    try:

        authority_raw = get_fmcsa_authority(
            usdot
        )

        authority = (
            normalize_fmcsa_authority_result(
                authority_raw
            )
        )

        result["authority"] = authority

        result["module_status"][
            "authority"
        ] = authority.get(
            "evidence_status",
            "UNKNOWN"
        )


        source = authority.get("source")

        if isinstance(source, dict):

            source_name = source.get("name")

            if source_name:

                result["sources"].append(
                    source_name
                )


        if authority.get("error"):

            result["errors"].append(
                "Authority: "
                + str(authority["error"])
            )


    except Exception as exc:

        result["module_status"][
            "authority"
        ] = "ERROR"

        result["errors"].append(
            "Authority: "
            f"{type(exc).__name__}: {exc}"
        )


    # ========================================================
    # 3. INSURANCE FILING EVIDENCE
    # ========================================================

    try:

        insurance_raw = get_fmcsa_insurance(
            usdot
        )

        insurance = (
            normalize_fmcsa_insurance_result(
                insurance_raw
            )
        )

        result["insurance"] = insurance

        result["module_status"][
            "insurance"
        ] = insurance.get(
            "evidence_status",
            "UNKNOWN"
        )


        source = insurance.get("source")

        if isinstance(source, dict):

            source_name = source.get("name")

            if source_name:

                result["sources"].append(
                    source_name
                )


        if insurance.get("error"):

            result["errors"].append(
                "Insurance: "
                + str(insurance["error"])
            )


    except Exception as exc:

        result["module_status"][
            "insurance"
        ] = "ERROR"

        result["errors"].append(
            "Insurance: "
            f"{type(exc).__name__}: {exc}"
        )


    # ========================================================
    # 4. INSURANCE LIFECYCLE
    # ========================================================

    try:

        lifecycle_raw = (
            get_fmcsa_insurance_lifecycle(
                usdot
            )
        )

        lifecycle = (
            normalize_fmcsa_insurance_lifecycle(
                lifecycle_raw
            )
        )

        result["insurance_lifecycle"] = (
            lifecycle
        )

        result["module_status"][
            "insurance_lifecycle"
        ] = lifecycle.get(
            "evidence_status",
            "UNKNOWN"
        )


        source = lifecycle.get("source")

        if isinstance(source, dict):

            source_name = source.get("name")

            if source_name:

                result["sources"].append(
                    source_name
                )


        if lifecycle.get("error"):

            result["errors"].append(
                "Insurance lifecycle: "
                + str(lifecycle["error"])
            )


    except Exception as exc:

        result["module_status"][
            "insurance_lifecycle"
        ] = "ERROR"

        result["errors"].append(
            "Insurance lifecycle: "
            f"{type(exc).__name__}: {exc}"
        )


    # ========================================================
    # SOURCE DEDUPLICATION
    # ========================================================

    result["sources"] = list(
        dict.fromkeys(
            result["sources"]
        )
    )


    # ========================================================
    # OVERALL EVIDENCE STATE
    #
    # This is NOT carrier qualification.
    # It describes completion of the verification workflow.
    # ========================================================

    module_values = list(
        result["module_status"].values()
    )


    if any(
        status == "ERROR"
        for status in module_values
    ):

        result["verification_status"] = (
            "PARTIAL_SOURCE_ERROR"
        )


    elif all(
        status in (
            "FOUND",
            "RETRIEVED"
        )
        for status in module_values
    ):

        result["verification_status"] = (
            "EVIDENCE_RETRIEVED"
        )


    else:

        result["verification_status"] = (
            "PARTIAL_EVIDENCE"
        )


    return result


print(
    "✓ VLCP unified verify_carrier() v1 loaded"
)

✓ VLCP unified verify_carrier() v1 loaded


In [31]:
# ============================================================
# VLCP — UNIFIED CARRIER VERIFICATION SUMMARY v1
#
# Tests the new verify_carrier() orchestrator without
# dumping raw source records.
#
# This is an EVIDENCE summary — not a qualification score.
# ============================================================


def print_vlcp_carrier_verification(result):

    print("\n" + "=" * 78)
    print("VLCP CARRIER VERIFICATION")
    print("=" * 78)

    print(
        "USDOT:",
        result.get("usdot")
    )

    # --------------------------------------------------------
    # IDENTITY
    # --------------------------------------------------------

    identity = result.get("identity") or {}

    print(
        "Carrier:",
        identity.get("display_name")
        or "UNKNOWN"
    )

    print(
        "Overall Evidence:",
        result.get("verification_status")
    )


    # --------------------------------------------------------
    # MODULE STATUS
    # --------------------------------------------------------

    print("\nMODULE STATUS")
    print("-" * 78)

    for module, status in (
        result.get("module_status")
        or {}
    ).items():

        print(
            f"{module:<22} {status}"
        )


    # --------------------------------------------------------
    # AUTHORITY
    # --------------------------------------------------------

    authority = (
        result.get("authority")
        or {}
    )

    authority_records = authority.get(
        "authority_records",
        []
    )

    print("\nOPERATING AUTHORITY")
    print("-" * 78)

    if authority_records:

        for record in authority_records:

            print(
                record.get("authority_type"),
                "|",
                record.get("authority_status"),
                "|",
                record.get("docket_number")
            )

    else:

        print("No authority records returned.")


    # --------------------------------------------------------
    # INSURANCE
    # --------------------------------------------------------

    insurance = (
        result.get("insurance")
        or {}
    )

    print("\nINSURANCE FILING EVIDENCE")
    print("-" * 78)

    print(
        "Raw records:",
        insurance.get(
            "raw_record_count",
            0
        )
    )

    print(
        "Unique filings:",
        insurance.get(
            "unique_record_count",
            0
        )
    )

    print(
        "Duplicates removed:",
        insurance.get(
            "duplicate_rows_removed",
            0
        )
    )


    # --------------------------------------------------------
    # INSURANCE LIFECYCLE
    # --------------------------------------------------------

    lifecycle = (
        result.get(
            "insurance_lifecycle"
        )
        or {}
    )

    filings = lifecycle.get(
        "filings",
        []
    )

    print("\nINSURANCE LIFECYCLE EVIDENCE")
    print("-" * 78)

    if filings:

        for filing in filings:

            print(
                filing.get(
                    "insurance_type_description"
                ),
                "|",
                filing.get(
                    "insurance_company"
                )
            )

            print(
                "  Effective:",
                filing.get(
                    "effective_date"
                ),
                "| Cancel:",
                filing.get(
                    "cancellation_effective_date"
                )
            )

            print(
                "  Evidence:",
                filing.get(
                    "lifecycle_status"
                )
            )

    else:

        print(
            "No lifecycle filings returned."
        )


    # --------------------------------------------------------
    # SOURCES
    # --------------------------------------------------------

    print("\nSOURCES")
    print("-" * 78)

    for source in result.get(
        "sources",
        []
    ):

        print("•", source)


    # --------------------------------------------------------
    # ERRORS
    # --------------------------------------------------------

    errors = result.get(
        "errors",
        []
    )

    print("\nERRORS")
    print("-" * 78)

    if errors:

        for error in errors:

            print("•", error)

    else:

        print("None")


    print("=" * 78)


# ============================================================
# FIRST END-TO-END TEST
# ============================================================

vlcp_test_951224 = verify_carrier(
    "951224"
)

print_vlcp_carrier_verification(
    vlcp_test_951224
)


VLCP CARRIER VERIFICATION
USDOT: 951224
Carrier: UNKNOWN
Overall Evidence: PARTIAL_SOURCE_ERROR

MODULE STATUS
------------------------------------------------------------------------------
identity               ERROR
authority              ERROR
insurance              ERROR
insurance_lifecycle    RETRIEVED

OPERATING AUTHORITY
------------------------------------------------------------------------------
No authority records returned.

INSURANCE FILING EVIDENCE
------------------------------------------------------------------------------
Raw records: 0
Unique filings: 0
Duplicates removed: 0

INSURANCE LIFECYCLE EVIDENCE
------------------------------------------------------------------------------
BIPD/Primary | NORTHLAND INSURANCE COMPANY
  Effective: 05/01/2025 | Cancel: None
  Evidence: NO_CANCELLATION_DATE_RETURNED
SURETY | AMERICAN ALTERNATIVE INSURANCE CORPORATION
  Effective: 10/01/2013 | Cancel: None
  Evidence: NO_CANCELLATION_DATE_RETURNED

SOURCES
----------------------

In [32]:
# ============================================================
# VLCP — UNIFIED VERIFICATION DEPENDENCY AUDIT
#
# Purpose:
# Identify exactly which previously verified functions remain
# loaded in the current Colab runtime.
#
# READ-ONLY — changes nothing.
# ============================================================

REQUIRED_VLCP_FUNCTIONS = [

    # Identity
    "get_fmcsa_carrier",
    "normalize_fmcsa_carrier_result",
    "resolve_carrier_identity",

    # Authority
    "get_fmcsa_authority",
    "normalize_fmcsa_authority_result",

    # Insurance
    "get_fmcsa_insurance",
    "normalize_fmcsa_insurance_result",

    # Insurance lifecycle
    "normalize_fmcsa_dot_for_actpend",
    "get_fmcsa_insurance_lifecycle",
    "normalize_fmcsa_insurance_lifecycle",

    # Unified orchestrator
    "verify_carrier",

    # Frozen equipment dependencies
    "get_fmcsa_inspection_units",
    "normalize_fmcsa_inspection_units",
    "build_unique_equipment_index",
    "build_vin_decode_index",
    "print_vlcp_equipment_summary",
    "run_vlcp_operational_equipment"
]


print("=" * 78)
print("VLCP RUNTIME DEPENDENCY AUDIT")
print("=" * 78)

loaded = []
missing = []


for function_name in REQUIRED_VLCP_FUNCTIONS:

    obj = globals().get(
        function_name
    )

    if callable(obj):

        loaded.append(
            function_name
        )

        print(
            f"✓ {function_name}"
        )

    else:

        missing.append(
            function_name
        )

        print(
            f"✗ {function_name}"
        )


print("\n" + "=" * 78)

print(
    f"Loaded:  {len(loaded)}"
)

print(
    f"Missing: {len(missing)}"
)

print("=" * 78)


if missing:

    print("\nMISSING DEPENDENCIES")

    for function_name in missing:

        print(
            "•",
            function_name
        )

else:

    print(
        "\n✓ All VLCP operational dependencies are loaded."
    )

VLCP RUNTIME DEPENDENCY AUDIT
✓ get_fmcsa_carrier
✓ normalize_fmcsa_carrier_result
✗ resolve_carrier_identity
✗ get_fmcsa_authority
✗ normalize_fmcsa_authority_result
✗ get_fmcsa_insurance
✗ normalize_fmcsa_insurance_result
✓ normalize_fmcsa_dot_for_actpend
✓ get_fmcsa_insurance_lifecycle
✓ normalize_fmcsa_insurance_lifecycle
✓ verify_carrier
✓ get_fmcsa_inspection_units
✓ normalize_fmcsa_inspection_units
✓ build_unique_equipment_index
✗ build_vin_decode_index
✗ print_vlcp_equipment_summary
✗ run_vlcp_operational_equipment

Loaded:  9
Missing: 8

MISSING DEPENDENCIES
• resolve_carrier_identity
• get_fmcsa_authority
• normalize_fmcsa_authority_result
• get_fmcsa_insurance
• normalize_fmcsa_insurance_result
• build_vin_decode_index
• print_vlcp_equipment_summary
• run_vlcp_operational_equipment


In [33]:
# ============================================================
# VLCP — CORE VERIFICATION DEPENDENCY RECOVERY
#
# Restores ONLY missing dependencies required by:
#
#     verify_carrier(usdot)
#
# Restores:
#   1. resolve_carrier_identity
#   2. get_fmcsa_authority
#   3. normalize_fmcsa_authority_result
#   4. get_fmcsa_insurance
#   5. normalize_fmcsa_insurance_result v2
#
# Does NOT modify:
#   - verify_carrier()
#   - insurance lifecycle module
#   - frozen equipment module
# ============================================================

import requests
import json
from datetime import datetime, UTC


# ============================================================
# 1. CARRIER IDENTITY
# ============================================================

def resolve_carrier_identity(usdot):
    """
    Retrieve and normalize FMCSA carrier identity.

    Source failures remain distinguishable from NOT_FOUND.
    """

    usdot = str(usdot).strip()

    identity = {
        "usdot": usdot,
        "legal_name": None,
        "dba_name": None,
        "display_name": None,
        "identity_status": "UNKNOWN",
        "identity_source": "FMCSA QCMobile",
        "error": None,
    }

    try:

        result = get_fmcsa_carrier(
            usdot,
            webkey=FMCSA_WEBKEY
        )

        identity["identity_status"] = (
            result.get(
                "lookup_status",
                "UNKNOWN"
            )
        )

        if not result.get("success"):

            identity["error"] = (
                result.get("error")
            )

            identity["display_name"] = (
                f"UNKNOWN CARRIER — USDOT {usdot}"
            )

            return identity


        normalized = (
            normalize_fmcsa_carrier_result(
                result
            )
        )


        if isinstance(normalized, dict):

            if isinstance(
                normalized.get("carrier"),
                dict
            ):

                carrier = normalized["carrier"]

            else:

                carrier = normalized

        else:

            carrier = {}


        identity["legal_name"] = (
            carrier.get("legal_name")
        )

        identity["dba_name"] = (
            carrier.get("dba_name")
        )


        if identity["legal_name"]:

            identity["display_name"] = (
                identity["legal_name"]
            )

        elif identity["dba_name"]:

            identity["display_name"] = (
                identity["dba_name"]
            )

        else:

            identity["display_name"] = (
                f"UNKNOWN CARRIER — USDOT {usdot}"
            )


        return identity


    except Exception as exc:

        identity["identity_status"] = "ERROR"

        identity["error"] = (
            f"{type(exc).__name__}: {exc}"
        )

        identity["display_name"] = (
            f"UNKNOWN CARRIER — USDOT {usdot}"
        )

        return identity


# ============================================================
# 2. OPERATING AUTHORITY
# ============================================================

FMCSA_MOTUS_CARRIER_DATASET = "inys-ebih"

FMCSA_MOTUS_CARRIER_URL = (
    "https://data.transportation.gov/"
    f"resource/{FMCSA_MOTUS_CARRIER_DATASET}.json"
)


def get_fmcsa_authority(
    usdot,
    timeout=30
):

    usdot = str(usdot).strip()

    result = {

        "usdot": usdot,

        "success": False,

        "lookup_status": "UNKNOWN",

        "source": {
            "name":
                "FMCSA Motus Carrier - All With History",

            "dataset_id":
                FMCSA_MOTUS_CARRIER_DATASET,

            "request_url":
                FMCSA_MOTUS_CARRIER_URL
        },

        "record_count": 0,

        "records": [],

        "error": None,

        "retrieved_at_utc": None
    }


    if not usdot:

        result["lookup_status"] = (
            "INVALID_INPUT"
        )

        result["error"] = (
            "USDOT is required."
        )

        return result


    if not usdot.isdigit():

        result["lookup_status"] = (
            "INVALID_INPUT"
        )

        result["error"] = (
            "USDOT must contain numbers only."
        )

        return result


    params = {

        "$where":
            f"usdot_number='{usdot}'",

        "$limit":
            5000
    }


    try:

        response = requests.get(
            FMCSA_MOTUS_CARRIER_URL,
            params=params,
            timeout=timeout
        )

        result["source"]["http_status"] = (
            response.status_code
        )

        result["source"]["resolved_url"] = (
            response.url
        )

        response.raise_for_status()

        rows = response.json()


        if not isinstance(rows, list):

            raise TypeError(
                "FMCSA authority response "
                "was not a list."
            )


        result["success"] = True

        result["record_count"] = len(rows)

        result["records"] = rows

        result["retrieved_at_utc"] = (
            datetime.now(UTC).isoformat()
        )


        if rows:

            result["lookup_status"] = "FOUND"

        else:

            result["lookup_status"] = (
                "NOT_FOUND"
            )


        return result


    except Exception as exc:

        result["lookup_status"] = (
            "SOURCE_ERROR"
        )

        result["error"] = (
            f"{type(exc).__name__}: {exc}"
        )

        result["retrieved_at_utc"] = (
            datetime.now(UTC).isoformat()
        )

        return result


def normalize_fmcsa_authority_result(
    authority_result
):

    evidence = {

        "evidence_type":
            "carrier_authority_fmcsa",

        "evidence_status":
            "UNAVAILABLE",

        "usdot":
            authority_result.get("usdot"),

        "source":
            authority_result.get("source"),

        "lookup_status":
            authority_result.get(
                "lookup_status"
            ),

        "authority_records": [],

        "retrieved_at_utc":
            authority_result.get(
                "retrieved_at_utc"
            ),

        "error":
            authority_result.get("error"),

        "raw_records":
            authority_result.get(
                "records",
                []
            )
    }


    if not authority_result.get("success"):

        return evidence


    rows = authority_result.get(
        "records",
        []
    )


    if not rows:

        evidence["evidence_status"] = (
            "NOT_FOUND"
        )

        return evidence


    normalized_records = []


    for row in rows:

        normalized_records.append({

            "usdot": str(
                row.get("usdot_number")
                or ""
            ).strip(),

            "docket_number":
                row.get("docket_number"),

            "authority_type":
                row.get("op_auth_type"),

            "authority_status":
                row.get("op_auth_status"),

            "minimum_coverage_amount":
                row.get("min_cov_amount"),

            "cargo_required":
                row.get("cargo_req"),

            "bond_required":
                row.get("bond_req"),

            "bipd_on_file":
                row.get("bipd_file"),

            "cargo_on_file":
                row.get("cargo_file"),

            "bond_on_file":
                row.get("bond_file"),

            "legal_name":
                row.get("legal_name"),

            "dba_name":
                row.get("dba_name")
        })


    evidence["authority_records"] = (
        normalized_records
    )

    evidence["evidence_status"] = (
        "RETRIEVED"
    )

    return evidence


# ============================================================
# 3. INSURANCE FILING EVIDENCE
# ============================================================

FMCSA_MOTUS_INSURANCE_DATASET = "c5y8-a4uz"

FMCSA_MOTUS_INSURANCE_URL = (
    "https://data.transportation.gov/"
    f"resource/{FMCSA_MOTUS_INSURANCE_DATASET}.json"
)


def get_fmcsa_insurance(
    usdot,
    timeout=30
):

    usdot = str(usdot).strip()

    result = {

        "usdot": usdot,

        "success": False,

        "lookup_status": "UNKNOWN",

        "source": {
            "name":
                "FMCSA Motus Insur - All With History",

            "dataset_id":
                FMCSA_MOTUS_INSURANCE_DATASET,

            "request_url":
                FMCSA_MOTUS_INSURANCE_URL
        },

        "record_count": 0,

        "records": [],

        "error": None,

        "retrieved_at_utc": None
    }


    if not usdot:

        result["lookup_status"] = (
            "INVALID_INPUT"
        )

        result["error"] = (
            "USDOT is required."
        )

        return result


    if not usdot.isdigit():

        result["lookup_status"] = (
            "INVALID_INPUT"
        )

        result["error"] = (
            "USDOT must contain numbers only."
        )

        return result


    params = {

        "$where":
            f"usdot_number='{usdot}'",

        "$limit":
            5000
    }


    try:

        response = requests.get(
            FMCSA_MOTUS_INSURANCE_URL,
            params=params,
            timeout=timeout
        )

        result["source"]["http_status"] = (
            response.status_code
        )

        result["source"]["resolved_url"] = (
            response.url
        )

        response.raise_for_status()

        rows = response.json()


        if not isinstance(rows, list):

            raise TypeError(
                "FMCSA insurance response "
                "was not a list."
            )


        result["success"] = True

        result["record_count"] = len(rows)

        result["records"] = rows

        result["retrieved_at_utc"] = (
            datetime.now(UTC).isoformat()
        )


        if rows:

            result["lookup_status"] = "FOUND"

        else:

            result["lookup_status"] = (
                "NOT_FOUND"
            )


        return result


    except Exception as exc:

        result["lookup_status"] = (
            "SOURCE_ERROR"
        )

        result["error"] = (
            f"{type(exc).__name__}: {exc}"
        )

        result["retrieved_at_utc"] = (
            datetime.now(UTC).isoformat()
        )

        return result


# ============================================================
# 4. INSURANCE NORMALIZER v2
# ============================================================

def normalize_fmcsa_insurance_result(
    insurance_result
):

    raw_rows = insurance_result.get(
        "records",
        []
    )


    evidence = {

        "evidence_type":
            "carrier_insurance_fmcsa",

        "evidence_status":
            "UNAVAILABLE",

        "usdot":
            insurance_result.get("usdot"),

        "source":
            insurance_result.get("source"),

        "lookup_status":
            insurance_result.get(
                "lookup_status"
            ),

        "insurance_records": [],

        "raw_record_count":
            len(raw_rows),

        "unique_record_count":
            0,

        "duplicate_rows_removed":
            0,

        "retrieved_at_utc":
            insurance_result.get(
                "retrieved_at_utc"
            ),

        "error":
            insurance_result.get("error"),

        "raw_records":
            raw_rows
    }


    if not insurance_result.get("success"):

        return evidence


    if not raw_rows:

        evidence["evidence_status"] = (
            "NOT_FOUND"
        )

        return evidence


    # Exact source-row deduplication
    unique_rows = []

    seen = set()


    for row in raw_rows:

        signature = json.dumps(
            row,
            sort_keys=True,
            default=str
        )


        if signature in seen:
            continue


        seen.add(signature)

        unique_rows.append(row)


    evidence["unique_record_count"] = (
        len(unique_rows)
    )

    evidence["duplicate_rows_removed"] = (
        len(raw_rows) - len(unique_rows)
    )


    normalized_records = []


    for row in unique_rows:

        normalized_records.append({

            "usdot": str(
                row.get("usdot_number")
                or ""
            ).strip(),

            "docket_number":
                row.get("docket_number"),

            "form_code":
                row.get("ins_form_code"),

            "insurance_type":
                row.get("ins_type_code"),

            "insurance_class":
                row.get("ins_class_code"),

            "maximum_coverage_amount":
                row.get("max_cov_amount"),

            "underlying_limit_amount":
                row.get("underl_lim_amount"),

            "policy_number":
                row.get("policy_no"),

            "effective_date":
                row.get("effective_date"),

            "insurance_company":
                row.get(
                    "insurance_company_name"
                ),

            "transaction_date":
                row.get("trans_date")
        })


    evidence["insurance_records"] = (
        normalized_records
    )

    evidence["evidence_status"] = (
        "RETRIEVED"
    )


    return evidence


print(
    "✓ VLCP core verification dependencies restored"
)

✓ VLCP core verification dependencies restored


In [34]:
verify_carrier("951224")

{'vlcp_verification_version': '1.0',
 'usdot': '951224',
 'verification_status': 'EVIDENCE_RETRIEVED',
 'identity': {'usdot': '951224',
  'legal_name': 'US SERVICES LLC',
  'dba_name': None,
  'display_name': 'US SERVICES LLC',
  'identity_status': 'FOUND',
  'identity_source': 'FMCSA QCMobile',
  'error': None},
 'authority': {'evidence_type': 'carrier_authority_fmcsa',
  'evidence_status': 'RETRIEVED',
  'usdot': '951224',
  'source': {'name': 'FMCSA Motus Carrier - All With History',
   'dataset_id': 'inys-ebih',
   'request_url': 'https://data.transportation.gov/resource/inys-ebih.json',
   'http_status': 200,
   'resolved_url': 'https://data.transportation.gov/resource/inys-ebih.json?%24where=usdot_number%3D%27951224%27&%24limit=5000'},
  'lookup_status': 'FOUND',
  'authority_records': [{'usdot': '951224',
    'docket_number': 'MC407574',
    'authority_type': 'Broker of Property (Except Household Goods)',
    'authority_status': 'Active',
    'minimum_coverage_amount': '0.000000

In [35]:
# ============================================================
# VLCP — UNIFIED VERIFY_CARRIER MULTI-CONDITION REGRESSION
#
# 80806:
#   Large established carrier / multiple authority & filings
#
# 2232566:
#   Different / partial-data carrier condition
#
# Purpose:
# Verify orchestration behavior, NOT carrier qualification.
# ============================================================

TEST_USDOTS = [
    "80806",
    "2232566"
]


for test_usdot in TEST_USDOTS:

    result = verify_carrier(
        test_usdot
    )

    identity = (
        result.get("identity")
        or {}
    )

    authority = (
        result.get("authority")
        or {}
    )

    insurance = (
        result.get("insurance")
        or {}
    )

    lifecycle = (
        result.get("insurance_lifecycle")
        or {}
    )

    print("\n" + "=" * 78)
    print("VLCP UNIFIED VERIFICATION REGRESSION")
    print("=" * 78)

    print(
        "Carrier:",
        identity.get("display_name")
        or "UNKNOWN"
    )

    print(
        "USDOT:",
        result.get("usdot")
    )

    print(
        "Overall Evidence:",
        result.get(
            "verification_status"
        )
    )

    print("\nMODULE STATUS")
    print("-" * 78)

    for module, status in (
        result.get("module_status")
        or {}
    ).items():

        print(
            f"{module:<22} {status}"
        )


    print("\nAUTHORITY")
    print("-" * 78)

    authority_records = (
        authority.get(
            "authority_records",
            []
        )
    )

    print(
        "Records:",
        len(authority_records)
    )

    for record in authority_records:

        print(
            "•",
            record.get("authority_type"),
            "|",
            record.get("authority_status"),
            "|",
            record.get("docket_number")
        )


    print("\nINSURANCE")
    print("-" * 78)

    print(
        "Raw:",
        insurance.get(
            "raw_record_count",
            0
        )
    )

    print(
        "Unique:",
        insurance.get(
            "unique_record_count",
            0
        )
    )

    print(
        "Duplicates removed:",
        insurance.get(
            "duplicate_rows_removed",
            0
        )
    )


    print("\nINSURANCE LIFECYCLE")
    print("-" * 78)

    filings = lifecycle.get(
        "filings",
        []
    )

    print(
        "Filings:",
        len(filings)
    )

    for filing in filings:

        print(
            "•",
            filing.get(
                "insurance_type_description"
            ),
            "|",
            filing.get(
                "docket_number"
            ),
            "|",
            filing.get(
                "lifecycle_status"
            )
        )


    print("\nERRORS")
    print("-" * 78)

    errors = result.get(
        "errors",
        []
    )

    if errors:

        for error in errors:

            print("•", error)

    else:

        print("None")


print("\n" + "=" * 78)
print("✓ Unified multi-condition regression complete")
print("=" * 78)


VLCP UNIFIED VERIFICATION REGRESSION
Carrier: J B HUNT TRANSPORT INC
USDOT: 80806
Overall Evidence: EVIDENCE_RETRIEVED

MODULE STATUS
------------------------------------------------------------------------------
identity               FOUND
authority              RETRIEVED
insurance              RETRIEVED
insurance_lifecycle    RETRIEVED

AUTHORITY
------------------------------------------------------------------------------
Records: 4
• Broker of Household Goods | Active | MC135797
• Motor Carrier of Property (Except Household Goods) | Active | MC135797
• Motor Carrier of Household Goods | Active | MC135797
• Broker of Property (Except Household Goods) | Active | MC135797

INSURANCE
------------------------------------------------------------------------------
Raw: 16
Unique: 5
Duplicates removed: 11

INSURANCE LIFECYCLE
------------------------------------------------------------------------------
Filings: 6
• BIPD/Excess | MC135797 | NO_CANCELLATION_DATE_RETURNED
• BIPD/Excess | 

In [36]:
# ============================================================
# VLCP — FROZEN EQUIPMENT MODULE RECOVERY AUDIT
#
# We are NOT modifying equipment logic.
# This identifies the surviving functions required to restore
# the previously verified equipment module.
# ============================================================

EQUIPMENT_DEPENDENCIES = [
    "normalize_vin",
    "is_candidate_vin",
    "decode_vin_nhtsa",
    "collect_vins_from_equipment_index",
    "build_vin_quality_index",
    "get_display_equipment_type",
    "get_display_verification",
    "clean_display_value",
    "title_display",
    "get_fmcsa_inspection_units",
    "normalize_fmcsa_inspection_units",
    "build_unique_equipment_index",
    "build_vin_decode_index",
    "print_vlcp_equipment_summary",
    "run_vlcp_operational_equipment"
]

print("=" * 78)
print("VLCP FROZEN EQUIPMENT RECOVERY AUDIT")
print("=" * 78)

available = []
missing = []

for name in EQUIPMENT_DEPENDENCIES:

    obj = globals().get(name)

    if callable(obj):
        available.append(name)
        print(f"✓ {name}")
    else:
        missing.append(name)
        print(f"✗ {name}")

print("\n" + "=" * 78)
print(f"Available: {len(available)}")
print(f"Missing:   {len(missing)}")
print("=" * 78)

if missing:
    print("\nRESTORE REQUIRED")
    for name in missing:
        print("•", name)
else:
    print("\n✓ Frozen equipment module is fully available.")

VLCP FROZEN EQUIPMENT RECOVERY AUDIT
✗ normalize_vin
✗ is_candidate_vin
✗ decode_vin_nhtsa
✗ collect_vins_from_equipment_index
✗ build_vin_quality_index
✗ get_display_equipment_type
✗ get_display_verification
✗ clean_display_value
✗ title_display
✓ get_fmcsa_inspection_units
✓ normalize_fmcsa_inspection_units
✓ build_unique_equipment_index
✗ build_vin_decode_index
✗ print_vlcp_equipment_summary
✗ run_vlcp_operational_equipment

Available: 3
Missing:   12

RESTORE REQUIRED
• normalize_vin
• is_candidate_vin
• decode_vin_nhtsa
• collect_vins_from_equipment_index
• build_vin_quality_index
• get_display_equipment_type
• get_display_verification
• clean_display_value
• title_display
• build_vin_decode_index
• print_vlcp_equipment_summary
• run_vlcp_operational_equipment


In [67]:
# ============================================================
# VLCP — FROZEN EQUIPMENT MODULE RECOVERY AUDIT
#
# We are NOT modifying equipment logic.
# This identifies the surviving functions required to restore
# the previously verified equipment module.
# ============================================================

EQUIPMENT_DEPENDENCIES = [
    "normalize_vin",
    "is_candidate_vin",
    "decode_vin_nhtsa",
    "collect_vins_from_equipment_index",
    "build_vin_quality_index",
    "get_display_equipment_type",
    "get_display_verification",
    "clean_display_value",
    "title_display",
    "get_fmcsa_inspection_units",
    "normalize_fmcsa_inspection_units",
    "build_unique_equipment_index",
    "build_vin_decode_index",
    "print_vlcp_equipment_summary",
    "run_vlcp_operational_equipment"
]

print("=" * 78)
print("VLCP FROZEN EQUIPMENT RECOVERY AUDIT")
print("=" * 78)

available = []
missing = []

for name in EQUIPMENT_DEPENDENCIES:

    obj = globals().get(name)

    if callable(obj):
        available.append(name)
        print(f"✓ {name}")
    else:
        missing.append(name)
        print(f"✗ {name}")

print("\n" + "=" * 78)
print(f"Available: {len(available)}")
print(f"Missing:   {len(missing)}")
print("=" * 78)

if missing:
    print("\nRESTORE REQUIRED")
    for name in missing:
        print("•", name)
else:
    print("\n✓ Frozen equipment module is fully available.")

VLCP FROZEN EQUIPMENT RECOVERY AUDIT
✓ normalize_vin
✓ is_candidate_vin
✓ decode_vin_nhtsa
✓ collect_vins_from_equipment_index
✓ build_vin_quality_index
✓ get_display_equipment_type
✓ get_display_verification
✓ clean_display_value
✓ title_display
✓ get_fmcsa_inspection_units
✓ normalize_fmcsa_inspection_units
✓ build_unique_equipment_index
✓ build_vin_decode_index
✓ print_vlcp_equipment_summary
✓ run_vlcp_operational_equipment

Available: 15
Missing:   0

✓ Frozen equipment module is fully available.


In [68]:
# ============================================================
# VLCP — OPERATIONAL CARRIER VERIFICATION v1
#
# Single operational entry point combining:
#
#   Core Carrier Evidence
#       Identity
#       Authority
#       Insurance
#       Insurance Lifecycle
#
#   Frozen Equipment Verification
#
# IMPORTANT:
# Evidence status is NOT broker/shipper qualification.
# ============================================================


def run_vlcp_verification(
    usdot,
    include_equipment=True,
    vin_limit=10,
    decode_delay=0.10
):
    """
    Run the complete VLCP carrier evidence workflow.

    Parameters
    ----------
    usdot : str/int
        Carrier USDOT number.

    include_equipment : bool
        Whether to run the frozen equipment evidence module.

    vin_limit : int or None
        Maximum VINs to decode/display during this run.
        None allows all candidate VINs.

    decode_delay : float
        Delay between NHTSA VIN requests.

    Returns
    -------
    dict
        Machine-readable VLCP verification package.

    NOTE:
    This function retrieves evidence.
    It does NOT determine whether a carrier satisfies
    a broker or shipper qualification policy.
    """

    usdot = str(usdot).strip()

    # --------------------------------------------------------
    # CORE VERIFICATION
    # --------------------------------------------------------

    result = verify_carrier(usdot)

    result["equipment"] = None

    result["module_status"]["equipment"] = (
        "NOT_REQUESTED"
    )


    # --------------------------------------------------------
    # EQUIPMENT
    # --------------------------------------------------------

    if include_equipment:

        try:

            equipment = (
                run_vlcp_operational_equipment(
                    usdot,
                    vin_limit=vin_limit,
                    decode_delay=decode_delay
                )
            )

            result["equipment"] = equipment

            result["module_status"][
                "equipment"
            ] = "RETRIEVED"


        except Exception as exc:

            result["module_status"][
                "equipment"
            ] = "ERROR"

            result["errors"].append(
                "Equipment: "
                f"{type(exc).__name__}: {exc}"
            )


    # --------------------------------------------------------
    # OPERATIONAL EVIDENCE STATUS
    #
    # Still NOT a qualification decision.
    # --------------------------------------------------------

    statuses = list(
        result["module_status"].values()
    )


    if any(
        status == "ERROR"
        for status in statuses
    ):

        result["operational_status"] = (
            "PARTIAL_SOURCE_ERROR"
        )


    elif result.get(
        "verification_status"
    ) == "EVIDENCE_RETRIEVED" and (
        not include_equipment
        or result["module_status"].get(
            "equipment"
        ) == "RETRIEVED"
    ):

        result["operational_status"] = (
            "EVIDENCE_RETRIEVED"
        )


    else:

        result["operational_status"] = (
            "PARTIAL_EVIDENCE"
        )


    return result


print(
    "✓ VLCP operational verification v1 loaded"
)

✓ VLCP operational verification v1 loaded


In [69]:
# ============================================================
# VLCP — FULL END-TO-END OPERATIONAL REGRESSION
#
# Known regression carrier:
# USDOT 951224 — US SERVICES LLC
#
# This executes:
#   Identity
#   Authority
#   Insurance
#   Insurance Lifecycle
#   FMCSA Equipment Evidence
#   VIN Extraction
#   NHTSA VIN Enrichment
#
# vin_limit=10 keeps the regression fast while still testing
# the complete pipeline.
# ============================================================

vlcp_full_test = run_vlcp_verification(
    "951224",
    include_equipment=True,
    vin_limit=10,
    decode_delay=0.10
)

print("\n" + "=" * 78)
print("VLCP FULL OPERATIONAL REGRESSION")
print("=" * 78)

identity = (
    vlcp_full_test.get("identity")
    or {}
)

print(
    "Carrier:",
    identity.get("display_name")
    or "UNKNOWN"
)

print(
    "USDOT:",
    vlcp_full_test.get("usdot")
)

print(
    "Core Evidence:",
    vlcp_full_test.get(
        "verification_status"
    )
)

print(
    "Operational Evidence:",
    vlcp_full_test.get(
        "operational_status"
    )
)


print("\nMODULE STATUS")
print("-" * 78)

for module, status in (
    vlcp_full_test.get(
        "module_status",
        {}
    )
).items():

    print(
        f"{module:<22} {status}"
    )


# ------------------------------------------------------------
# EQUIPMENT COUNTS
# ------------------------------------------------------------

equipment = (
    vlcp_full_test.get("equipment")
    or {}
)

print("\nEQUIPMENT EVIDENCE")
print("-" * 78)

print(
    "Inspections found:",
    equipment.get(
        "inspection_count",
        equipment.get(
            "inspections_found",
            "N/A"
        )
    )
)

print(
    "Equipment observations:",
    equipment.get(
        "equipment_observation_count",
        equipment.get(
            "equipment_observations",
            "N/A"
        )
    )
)

print(
    "Unique equipment observed:",
    equipment.get(
        "unique_equipment_count",
        equipment.get(
            "likely_unique_equipment",
            "N/A"
        )
    )
)


# ------------------------------------------------------------
# ERROR CHECK
# ------------------------------------------------------------

print("\nERRORS")
print("-" * 78)

errors = vlcp_full_test.get(
    "errors",
    []
)

if errors:

    for error in errors:
        print("•", error)

else:

    print("None")


print("\n" + "=" * 78)

if (
    vlcp_full_test.get(
        "operational_status"
    ) == "EVIDENCE_RETRIEVED"
    and not errors
):

    print(
        "✓ FULL VLCP OPERATIONAL PIPELINE COMPLETED"
    )

else:

    print(
        "⚠ VLCP PIPELINE COMPLETED WITH PARTIAL EVIDENCE OR ERRORS"
    )

print("=" * 78)

VLCP AUTHORITATIVE VIN DECODING
Candidate VINs: 10
Source: NHTSA vPIC
----------------------------------------------------------------------
[1/10] 1AJC40260T1002016 -> DECODED
[2/10] 1AJC40265T1002769 -> DECODED
[3/10] 1DW1C452X1E499580 -> DECODED
[4/10] 1FUGGHDV9LLLT8409 -> DECODED
[5/10] 1FUJA6CGX3LH34804 -> DECODED
[6/10] 1FUJA6CK28DY52591 -> DECODED
[7/10] 1FUJA6CK36LV99316 -> DECODED
[8/10] 1FUJA6CK47LX77848 -> DECODED
[9/10] 1FUJA6CK57LW40286 -> DECODED
[10/10] 1FUJA6CK65LN64042 -> DECODED
----------------------------------------------------------------------
Decode summary:
  DECODED: 10

VLCP Verified Equipment Summary

Carrier: US SERVICES LLC
USDOT: 951224

Equipment | Year | Make               | Model              | VIN               | Verification
----------+------+--------------------+--------------------+-------------------+-------------
Tractor   | 2003 | Freightliner       | Columbia           | 1FUJA6CGX3LH34804 | Verified    
Tractor   | 2005 | Freightliner       | C

In [70]:
# ============================================================
# VLCP — VERIFIED RUNTIME MANIFEST
#
# Checkpoint before persistent package consolidation.
# Does not modify any VLCP functions.
# ============================================================

import inspect
from datetime import datetime, UTC


VLCP_VERIFIED_RUNTIME = {

    "checkpoint":
        "VLCP_OPERATIONAL_VERIFICATION_V1",

    "status":
        "VERIFIED_FROZEN",

    "created_at_utc":
        datetime.now(UTC).isoformat(),

    "core": [
        "get_fmcsa_carrier",
        "normalize_fmcsa_carrier_result",
        "resolve_carrier_identity",

        "get_fmcsa_authority",
        "normalize_fmcsa_authority_result",

        "get_fmcsa_insurance",
        "normalize_fmcsa_insurance_result",

        "normalize_fmcsa_dot_for_actpend",
        "get_fmcsa_insurance_lifecycle",
        "normalize_fmcsa_insurance_lifecycle",

        "verify_carrier",
    ],

    "equipment": [
        "get_fmcsa_inspection_units",
        "normalize_fmcsa_inspection_units",
        "build_unique_equipment_index",

        "normalize_vin",
        "is_candidate_vin",
        "decode_vin_nhtsa",
        "collect_vins_from_equipment_index",
        "build_vin_quality_index",
        "build_vin_decode_index",

        "clean_display_value",
        "title_display",
        "get_display_equipment_type",
        "get_display_verification",
        "print_vlcp_equipment_summary",

        "run_vlcp_operational_equipment",
    ],

    "orchestration": [
        "run_vlcp_verification",
    ],
}


all_required = (
    VLCP_VERIFIED_RUNTIME["core"]
    + VLCP_VERIFIED_RUNTIME["equipment"]
    + VLCP_VERIFIED_RUNTIME["orchestration"]
)


missing = []
source_available = []


print("=" * 78)
print("VLCP VERIFIED RUNTIME CHECKPOINT")
print("=" * 78)


for name in all_required:

    obj = globals().get(name)

    if not callable(obj):

        missing.append(name)

        print(
            f"✗ {name:<42} MISSING"
        )

        continue


    try:

        inspect.getsource(obj)

        source_available.append(name)

        print(
            f"✓ {name:<42} READY"
        )

    except Exception:

        print(
            f"⚠ {name:<42} LOADED / SOURCE UNAVAILABLE"
        )


print("\n" + "=" * 78)

print(
    "Required functions:",
    len(all_required)
)

print(
    "Loaded:",
    len(all_required) - len(missing)
)

print(
    "Missing:",
    len(missing)
)

print(
    "Source extractable:",
    len(source_available)
)


if not missing:

    print(
        "\n✓ VLCP OPERATIONAL VERIFICATION v1 "
        "RUNTIME CHECKPOINT PASSED"
    )

else:

    print(
        "\n⚠ CHECKPOINT FAILED — "
        "do not package yet"
    )

print("=" * 78)

VLCP VERIFIED RUNTIME CHECKPOINT
✓ get_fmcsa_carrier                          READY
✓ normalize_fmcsa_carrier_result             READY
✓ resolve_carrier_identity                   READY
✓ get_fmcsa_authority                        READY
✓ normalize_fmcsa_authority_result           READY
✓ get_fmcsa_insurance                        READY
✓ normalize_fmcsa_insurance_result           READY
✓ normalize_fmcsa_dot_for_actpend            READY
✓ get_fmcsa_insurance_lifecycle              READY
✓ normalize_fmcsa_insurance_lifecycle        READY
✓ verify_carrier                             READY
✓ get_fmcsa_inspection_units                 READY
✓ normalize_fmcsa_inspection_units           READY
✓ build_unique_equipment_index               READY
✓ normalize_vin                              READY
✓ is_candidate_vin                           READY
✓ decode_vin_nhtsa                           READY
✓ collect_vins_from_equipment_index          READY
✓ build_vin_quality_index                    READ

In [71]:
# ============================================================
# VLCP — VERIFIED SOURCE SNAPSHOT v1
#
# Exports the EXACT source code of the 27 functions currently
# loaded in the verified runtime.
#
# Purpose:
#   - Preserve the working implementation
#   - Create a recovery checkpoint
#   - Prevent reconstruction from memory
#
# This does NOT modify any VLCP function.
# ============================================================

import inspect
import json
import os
from datetime import datetime, UTC


SNAPSHOT_DIR = "/content/vlcp_verified_snapshot_v1"

os.makedirs(
    SNAPSHOT_DIR,
    exist_ok=True
)


# ------------------------------------------------------------
# EXPORT FUNCTION SOURCE
# ------------------------------------------------------------

function_sources = {}


for function_name in all_required:

    obj = globals()[function_name]

    function_sources[function_name] = (
        inspect.getsource(obj)
    )


# ------------------------------------------------------------
# WRITE COMPLETE SOURCE SNAPSHOT
# ------------------------------------------------------------

snapshot_file = os.path.join(
    SNAPSHOT_DIR,
    "vlcp_verified_functions_v1.py"
)


with open(
    snapshot_file,
    "w",
    encoding="utf-8"
) as f:

    f.write(
        "# VLCP Operational Verification v1\n"
    )

    f.write(
        "# VERIFIED / FROZEN runtime snapshot\n"
    )

    f.write(
        "# Generated from successfully tested "
        "Colab runtime.\n\n"
    )

    for function_name in all_required:

        f.write(
            "\n\n# "
            + "=" * 70
            + "\n"
        )

        f.write(
            f"# FUNCTION: {function_name}\n"
        )

        f.write(
            "# "
            + "=" * 70
            + "\n\n"
        )

        f.write(
            function_sources[
                function_name
            ]
        )

        f.write("\n")


# ------------------------------------------------------------
# WRITE MANIFEST
# ------------------------------------------------------------

manifest = {

    "checkpoint":
        "VLCP_OPERATIONAL_VERIFICATION_V1",

    "status":
        "VERIFIED_FROZEN",

    "snapshot_created_at_utc":
        datetime.now(UTC).isoformat(),

    "function_count":
        len(all_required),

    "functions":
        all_required,

    "groups": {

        "core":
            VLCP_VERIFIED_RUNTIME["core"],

        "equipment":
            VLCP_VERIFIED_RUNTIME["equipment"],

        "orchestration":
            VLCP_VERIFIED_RUNTIME[
                "orchestration"
            ]
    },

    "known_regression": {

        "usdot":
            "951224",

        "carrier":
            "US SERVICES LLC",

        "core_status":
            "EVIDENCE_RETRIEVED",

        "operational_status":
            "EVIDENCE_RETRIEVED",

        "equipment_baseline": {

            "inspections_found":
                142,

            "equipment_observations":
                263,

            "unique_equipment_observed":
                201,

            "vin_test_count":
                10
        }
    },

    "design_rule":
        (
            "FACT -> EVIDENCE -> POLICY -> DECISION"
        )
}


manifest_file = os.path.join(
    SNAPSHOT_DIR,
    "manifest.json"
)


with open(
    manifest_file,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        manifest,
        f,
        indent=2
    )


# ------------------------------------------------------------
# VALIDATE SNAPSHOT
# ------------------------------------------------------------

with open(
    snapshot_file,
    "r",
    encoding="utf-8"
) as f:

    exported_source = f.read()


missing_from_snapshot = [

    name
    for name in all_required
    if f"def {name}(" not in exported_source
]


print("=" * 78)
print("VLCP VERIFIED SOURCE SNAPSHOT")
print("=" * 78)

print(
    "Snapshot directory:",
    SNAPSHOT_DIR
)

print(
    "Functions exported:",
    len(function_sources)
)

print(
    "Functions expected:",
    len(all_required)
)

print(
    "Missing from snapshot:",
    len(missing_from_snapshot)
)

print(
    "Source file:",
    snapshot_file
)

print(
    "Manifest:",
    manifest_file
)


if not missing_from_snapshot:

    print(
        "\n✓ VERIFIED VLCP SOURCE SNAPSHOT CREATED"
    )

else:

    print(
        "\n⚠ SNAPSHOT INCOMPLETE"
    )

    for name in missing_from_snapshot:

        print(
            "•",
            name
        )

print("=" * 78)

VLCP VERIFIED SOURCE SNAPSHOT
Snapshot directory: /content/vlcp_verified_snapshot_v1
Functions exported: 27
Functions expected: 27
Missing from snapshot: 0
Source file: /content/vlcp_verified_snapshot_v1/vlcp_verified_functions_v1.py
Manifest: /content/vlcp_verified_snapshot_v1/manifest.json

✓ VERIFIED VLCP SOURCE SNAPSHOT CREATED


# New Section

# New Section

# New Section